# Sales Order, Backlog & Inventory Optimization Analytics

## 02 — Data Quality Audit

### Objective

The purpose of this notebook is to systematically assess the quality, consistency, validity, and integrity of the raw datasets before any cleaning or transformation is performed.

### Scope

This notebook will evaluate:

- Missing values
- Duplicate records
- Primary and composite key integrity
- Referential integrity between related tables
- Data type issues
- Date validity and chronological consistency
- Negative, zero, or impossible quantities
- Invalid prices, costs, discounts, and monetary values
- Inconsistent categorical values
- Sales-order quantity consistency
- Backlog quantity and value consistency
- Inventory quantity consistency
- Stockout-status consistency
- Shipment and delivery consistency
- Purchase-order and supplier consistency
- Pre-calculated metric validation
- Cross-table business-rule inconsistencies
- Overall data-quality findings

> **Important:** This notebook only identifies and documents data-quality issues. No records will be cleaned, corrected, removed, imputed, or transformed here. All corrections will be performed later in `03_Data_Cleaning.ipynb`.

In [1]:
# Import required libraries

import pandas as pd
import numpy as np

from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)

print("Libraries imported successfully.")

Libraries imported successfully.


## Load Raw Datasets

Load the original raw datasets for data-quality assessment. The source files remain unchanged throughout this notebook.

In [3]:
# Define project and raw data paths

project_path = Path.cwd().parent
raw_data_path = project_path / "data" / "raw"

print("Project Path :", project_path)
print("Raw Data Path:", raw_data_path)
print("Raw Data Folder Exists:", raw_data_path.exists())

Project Path : c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics
Raw Data Path: c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics\data\raw
Raw Data Folder Exists: True


In [4]:
# Load raw datasets

customers = pd.read_csv(raw_data_path / "customers.csv")
products = pd.read_csv(raw_data_path / "products.csv")
sales_orders = pd.read_csv(raw_data_path / "sales_orders.csv")
sales_order_items = pd.read_csv(raw_data_path / "sales_order_items.csv")
inventory = pd.read_csv(raw_data_path / "inventory.csv")
inventory_transactions = pd.read_csv(raw_data_path / "inventory_transactions.csv")
warehouses = pd.read_csv(raw_data_path / "warehouses.csv")
shipments = pd.read_csv(raw_data_path / "shipments.csv")
suppliers = pd.read_csv(raw_data_path / "suppliers.csv")
purchase_orders = pd.read_csv(raw_data_path / "purchase_orders.csv")

datasets = {
    "Customers": customers,
    "Products": products,
    "Sales Orders": sales_orders,
    "Sales Order Items": sales_order_items,
    "Inventory": inventory,
    "Inventory Transactions": inventory_transactions,
    "Warehouses": warehouses,
    "Shipments": shipments,
    "Suppliers": suppliers,
    "Purchase Orders": purchase_orders
}

print(f"Successfully loaded {len(datasets)} datasets.")

Successfully loaded 10 datasets.


## 1. Missing Values Audit

Evaluate missing values across all datasets to identify incomplete fields and quantify the percentage of missing data in each column.

Missing values are only identified and documented at this stage. No imputation, deletion, or replacement is performed.

In [5]:
# Missing values audit

missing_summary = []

for table_name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()
        missing_pct = (missing_count / len(df)) * 100

        missing_summary.append({
            "Dataset": table_name,
            "Column": column,
            "Missing_Count": missing_count,
            "Missing_Percentage": round(missing_pct, 2)
        })

missing_summary = pd.DataFrame(missing_summary)

# Display only columns containing missing values
missing_issues = (
    missing_summary[missing_summary["Missing_Count"] > 0]
    .sort_values(
        ["Missing_Percentage", "Missing_Count"],
        ascending=False
    )
    .reset_index(drop=True)
)

if missing_issues.empty:
    print("No missing values detected across the datasets.")
else:
    display(missing_issues)

,Dataset,Column,Missing_Count,Missing_Percentage
0,Suppliers,quality_rating,1,1.00
1,Customers,credit_tier,20,0.80
2,Products,subcategory,3,0.60
3,Purchase Orders,actual_receipt_date,30,0.35
4,Purchase Orders,supplier_delay_days,30,0.35


## 2. Duplicate Records Audit

Check each dataset for exact duplicate rows that may result in duplicated business records or inflated analytical results.

This step only identifies duplicate records. No duplicates are removed during the audit.

In [6]:
# Check exact duplicate rows across all datasets

duplicate_summary = []

for table_name, df in datasets.items():

    duplicate_count = df.duplicated().sum()
    duplicate_pct = (duplicate_count / len(df)) * 100

    duplicate_summary.append({
        "Dataset": table_name,
        "Total_Rows": len(df),
        "Duplicate_Rows": duplicate_count,
        "Duplicate_Percentage": round(duplicate_pct, 2)
    })

duplicate_summary = pd.DataFrame(duplicate_summary)

display(duplicate_summary)

,Dataset,Total_Rows,Duplicate_Rows,Duplicate_Percentage
0,Customers,2500,0,0.00
1,Products,500,0,0.00
2,Sales Orders,25000,0,0.00
3,Sales Order Items,61397,0,0.00
4,Inventory,3000,0,0.00
5,Inventory Transactions,110045,45,0.04
6,Warehouses,6,0,0.00
7,Shipments,24798,0,0.00
8,Suppliers,100,0,0.00
9,Purchase Orders,8500,0,0.00


## 3. Primary Key Integrity Audit

Evaluate candidate primary keys for missing and duplicate identifiers.

A valid primary key should:

- Contain no missing values.
- Uniquely identify every record.
- Contain no duplicated identifiers.

The Inventory table will be evaluated separately using its composite key of `product_id + warehouse_id`.

In [7]:
# Primary key integrity audit

primary_keys = {
    "Customers": "customer_id",
    "Products": "product_id",
    "Sales Orders": "order_id",
    "Sales Order Items": "order_item_id",
    "Inventory Transactions": "transaction_id",
    "Warehouses": "warehouse_id",
    "Shipments": "shipment_id",
    "Suppliers": "supplier_id",
    "Purchase Orders": "po_id"
}

pk_summary = []

for table_name, key in primary_keys.items():
    df = datasets[table_name]

    pk_summary.append({
        "Dataset": table_name,
        "Primary_Key": key,
        "Total_Rows": len(df),
        "Missing_Keys": df[key].isna().sum(),
        "Unique_Keys": df[key].nunique(dropna=True),
        "Duplicate_Keys": df[key].duplicated().sum()
    })

pk_summary = pd.DataFrame(pk_summary)

display(pk_summary)

,Dataset,Primary_Key,Total_Rows,Missing_Keys,Unique_Keys,Duplicate_Keys
0,Customers,customer_id,2500,0,2500,0
1,Products,product_id,500,0,500,0
2,Sales Orders,order_id,25000,0,25000,0
3,Sales Order Items,order_item_id,61397,0,61397,0
4,Inventory Transactions,transaction_id,110045,0,110000,45
5,Warehouses,warehouse_id,6,0,6,0
6,Shipments,shipment_id,24798,0,24798,0
7,Suppliers,supplier_id,100,0,100,0
8,Purchase Orders,po_id,8500,0,8500,0


### Inventory Composite Key Integrity

The Inventory table does not contain a standalone primary key. The combination of `product_id` and `warehouse_id` is therefore evaluated as its candidate composite key.

In [8]:
# Inventory composite key integrity audit

composite_key = ["product_id", "warehouse_id"]

missing_composite_keys = inventory[composite_key].isna().any(axis=1).sum()

duplicate_composite_keys = inventory.duplicated(
    subset=composite_key,
    keep=False
).sum()

unique_combinations = (
    inventory[composite_key]
    .drop_duplicates()
    .shape[0]
)

print(f"Total Inventory Rows             : {len(inventory):,}")
print(f"Unique Product-Warehouse Pairs   : {unique_combinations:,}")
print(f"Missing Composite Key Rows       : {missing_composite_keys:,}")
print(f"Duplicate Composite Key Rows     : {duplicate_composite_keys:,}")

Total Inventory Rows             : 3,000
Unique Product-Warehouse Pairs   : 3,000
Missing Composite Key Rows       : 0
Duplicate Composite Key Rows     : 0


## 4. Referential Integrity Audit

Validate foreign-key relationships across related datasets to identify orphan records.

An orphan record occurs when a foreign-key value in a child table does not have a corresponding key in its parent table.

No records are modified during this check.

In [9]:
# Referential integrity audit

relationship_checks = [
    ("Sales Orders", "customer_id", "Customers", "customer_id"),
    ("Sales Orders", "warehouse_id", "Warehouses", "warehouse_id"),

    ("Sales Order Items", "order_id", "Sales Orders", "order_id"),
    ("Sales Order Items", "product_id", "Products", "product_id"),
    ("Sales Order Items", "warehouse_id", "Warehouses", "warehouse_id"),

    ("Inventory", "product_id", "Products", "product_id"),
    ("Inventory", "warehouse_id", "Warehouses", "warehouse_id"),

    ("Inventory Transactions", "product_id", "Products", "product_id"),
    ("Inventory Transactions", "warehouse_id", "Warehouses", "warehouse_id"),

    ("Shipments", "order_id", "Sales Orders", "order_id"),
    ("Shipments", "warehouse_id", "Warehouses", "warehouse_id"),

    ("Products", "supplier_id", "Suppliers", "supplier_id"),

    ("Purchase Orders", "supplier_id", "Suppliers", "supplier_id"),
    ("Purchase Orders", "product_id", "Products", "product_id"),
    ("Purchase Orders", "warehouse_id", "Warehouses", "warehouse_id")
]

referential_results = []

for child_table, foreign_key, parent_table, parent_key in relationship_checks:

    child_df = datasets[child_table]
    parent_df = datasets[parent_table]

    valid_parent_keys = set(parent_df[parent_key].dropna())

    orphan_mask = (
        child_df[foreign_key].notna()
        & ~child_df[foreign_key].isin(valid_parent_keys)
    )

    orphan_count = orphan_mask.sum()

    referential_results.append({
        "Child_Table": child_table,
        "Foreign_Key": foreign_key,
        "Parent_Table": parent_table,
        "Parent_Key": parent_key,
        "Orphan_Rows": orphan_count,
        "Orphan_Percentage": round(
            (orphan_count / len(child_df)) * 100, 2
        )
    })

referential_summary = pd.DataFrame(referential_results)

display(referential_summary)

,Child_Table,Foreign_Key,Parent_Table,Parent_Key,Orphan_Rows,Orphan_Percentage
0,Sales Orders,customer_id,Customers,customer_id,0,0.0
1,Sales Orders,warehouse_id,Warehouses,warehouse_id,0,0.0
2,Sales Order Items,order_id,Sales Orders,order_id,0,0.0
3,Sales Order Items,product_id,Products,product_id,0,0.0
4,Sales Order Items,warehouse_id,Warehouses,warehouse_id,0,0.0
5,Inventory,product_id,Products,product_id,0,0.0
6,Inventory,warehouse_id,Warehouses,warehouse_id,0,0.0
7,Inventory Transactions,product_id,Products,product_id,0,0.0
8,Inventory Transactions,warehouse_id,Warehouses,warehouse_id,0,0.0
9,Shipments,order_id,Sales Orders,order_id,0,0.0


## 5. Data Type Quality Audit

Evaluate whether important columns are stored using data types appropriate for their business meaning.

Special attention is given to date fields and numerical business measures. This step only identifies potential datatype issues; no conversion is performed.

In [10]:
# Define columns expected to represent dates

expected_date_columns = {
    "Customers": ["customer_since"],
    "Sales Orders": [
        "order_date",
        "requested_delivery_date"
    ],
    "Sales Order Items": [
        "order_date"
    ],
    "Inventory Transactions": [
        "transaction_date"
    ],
    "Shipments": [
        "shipment_date",
        "requested_delivery_date",
        "actual_delivery_date"
    ],
    "Purchase Orders": [
        "po_date",
        "expected_date",
        "actual_receipt_date"
    ]
}

date_type_results = []

for table_name, columns in expected_date_columns.items():

    df = datasets[table_name]

    for column in columns:

        current_dtype = str(df[column].dtype)

        is_datetime = pd.api.types.is_datetime64_any_dtype(
            df[column]
        )

        date_type_results.append({
            "Dataset": table_name,
            "Column": column,
            "Current_Dtype": current_dtype,
            "Expected_Type": "datetime",
            "Requires_Conversion": not is_datetime
        })

date_type_summary = pd.DataFrame(date_type_results)

display(date_type_summary)

,Dataset,Column,Current_Dtype,Expected_Type,Requires_Conversion
0,Customers,customer_since,str,datetime,True
1,Sales Orders,order_date,str,datetime,True
2,Sales Orders,requested_delivery_date,str,datetime,True
3,Sales Order Items,order_date,str,datetime,True
4,Inventory Transactions,transaction_date,str,datetime,True
5,Shipments,shipment_date,str,datetime,True
6,Shipments,requested_delivery_date,str,datetime,True
7,Shipments,actual_delivery_date,str,datetime,True
8,Purchase Orders,po_date,str,datetime,True
9,Purchase Orders,expected_date,str,datetime,True


### Date Parseability Check

Evaluate whether the raw string-based date fields contain invalid or unparseable date values.

This check does not modify the original columns. Temporary datetime conversion is used only to identify invalid date records.

In [11]:
# Check whether raw date values can be parsed successfully

date_parse_results = []

for table_name, columns in expected_date_columns.items():

    df = datasets[table_name]

    for column in columns:

        original_non_null = df[column].notna()

        parsed_dates = pd.to_datetime(
            df[column],
            errors="coerce"
        )

        invalid_mask = original_non_null & parsed_dates.isna()
        invalid_count = invalid_mask.sum()

        date_parse_results.append({
            "Dataset": table_name,
            "Column": column,
            "Non_Null_Values": original_non_null.sum(),
            "Invalid_Date_Values": invalid_count,
            "Invalid_Percentage": round(
                (invalid_count / original_non_null.sum()) * 100, 2
            ) if original_non_null.sum() > 0 else 0
        })

date_parse_summary = pd.DataFrame(date_parse_results)

display(date_parse_summary)

,Dataset,Column,Non_Null_Values,Invalid_Date_Values,Invalid_Percentage
0,Customers,customer_since,2500,0,0.0
1,Sales Orders,order_date,25000,0,0.0
2,Sales Orders,requested_delivery_date,25000,0,0.0
3,Sales Order Items,order_date,61397,0,0.0
4,Inventory Transactions,transaction_date,110045,0,0.0
5,Shipments,shipment_date,24798,0,0.0
6,Shipments,requested_delivery_date,24798,0,0.0
7,Shipments,actual_delivery_date,24798,0,0.0
8,Purchase Orders,po_date,8500,0,0.0
9,Purchase Orders,expected_date,8500,0,0.0


## 6. Chronological Date Consistency Audit

Evaluate whether related business dates follow logically valid chronological sequences.

Temporary datetime conversion is used for validation only. The original raw datasets remain unchanged.

In [12]:
# Chronological date consistency checks

# Temporary date conversion for audit purposes only
so_order_date = pd.to_datetime(sales_orders["order_date"], errors="coerce")
so_requested_date = pd.to_datetime(
    sales_orders["requested_delivery_date"], errors="coerce"
)

ship_date = pd.to_datetime(shipments["shipment_date"], errors="coerce")
ship_requested = pd.to_datetime(
    shipments["requested_delivery_date"], errors="coerce"
)
ship_actual = pd.to_datetime(
    shipments["actual_delivery_date"], errors="coerce"
)

po_date = pd.to_datetime(purchase_orders["po_date"], errors="coerce")
po_expected = pd.to_datetime(
    purchase_orders["expected_date"], errors="coerce"
)
po_actual = pd.to_datetime(
    purchase_orders["actual_receipt_date"], errors="coerce"
)

chronology_results = pd.DataFrame([
    {
        "Check": "Requested delivery before order date",
        "Invalid_Rows": (so_requested_date < so_order_date).sum()
    },
    {
        "Check": "Shipment before order requested delivery context",
        "Invalid_Rows": (ship_date < pd.to_datetime(
            shipments["requested_delivery_date"], errors="coerce"
        )).sum()
    },
    {
        "Check": "Actual delivery before shipment date",
        "Invalid_Rows": (ship_actual < ship_date).sum()
    },
    {
        "Check": "PO expected date before PO date",
        "Invalid_Rows": (po_expected < po_date).sum()
    },
    {
        "Check": "Actual receipt before PO date",
        "Invalid_Rows": (po_actual < po_date).sum()
    }
])

display(chronology_results)

,Check,Invalid_Rows
0,Requested delivery before order date,0
1,Shipment before order requested delivery context,19105
2,Actual delivery before shipment date,4467
3,PO expected date before PO date,0
4,Actual receipt before PO date,0


In [13]:
# Investigate records where actual delivery date is before shipment date

delivery_before_shipment = shipments[
    ship_actual < ship_date
].copy()

print(
    "Records with actual delivery before shipment:",
    f"{len(delivery_before_shipment):,}"
)

display(
    delivery_before_shipment[
        [
            "shipment_id",
            "order_id",
            "shipment_date",
            "requested_delivery_date",
            "actual_delivery_date",
            "delivery_delay_days",
            "on_time_flag",
            "otif_flag"
        ]
    ].head(10)
)

Records with actual delivery before shipment: 4,467


,shipment_id,order_id,shipment_date,requested_delivery_date,actual_delivery_date,delivery_delay_days,on_time_flag,otif_flag
3,SHP000004,SO000004,2025-11-18,2025-11-17,2025-11-16,-1,Y,N
4,SHP000005,SO000005,2026-07-11,2026-07-09,2026-07-08,-1,Y,N
7,SHP000008,SO000008,2025-09-17,2025-09-14,2025-09-12,-2,Y,Y
13,SHP000014,SO000014,2024-07-04,2024-06-30,2024-06-29,-1,Y,Y
16,SHP000017,SO000018,2025-10-13,2025-10-11,2025-10-10,-1,Y,Y
17,SHP000018,SO000019,2024-12-04,2024-12-02,2024-12-02,0,Y,Y
52,SHP000053,SO000055,2026-02-12,2026-02-12,2026-02-11,-1,Y,Y
55,SHP000056,SO000058,2026-08-05,2026-08-05,2026-08-03,-2,Y,Y
58,SHP000059,SO000061,2024-10-24,2024-10-24,2024-10-23,-1,Y,N
60,SHP000061,SO000063,2025-12-29,2025-12-26,2025-12-26,0,Y,Y


In [14]:
# Calculate date difference for affected records

date_difference = (
    pd.to_datetime(delivery_before_shipment["shipment_date"])
    - pd.to_datetime(delivery_before_shipment["actual_delivery_date"])
).dt.days

print("\nDays actual delivery precedes shipment:")
print(date_difference.describe())


Days actual delivery precedes shipment:
count    4467.000000
mean        2.366913
std         1.304135
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         6.000000
dtype: float64


## 7. Quantity Validity Audit

Evaluate important quantity fields for negative values and other potentially invalid numerical conditions.

Negative quantities are assessed according to business context. For example, negative inventory transaction quantities may legitimately represent outbound inventory movements, while negative ordered or inventory-on-hand quantities may indicate data-quality problems.

In [15]:
# Audit quantity fields for negative values

quantity_checks = {
    "Sales Orders": [
        "ordered_qty",
        "fulfilled_qty",
        "backlog_qty"
    ],
    "Sales Order Items": [
        "ordered_qty",
        "fulfilled_qty",
        "backlog_qty"
    ],
    "Inventory": [
        "on_hand_qty",
        "allocated_qty",
        "available_qty",
        "safety_stock",
        "reorder_point"
    ],
    "Shipments": [
        "shipped_qty"
    ],
    "Purchase Orders": [
        "ordered_qty",
        "received_qty"
    ]
}

quantity_results = []

for table_name, columns in quantity_checks.items():

    df = datasets[table_name]

    for column in columns:

        quantity_results.append({
            "Dataset": table_name,
            "Column": column,
            "Min_Value": df[column].min(),
            "Zero_Count": (df[column] == 0).sum(),
            "Negative_Count": (df[column] < 0).sum()
        })

quantity_summary = pd.DataFrame(quantity_results)

display(quantity_summary)

,Dataset,Column,Min_Value,Zero_Count,Negative_Count
0,Sales Orders,ordered_qty,1,0,0
1,Sales Orders,fulfilled_qty,0,202,0
2,Sales Orders,backlog_qty,0,14150,0
3,Sales Order Items,ordered_qty,1,0,0
4,Sales Order Items,fulfilled_qty,0,2777,0
5,Sales Order Items,backlog_qty,0,48119,0
6,Inventory,on_hand_qty,0,219,0
7,Inventory,allocated_qty,0,625,0
8,Inventory,available_qty,0,240,0
9,Inventory,safety_stock,1,0,0


### Inventory Transaction Quantity Validation

Inventory transaction quantities require business-context validation because positive and negative movements may represent different transaction types.

The relationship between `transaction_type` and the sign of `quantity` is therefore evaluated separately.

In [16]:
# Review transaction types and quantity behavior

transaction_quantity_summary = (
    inventory_transactions
    .groupby("transaction_type")
    .agg(
        Transaction_Count=("transaction_id", "count"),
        Min_Quantity=("quantity", "min"),
        Max_Quantity=("quantity", "max"),
        Zero_Count=("quantity", lambda x: (x == 0).sum()),
        Positive_Count=("quantity", lambda x: (x > 0).sum()),
        Negative_Count=("quantity", lambda x: (x < 0).sum())
    )
    .reset_index()
)

display(transaction_quantity_summary)

,transaction_type,Transaction_Count,Min_Quantity,Max_Quantity,Zero_Count,Positive_Count,Negative_Count
0,ADJUSTMENT_IN,5511,1,39,0,5511,0
1,ADJUSTMENT_OUT,5504,-39,-1,0,0,5504
2,PO_RECEIPT,27423,1,39,0,27423,0
3,RETURN_IN,8759,1,39,0,8759,0
4,SALE_ISSUE,62848,-39,-1,0,0,62848


## 8. Price, Cost, Discount & Monetary Value Audit

Evaluate pricing, cost, discount, and monetary-value fields for invalid values.

Business rules include:

- Unit prices and unit costs should not be negative.
- Order, backlog, inventory, and purchase-order values should not be negative.
- Discount percentages should normally remain between 0% and 100%.
- Zero monetary values are reviewed separately because they may be valid under specific business conditions.

In [17]:
# Audit price, cost, and monetary fields

financial_fields = {
    "Products": [
        "unit_cost",
        "unit_price"
    ],
    "Sales Orders": [
        "order_value",
        "backlog_value"
    ],
    "Sales Order Items": [
        "unit_price",
        "gross_value",
        "net_order_value",
        "backlog_value"
    ],
    "Inventory": [
        "unit_cost",
        "inventory_value"
    ],
    "Purchase Orders": [
        "unit_cost",
        "po_value"
    ]
}

financial_results = []

for table_name, columns in financial_fields.items():

    df = datasets[table_name]

    for column in columns:

        financial_results.append({
            "Dataset": table_name,
            "Column": column,
            "Min_Value": df[column].min(),
            "Max_Value": df[column].max(),
            "Zero_Count": (df[column] == 0).sum(),
            "Negative_Count": (df[column] < 0).sum()
        })

financial_summary = pd.DataFrame(financial_results)

display(financial_summary)

,Dataset,Column,Min_Value,Max_Value,Zero_Count,Negative_Count
0,Products,unit_cost,107.64,6492.49,0,0
1,Products,unit_price,130.81,9481.31,0,0
2,Sales Orders,order_value,364.79,861989.94,0,0
3,Sales Orders,backlog_value,0.00,719326.18,14150,0
4,Sales Order Items,unit_price,130.81,9481.31,0,0
5,Sales Order Items,gross_value,132.17,749987.42,0,0
6,Sales Order Items,net_order_value,121.60,727524.48,0,0
7,Sales Order Items,backlog_value,0.00,712488.05,48119,0
8,Inventory,unit_cost,107.64,6492.49,0,0
9,Inventory,inventory_value,0.00,1618866.55,219,0


In [18]:
# Validate discount percentages

discount_invalid = (
    (sales_order_items["discount_pct"] < 0)
    | (sales_order_items["discount_pct"] > 100)
)

print(
    "Invalid discount percentages:",
    f"{discount_invalid.sum():,}"
)

print(
    "Discount range:",
    sales_order_items["discount_pct"].min(),
    "to",
    sales_order_items["discount_pct"].max()
)

Invalid discount percentages: 0
Discount range: 0.0 to 0.15


In [19]:
# Validate discount as decimal percentage (0 to 1)

invalid_discount_scale = (
    (sales_order_items["discount_pct"] < 0)
    | (sales_order_items["discount_pct"] > 1)
)

print(
    "Invalid discounts outside 0-1 range:",
    invalid_discount_scale.sum()
)

print(
    "Observed discount range:",
    sales_order_items["discount_pct"].min(),
    "to",
    sales_order_items["discount_pct"].max()
)

print(
    "Observed discount percentage range:",
    f"{sales_order_items['discount_pct'].min() * 100:.0f}% to "
    f"{sales_order_items['discount_pct'].max() * 100:.0f}%"
)

Invalid discounts outside 0-1 range: 0
Observed discount range: 0.0 to 0.15
Observed discount percentage range: 0% to 15%


## 9. Sales Order Quantity Consistency Audit

Validate whether ordered, fulfilled, and backlog quantities follow the expected business relationship:

**Ordered Quantity = Fulfilled Quantity + Backlog Quantity**

The rule is evaluated independently at both the sales-order and sales-order-item levels.

In [20]:
# Validate quantity consistency

order_qty_invalid = (
    sales_orders["ordered_qty"]
    != sales_orders["fulfilled_qty"] + sales_orders["backlog_qty"]
)

item_qty_invalid = (
    sales_order_items["ordered_qty"]
    != sales_order_items["fulfilled_qty"] + sales_order_items["backlog_qty"]
)

quantity_consistency = pd.DataFrame([
    {
        "Dataset": "Sales Orders",
        "Rule": "ordered_qty = fulfilled_qty + backlog_qty",
        "Total_Rows": len(sales_orders),
        "Invalid_Rows": order_qty_invalid.sum(),
        "Invalid_Percentage": round(
            order_qty_invalid.mean() * 100, 2
        )
    },
    {
        "Dataset": "Sales Order Items",
        "Rule": "ordered_qty = fulfilled_qty + backlog_qty",
        "Total_Rows": len(sales_order_items),
        "Invalid_Rows": item_qty_invalid.sum(),
        "Invalid_Percentage": round(
            item_qty_invalid.mean() * 100, 2
        )
    }
])

display(quantity_consistency)

,Dataset,Rule,Total_Rows,Invalid_Rows,Invalid_Percentage
0,Sales Orders,ordered_qty = fulfilled_qty + backlog_qty,25000,0,0.0
1,Sales Order Items,ordered_qty = fulfilled_qty + backlog_qty,61397,0,0.0


## 10. Fill Rate Validation

Validate the pre-calculated order-level and line-level fill-rate fields against fulfilled and ordered quantities.

Expected business logic:

**Fill Rate (%) = (Fulfilled Quantity / Ordered Quantity) × 100**

A small numerical tolerance is used to account for rounding.

In [21]:
# Recalculate expected fill rates

expected_order_fill_rate = (
    sales_orders["fulfilled_qty"]
    / sales_orders["ordered_qty"]
    * 100
)

expected_line_fill_rate = (
    sales_order_items["fulfilled_qty"]
    / sales_order_items["ordered_qty"]
    * 100
)

# Compare using rounding tolerance
order_fill_invalid = ~np.isclose(
    sales_orders["order_fill_rate_pct"],
    expected_order_fill_rate,
    atol=0.01
)

line_fill_invalid = ~np.isclose(
    sales_order_items["line_fill_rate_pct"],
    expected_line_fill_rate,
    atol=0.01
)

fill_rate_validation = pd.DataFrame([
    {
        "Dataset": "Sales Orders",
        "Field": "order_fill_rate_pct",
        "Total_Rows": len(sales_orders),
        "Invalid_Rows": order_fill_invalid.sum(),
        "Invalid_Percentage": round(
            order_fill_invalid.mean() * 100, 2
        )
    },
    {
        "Dataset": "Sales Order Items",
        "Field": "line_fill_rate_pct",
        "Total_Rows": len(sales_order_items),
        "Invalid_Rows": line_fill_invalid.sum(),
        "Invalid_Percentage": round(
            line_fill_invalid.mean() * 100, 2
        )
    }
])

display(fill_rate_validation)

,Dataset,Field,Total_Rows,Invalid_Rows,Invalid_Percentage
0,Sales Orders,order_fill_rate_pct,25000,0,0.0
1,Sales Order Items,line_fill_rate_pct,61397,0,0.0


## 11. Backlog Value Validation

Validate whether the pre-calculated backlog value is consistent with backlog quantity, unit price, and applicable line-level discounts.

Expected line-level business logic:

**Backlog Value = Backlog Quantity × Unit Price × (1 - Discount Rate)**

A small tolerance is used to account for monetary rounding.

In [22]:
# Calculate expected line-level backlog value

expected_line_backlog_value = (
    sales_order_items["backlog_qty"]
    * sales_order_items["unit_price"]
    * (1 - sales_order_items["discount_pct"])
)

line_backlog_invalid = ~np.isclose(
    sales_order_items["backlog_value"],
    expected_line_backlog_value,
    atol=0.02
)

print(f"Total Sales Order Items : {len(sales_order_items):,}")
print(f"Invalid Backlog Values  : {line_backlog_invalid.sum():,}")
print(
    f"Invalid Percentage      : "
    f"{line_backlog_invalid.mean() * 100:.2f}%"
)

Total Sales Order Items : 61,397
Invalid Backlog Values  : 0
Invalid Percentage      : 0.00%


### Order-Level Backlog Value Reconciliation

Validate whether the order-level `backlog_value` equals the aggregated backlog value of all corresponding sales-order items.

Expected relationship:

**Order Backlog Value = Sum of Line-Level Backlog Values for the Order**

A small tolerance is applied for monetary rounding.

In [23]:
# Aggregate line-level backlog value by order

item_backlog_by_order = (
    sales_order_items
    .groupby("order_id", as_index=False)
    .agg(
        calculated_backlog_value=("backlog_value", "sum")
    )
)

# Compare with order-level backlog value

backlog_reconciliation = sales_orders[
    ["order_id", "backlog_value"]
].merge(
    item_backlog_by_order,
    on="order_id",
    how="left"
)

backlog_reconciliation["difference"] = (
    backlog_reconciliation["backlog_value"]
    - backlog_reconciliation["calculated_backlog_value"]
)

backlog_reconciliation["invalid_flag"] = ~np.isclose(
    backlog_reconciliation["backlog_value"],
    backlog_reconciliation["calculated_backlog_value"],
    atol=0.05
)

print(f"Total Sales Orders       : {len(backlog_reconciliation):,}")
print(
    f"Invalid Backlog Values   : "
    f"{backlog_reconciliation['invalid_flag'].sum():,}"
)
print(
    f"Invalid Percentage       : "
    f"{backlog_reconciliation['invalid_flag'].mean() * 100:.2f}%"
)

Total Sales Orders       : 25,000
Invalid Backlog Values   : 0
Invalid Percentage       : 0.00%


## 12. Order-to-Line Quantity Reconciliation

Validate whether order-level quantities reconcile with the aggregated quantities of their corresponding sales-order items.

Expected relationships:

- **Order Ordered Quantity = Sum of Line Ordered Quantities**
- **Order Fulfilled Quantity = Sum of Line Fulfilled Quantities**
- **Order Backlog Quantity = Sum of Line Backlog Quantities**

This ensures consistency between the order header and order-line datasets.

In [24]:
# Aggregate sales order item quantities to order level

item_qty_by_order = (
    sales_order_items
    .groupby("order_id", as_index=False)
    .agg(
        item_ordered_qty=("ordered_qty", "sum"),
        item_fulfilled_qty=("fulfilled_qty", "sum"),
        item_backlog_qty=("backlog_qty", "sum")
    )
)

# Merge with sales order header quantities

order_qty_reconciliation = sales_orders[
    [
        "order_id",
        "ordered_qty",
        "fulfilled_qty",
        "backlog_qty"
    ]
].merge(
    item_qty_by_order,
    on="order_id",
    how="left"
)

# Validate each quantity

ordered_invalid = (
    order_qty_reconciliation["ordered_qty"]
    != order_qty_reconciliation["item_ordered_qty"]
)

fulfilled_invalid = (
    order_qty_reconciliation["fulfilled_qty"]
    != order_qty_reconciliation["item_fulfilled_qty"]
)

backlog_invalid = (
    order_qty_reconciliation["backlog_qty"]
    != order_qty_reconciliation["item_backlog_qty"]
)

reconciliation_summary = pd.DataFrame([
    {
        "Measure": "Ordered Quantity",
        "Invalid_Rows": ordered_invalid.sum(),
        "Invalid_Percentage": round(ordered_invalid.mean() * 100, 2)
    },
    {
        "Measure": "Fulfilled Quantity",
        "Invalid_Rows": fulfilled_invalid.sum(),
        "Invalid_Percentage": round(fulfilled_invalid.mean() * 100, 2)
    },
    {
        "Measure": "Backlog Quantity",
        "Invalid_Rows": backlog_invalid.sum(),
        "Invalid_Percentage": round(backlog_invalid.mean() * 100, 2)
    }
])

display(reconciliation_summary)

,Measure,Invalid_Rows,Invalid_Percentage
0,Ordered Quantity,0,0.0
1,Fulfilled Quantity,0,0.0
2,Backlog Quantity,0,0.0


## 13. Order Value Reconciliation

Validate whether the order-level `order_value` reconciles with the aggregated `net_order_value` of its corresponding sales-order items.

Expected relationship:

**Order Value = Sum of Line-Level Net Order Values**

A small monetary tolerance is applied to account for rounding.

In [25]:
# Aggregate line-level net order value by order

item_value_by_order = (
    sales_order_items
    .groupby("order_id", as_index=False)
    .agg(
        calculated_order_value=("net_order_value", "sum")
    )
)

# Merge with order-level values

order_value_reconciliation = sales_orders[
    ["order_id", "order_value"]
].merge(
    item_value_by_order,
    on="order_id",
    how="left"
)

# Calculate difference

order_value_reconciliation["difference"] = (
    order_value_reconciliation["order_value"]
    - order_value_reconciliation["calculated_order_value"]
)

# Validate with rounding tolerance

order_value_reconciliation["invalid_flag"] = ~np.isclose(
    order_value_reconciliation["order_value"],
    order_value_reconciliation["calculated_order_value"],
    atol=0.05
)

print(f"Total Sales Orders      : {len(order_value_reconciliation):,}")
print(
    f"Invalid Order Values    : "
    f"{order_value_reconciliation['invalid_flag'].sum():,}"
)
print(
    f"Invalid Percentage      : "
    f"{order_value_reconciliation['invalid_flag'].mean() * 100:.2f}%"
)

Total Sales Orders      : 25,000
Invalid Order Values    : 0
Invalid Percentage      : 0.00%


## 14. Inventory Quantity Consistency Audit

Validate the relationship between on-hand, allocated, and available inventory quantities.

Expected business logic:

**Available Quantity = On-Hand Quantity - Allocated Quantity**

This check determines whether the pre-calculated `available_qty` field is internally consistent with the underlying inventory quantities.

In [26]:
# Validate available inventory quantity

expected_available_qty = (
    inventory["on_hand_qty"]
    - inventory["allocated_qty"]
)

available_qty_invalid = (
    inventory["available_qty"]
    != expected_available_qty
)

print(f"Total Inventory Records : {len(inventory):,}")
print(f"Invalid Available Qty   : {available_qty_invalid.sum():,}")
print(
    f"Invalid Percentage      : "
    f"{available_qty_invalid.mean() * 100:.2f}%"
)

Total Inventory Records : 3,000
Invalid Available Qty   : 0
Invalid Percentage      : 0.00%


## 15. Inventory Value Validation

Validate whether the pre-calculated inventory value is consistent with on-hand quantity and unit cost.

Expected business logic:

**Inventory Value = On-Hand Quantity × Unit Cost**

A small monetary tolerance is applied to account for rounding.

In [27]:
# Calculate expected inventory value

expected_inventory_value = (
    inventory["on_hand_qty"]
    * inventory["unit_cost"]
)

inventory_value_invalid = ~np.isclose(
    inventory["inventory_value"],
    expected_inventory_value,
    atol=0.05
)

print(f"Total Inventory Records : {len(inventory):,}")
print(f"Invalid Inventory Value : {inventory_value_invalid.sum():,}")
print(
    f"Invalid Percentage      : "
    f"{inventory_value_invalid.mean() * 100:.2f}%"
)

Total Inventory Records : 3,000
Invalid Inventory Value : 0
Invalid Percentage      : 0.00%


## 16. Safety Stock & Reorder Point Logic Audit

Evaluate whether inventory planning parameters follow logical relationships.

The primary rule evaluated is:

**Reorder Point >= Safety Stock**

A reorder point below safety stock may indicate an inconsistent inventory-planning parameter and requires further investigation.

In [28]:
# Validate relationship between reorder point and safety stock

reorder_below_safety = (
    inventory["reorder_point"]
    < inventory["safety_stock"]
)

print(f"Total Inventory Records          : {len(inventory):,}")
print(f"Reorder Point < Safety Stock     : {reorder_below_safety.sum():,}")
print(
    f"Invalid Percentage               : "
    f"{reorder_below_safety.mean() * 100:.2f}%"
)

print("\nPlanning parameter ranges:")
print(
    f"Safety Stock  : "
    f"{inventory['safety_stock'].min():,} to "
    f"{inventory['safety_stock'].max():,}"
)
print(
    f"Reorder Point : "
    f"{inventory['reorder_point'].min():,} to "
    f"{inventory['reorder_point'].max():,}"
)

Total Inventory Records          : 3,000
Reorder Point < Safety Stock     : 0
Invalid Percentage               : 0.00%

Planning parameter ranges:
Safety Stock  : 1 to 48
Reorder Point : 2 to 135


## 17. Stock Status Consistency Audit

Evaluate whether inventory stock-status classifications are consistent with the underlying available quantity, safety stock, and reorder-point values.

Before defining validation rules, the existing stock-status categories and their numerical inventory characteristics are reviewed.

In [29]:
# Review stock status against inventory planning measures

stock_status_profile = (
    inventory
    .groupby("stock_status")
    .agg(
        Record_Count=("product_id", "count"),
        Min_Available_Qty=("available_qty", "min"),
        Max_Available_Qty=("available_qty", "max"),
        Avg_Available_Qty=("available_qty", "mean"),
        Min_Safety_Stock=("safety_stock", "min"),
        Max_Safety_Stock=("safety_stock", "max"),
        Min_Reorder_Point=("reorder_point", "min"),
        Max_Reorder_Point=("reorder_point", "max")
    )
    .round(2)
    .reset_index()
)

display(stock_status_profile)

,stock_status,Record_Count,Min_Available_Qty,Max_Available_Qty,Avg_Available_Qty,Min_Safety_Stock,Max_Safety_Stock,Min_Reorder_Point,Max_Reorder_Point
0,Healthy,2077,1,224,25.46,1,48,2,135
1,Low Stock,205,1,23,4.89,2,32,4,95
2,Overstock,478,7,277,50.90,1,38,3,124
3,Stockout,240,0,0,0.00,1,44,2,105


### Stock Status Rule Investigation

Investigate how existing stock-status labels relate to each SKU-warehouse record's safety stock and reorder point.

This allows the classification logic to be inferred from the data before performing a formal consistency validation.

In [30]:
# Compare stock status with row-level inventory thresholds

stock_rule_profile = (
    inventory
    .assign(
        Available_Zero=inventory["available_qty"] == 0,
        Below_Safety=inventory["available_qty"] < inventory["safety_stock"],
        Below_Reorder=inventory["available_qty"] < inventory["reorder_point"],
        At_Or_Above_Reorder=inventory["available_qty"] >= inventory["reorder_point"]
    )
    .groupby("stock_status")
    .agg(
        Records=("product_id", "count"),
        Available_Zero=("Available_Zero", "sum"),
        Below_Safety=("Below_Safety", "sum"),
        Below_Reorder=("Below_Reorder", "sum"),
        At_Or_Above_Reorder=("At_Or_Above_Reorder", "sum")
    )
    .reset_index()
)

display(stock_rule_profile)

,stock_status,Records,Available_Zero,Below_Safety,Below_Reorder,At_Or_Above_Reorder
0,Healthy,2077,0,0,782,1295
1,Low Stock,205,0,205,205,0
2,Overstock,478,0,0,0,478
3,Stockout,240,240,240,240,0


### Healthy vs Overstock Classification Investigation

The previous analysis indicates that reorder point alone does not distinguish Healthy inventory from Overstock inventory.

Additional inventory ratios are therefore reviewed to understand the underlying classification pattern before validating the existing stock-status field.

In [31]:
# Investigate Healthy vs Overstock classification

stock_status_investigation = inventory.copy()

stock_status_investigation["available_to_reorder_ratio"] = (
    stock_status_investigation["available_qty"]
    / stock_status_investigation["reorder_point"]
)

stock_status_investigation["available_days_supply"] = (
    stock_status_investigation["available_qty"]
    / stock_status_investigation["avg_daily_demand"]
)

stock_ratio_profile = (
    stock_status_investigation
    .groupby("stock_status")
    .agg(
        Records=("product_id", "count"),

        Min_Available_Reorder_Ratio=(
            "available_to_reorder_ratio", "min"
        ),
        Avg_Available_Reorder_Ratio=(
            "available_to_reorder_ratio", "mean"
        ),
        Max_Available_Reorder_Ratio=(
            "available_to_reorder_ratio", "max"
        ),

        Min_Days_Supply=("available_days_supply", "min"),
        Avg_Days_Supply=("available_days_supply", "mean"),
        Max_Days_Supply=("available_days_supply", "max")
    )
    .round(2)
    .reset_index()
)

display(stock_ratio_profile)

,stock_status,Records,Min_Available_Reorder_Ratio,Avg_Available_Reorder_Ratio,Max_Available_Reorder_Ratio,Min_Days_Supply,Avg_Days_Supply,Max_Days_Supply
0,Healthy,2077,0.20,1.17,2.00,5.04,31.95,88.89
1,Low Stock,205,0.03,0.21,0.44,0.82,5.61,11.70
2,Overstock,478,2.02,2.27,2.67,33.33,60.74,109.09
3,Stockout,240,0.00,0.00,0.00,0.00,0.00,0.00


### Stock Status Classification Validation

Based on the observed inventory patterns, the following stock-status logic is evaluated:

- **Stockout:** Available Quantity = 0
- **Low Stock:** Available Quantity > 0 and below Safety Stock
- **Overstock:** Available Quantity > 2 × Reorder Point
- **Healthy:** Remaining inventory positions

The inferred business rule is compared with the existing `stock_status` field to assess classification consistency.

In [32]:
# Reconstruct expected stock status using inferred business rules

conditions = [
    inventory["available_qty"] == 0,

    (inventory["available_qty"] > 0)
    & (inventory["available_qty"] < inventory["safety_stock"]),

    inventory["available_qty"] > (
        2 * inventory["reorder_point"]
    )
]

choices = [
    "Stockout",
    "Low Stock",
    "Overstock"
]

expected_stock_status = np.select(
    conditions,
    choices,
    default="Healthy"
)

stock_status_invalid = (
    inventory["stock_status"]
    != expected_stock_status
)

print(f"Total Inventory Records  : {len(inventory):,}")
print(f"Invalid Stock Status     : {stock_status_invalid.sum():,}")
print(
    f"Invalid Percentage       : "
    f"{stock_status_invalid.mean() * 100:.2f}%"
)

Total Inventory Records  : 3,000
Invalid Stock Status     : 0
Invalid Percentage       : 0.00%


## 18. Stockout Flag Consistency Audit

Evaluate the existing sales-order-item `stockout_flag` and investigate its relationship with fulfillment and backlog quantities.

The existing flag logic is first profiled before applying any validation rule.

In [33]:
# Profile stockout flag against fulfillment and backlog

stockout_flag_profile = (
    sales_order_items
    .groupby("stockout_flag")
    .agg(
        Line_Count=("order_item_id", "count"),
        Total_Ordered_Qty=("ordered_qty", "sum"),
        Total_Fulfilled_Qty=("fulfilled_qty", "sum"),
        Total_Backlog_Qty=("backlog_qty", "sum"),
        Lines_With_Backlog=(
            "backlog_qty",
            lambda x: (x > 0).sum()
        ),
        Lines_Without_Backlog=(
            "backlog_qty",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

display(stockout_flag_profile)

,stockout_flag,Line_Count,Total_Ordered_Qty,Total_Fulfilled_Qty,Total_Backlog_Qty,Lines_With_Backlog,Lines_Without_Backlog
0,N,51151,418940,411731,7209,3032,48119
1,Y,10246,82815,28845,53970,10246,0


### Stockout Flag Logic Investigation

Investigate whether the sales-order-item stockout flag corresponds to the current inventory position of the ordered product at the associated warehouse.

The analysis compares stockout flags with inventory availability and stock status without modifying either dataset.

In [34]:
# Join order items with product-warehouse inventory position

stockout_investigation = sales_order_items[
    [
        "order_item_id",
        "product_id",
        "warehouse_id",
        "ordered_qty",
        "fulfilled_qty",
        "backlog_qty",
        "stockout_flag"
    ]
].merge(
    inventory[
        [
            "product_id",
            "warehouse_id",
            "available_qty",
            "safety_stock",
            "reorder_point",
            "stock_status"
        ]
    ],
    on=["product_id", "warehouse_id"],
    how="left"
)

# Profile inventory status by stockout flag

stockout_inventory_profile = (
    stockout_investigation
    .groupby(["stockout_flag", "stock_status"])
    .agg(
        Line_Count=("order_item_id", "count"),
        Avg_Available_Qty=("available_qty", "mean"),
        Lines_With_Backlog=(
            "backlog_qty",
            lambda x: (x > 0).sum()
        )
    )
    .round(2)
    .reset_index()
)

display(stockout_inventory_profile)

,stockout_flag,stock_status,Line_Count,Avg_Available_Qty,Lines_With_Backlog
0,N,Healthy,35067,37.60,2098
1,N,Low Stock,3810,6.25,218
2,N,Overstock,8363,66.47,498
3,N,Stockout,3911,0.00,218
4,Y,Healthy,7031,50.52,7031
5,Y,Low Stock,735,8.03,735
6,Y,Overstock,1664,82.88,1664
7,Y,Stockout,816,0.00,816


### Stockout Flag vs Fulfillment Severity

Since the current inventory snapshot cannot directly validate historical stockout events, the stockout flag is further evaluated against line-level fulfillment performance and backlog severity.

In [35]:
# Compare stockout flag with fulfillment severity

stockout_severity = (
    sales_order_items
    .groupby("stockout_flag")
    .agg(
        Line_Count=("order_item_id", "count"),
        Avg_Ordered_Qty=("ordered_qty", "mean"),
        Avg_Fulfilled_Qty=("fulfilled_qty", "mean"),
        Avg_Backlog_Qty=("backlog_qty", "mean"),
        Avg_Fill_Rate=("line_fill_rate_pct", "mean"),
        Min_Fill_Rate=("line_fill_rate_pct", "min"),
        Max_Fill_Rate=("line_fill_rate_pct", "max"),
        Fully_Unfulfilled_Lines=(
            "fulfilled_qty",
            lambda x: (x == 0).sum()
        )
    )
    .round(2)
    .reset_index()
)

display(stockout_severity)

,stockout_flag,Line_Count,Avg_Ordered_Qty,Avg_Fulfilled_Qty,Avg_Backlog_Qty,Avg_Fill_Rate,Min_Fill_Rate,Max_Fill_Rate,Fully_Unfulfilled_Lines
0,N,51151,8.19,8.05,0.14,98.10,0.0,100.0,20
1,Y,10246,8.08,2.82,5.27,33.09,0.0,75.0,2757


## 19. Backlog Status Consistency Audit

Evaluate whether the existing line-level `backlog_status` classifications are consistent with backlog quantity and fulfillment performance.

The current classifications are first profiled before defining a formal validation rule.

In [36]:
# Profile backlog status against backlog and fulfillment quantities

backlog_status_profile = (
    sales_order_items
    .groupby("backlog_status")
    .agg(
        Line_Count=("order_item_id", "count"),
        Min_Backlog_Qty=("backlog_qty", "min"),
        Max_Backlog_Qty=("backlog_qty", "max"),
        Avg_Backlog_Qty=("backlog_qty", "mean"),
        Min_Fill_Rate=("line_fill_rate_pct", "min"),
        Max_Fill_Rate=("line_fill_rate_pct", "max"),
        Avg_Fill_Rate=("line_fill_rate_pct", "mean"),
        Zero_Backlog_Lines=(
            "backlog_qty",
            lambda x: (x == 0).sum()
        ),
        Positive_Backlog_Lines=(
            "backlog_qty",
            lambda x: (x > 0).sum()
        )
    )
    .round(2)
    .reset_index()
)

display(backlog_status_profile)

,backlog_status,Line_Count,Min_Backlog_Qty,Max_Backlog_Qty,Avg_Backlog_Qty,Min_Fill_Rate,Max_Fill_Rate,Avg_Fill_Rate,Zero_Backlog_Lines,Positive_Backlog_Lines
0,Cancelled,1349,1,72,4.73,0.0,75.0,41.64,0,1349
1,Later Fulfilled,6365,1,96,4.63,0.0,75.0,40.61,0,6365
2,No Backlog,48119,0,0,0.00,100.0,100.0,100.00,48119,0
3,Open Backlog,5564,1,91,4.55,0.0,75.0,41.40,0,5564


### Core Backlog Status Validation

The detailed backlog lifecycle categories cannot be reconstructed solely from the current backlog quantity.

However, the fundamental classification rule can be validated:

- **No Backlog** should have zero backlog quantity.
- **Cancelled, Later Fulfilled, and Open Backlog** should have positive backlog quantity.

In [37]:
# Validate core backlog status logic

invalid_no_backlog = (
    (sales_order_items["backlog_status"] == "No Backlog")
    & (sales_order_items["backlog_qty"] != 0)
)

invalid_positive_backlog = (
    (sales_order_items["backlog_status"] != "No Backlog")
    & (sales_order_items["backlog_qty"] <= 0)
)

backlog_status_validation = pd.DataFrame([
    {
        "Rule": "No Backlog must have backlog_qty = 0",
        "Invalid_Rows": invalid_no_backlog.sum()
    },
    {
        "Rule": "Other statuses must have backlog_qty > 0",
        "Invalid_Rows": invalid_positive_backlog.sum()
    }
])

display(backlog_status_validation)

print(
    "\nTotal inconsistent backlog-status records:",
    f"{(invalid_no_backlog | invalid_positive_backlog).sum():,}"
)

,Rule,Invalid_Rows
0,No Backlog must have backlog_qty = 0,0
1,Other statuses must have backlog_qty > 0,0



Total inconsistent backlog-status records: 0


## 20. Sales Order Status Consistency Audit

Evaluate how order-level status classifications relate to ordered, fulfilled, and backlog quantities.

The existing status categories are first profiled before formal business-rule validation is performed.

In [38]:
# Profile order status against fulfillment and backlog

order_status_profile = (
    sales_orders
    .groupby("order_status")
    .agg(
        Order_Count=("order_id", "count"),

        Min_Ordered_Qty=("ordered_qty", "min"),
        Max_Ordered_Qty=("ordered_qty", "max"),

        Min_Fulfilled_Qty=("fulfilled_qty", "min"),
        Max_Fulfilled_Qty=("fulfilled_qty", "max"),

        Min_Backlog_Qty=("backlog_qty", "min"),
        Max_Backlog_Qty=("backlog_qty", "max"),

        Avg_Fill_Rate=("order_fill_rate_pct", "mean"),

        Zero_Backlog_Orders=(
            "backlog_qty",
            lambda x: (x == 0).sum()
        ),

        Positive_Backlog_Orders=(
            "backlog_qty",
            lambda x: (x > 0).sum()
        ),

        Zero_Fulfilled_Orders=(
            "fulfilled_qty",
            lambda x: (x == 0).sum()
        )
    )
    .round(2)
    .reset_index()
)

display(order_status_profile)

,order_status,Order_Count,Min_Ordered_Qty,Max_Ordered_Qty,Min_Fulfilled_Qty,Max_Fulfilled_Qty,Min_Backlog_Qty,Max_Backlog_Qty,Avg_Fill_Rate,Zero_Backlog_Orders,Positive_Backlog_Orders,Zero_Fulfilled_Orders
0,Backordered,198,1,80,0,0,1,80,0.00,0,198,198
1,Cancelled,275,3,106,0,94,1,92,70.95,0,275,4
2,Fulfilled,14150,1,158,1,158,0,0,100.00,14150,0,0
3,Partially Fulfilled,10377,2,148,1,141,1,112,72.43,0,10377,0


### Core Sales Order Status Validation

Based on the observed patterns, the following quantity-driven status rules can be validated:

- **Fulfilled:** Backlog Quantity = 0 and Fulfilled Quantity = Ordered Quantity
- **Backordered:** Fulfilled Quantity = 0 and Backlog Quantity = Ordered Quantity
- **Partially Fulfilled:** Fulfilled Quantity > 0 and Backlog Quantity > 0

`Cancelled` is treated separately because cancellation is a business lifecycle event and cannot be reliably inferred from quantities alone.

In [39]:
# Validate quantity-driven sales order statuses

fulfilled_invalid = (
    (sales_orders["order_status"] == "Fulfilled")
    & (
        (sales_orders["backlog_qty"] != 0)
        | (sales_orders["fulfilled_qty"] != sales_orders["ordered_qty"])
    )
)

backordered_invalid = (
    (sales_orders["order_status"] == "Backordered")
    & (
        (sales_orders["fulfilled_qty"] != 0)
        | (sales_orders["backlog_qty"] != sales_orders["ordered_qty"])
    )
)

partial_invalid = (
    (sales_orders["order_status"] == "Partially Fulfilled")
    & (
        (sales_orders["fulfilled_qty"] <= 0)
        | (sales_orders["backlog_qty"] <= 0)
    )
)

order_status_validation = pd.DataFrame([
    {
        "Status": "Fulfilled",
        "Invalid_Rows": fulfilled_invalid.sum()
    },
    {
        "Status": "Backordered",
        "Invalid_Rows": backordered_invalid.sum()
    },
    {
        "Status": "Partially Fulfilled",
        "Invalid_Rows": partial_invalid.sum()
    }
])

display(order_status_validation)

print(
    "\nTotal quantity-driven status inconsistencies:",
    (
        fulfilled_invalid
        | backordered_invalid
        | partial_invalid
    ).sum()
)

,Status,Invalid_Rows
0,Fulfilled,0
1,Backordered,0
2,Partially Fulfilled,0



Total quantity-driven status inconsistencies: 0


## 21. Shipment Delivery Delay Validation

Validate the pre-calculated `delivery_delay_days` field against requested and actual delivery dates.

Expected business logic:

**Delivery Delay Days = Actual Delivery Date - Requested Delivery Date**

Interpretation:

- Negative value = Early delivery
- Zero = Delivered on requested date
- Positive value = Late delivery

In [40]:
# Recalculate shipment delivery delay

requested_date = pd.to_datetime(
    shipments["requested_delivery_date"],
    errors="coerce"
)

actual_date = pd.to_datetime(
    shipments["actual_delivery_date"],
    errors="coerce"
)

expected_delivery_delay = (
    actual_date - requested_date
).dt.days

delivery_delay_invalid = (
    shipments["delivery_delay_days"]
    != expected_delivery_delay
)

print(f"Total Shipment Records   : {len(shipments):,}")
print(f"Invalid Delivery Delays  : {delivery_delay_invalid.sum():,}")
print(
    f"Invalid Percentage       : "
    f"{delivery_delay_invalid.mean() * 100:.2f}%"
)

print("\nDelivery Delay Range:")
print(
    f"{shipments['delivery_delay_days'].min()} "
    f"to {shipments['delivery_delay_days'].max()} days"
)

Total Shipment Records   : 24,798
Invalid Delivery Delays  : 0
Invalid Percentage       : 0.00%

Delivery Delay Range:
-2 to 10 days


### On-Time Delivery Flag Validation

Validate whether the existing `on_time_flag` is consistent with the calculated delivery-delay measure.

Expected business logic:

- **Y:** Delivery Delay Days <= 0
- **N:** Delivery Delay Days > 0

Early deliveries and deliveries made exactly on the requested date are therefore considered on time.

In [41]:
# Reconstruct expected on-time flag

expected_on_time_flag = np.where(
    shipments["delivery_delay_days"] <= 0,
    "Y",
    "N"
)

on_time_invalid = (
    shipments["on_time_flag"] != expected_on_time_flag
)

print(f"Total Shipment Records : {len(shipments):,}")
print(f"Invalid On-Time Flags  : {on_time_invalid.sum():,}")
print(
    f"Invalid Percentage     : "
    f"{on_time_invalid.mean() * 100:.2f}%"
)

print("\nExisting On-Time Flag Distribution:")
print(shipments["on_time_flag"].value_counts())

Total Shipment Records : 24,798
Invalid On-Time Flags  : 0
Invalid Percentage     : 0.00%

Existing On-Time Flag Distribution:
on_time_flag
Y    18869
N     5929
Name: count, dtype: int64


### OTIF Flag Investigation

Investigate the existing `otif_flag` to understand how On-Time-In-Full performance is represented in the shipment data.

The relationship between OTIF status, on-time delivery, and shipped quantity is profiled before defining a formal validation rule.

In [42]:
# Profile OTIF flag against on-time performance

otif_profile = (
    shipments
    .groupby(["otif_flag", "on_time_flag"])
    .agg(
        Shipment_Count=("shipment_id", "count"),
        Min_Shipped_Qty=("shipped_qty", "min"),
        Max_Shipped_Qty=("shipped_qty", "max"),
        Avg_Shipped_Qty=("shipped_qty", "mean")
    )
    .round(2)
    .reset_index()
)

display(otif_profile)

print("\nOTIF Distribution:")
print(shipments["otif_flag"].value_counts())

,otif_flag,on_time_flag,Shipment_Count,Min_Shipped_Qty,Max_Shipped_Qty,Avg_Shipped_Qty
0,N,N,5929,1,158,17.39
1,N,Y,6657,1,141,16.92
2,Y,Y,12212,1,158,18.41



OTIF Distribution:
otif_flag
N    12586
Y    12212
Name: count, dtype: int64


### OTIF In-Full Component Investigation

OTIF requires both timely delivery and complete fulfillment.

To understand the existing OTIF logic, shipment quantities are compared with the corresponding sales-order quantities before a formal OTIF validation rule is applied.

In [43]:
# Join shipments with sales order quantities

otif_investigation = shipments[
    [
        "shipment_id",
        "order_id",
        "shipped_qty",
        "on_time_flag",
        "otif_flag"
    ]
].merge(
    sales_orders[
        [
            "order_id",
            "ordered_qty",
            "fulfilled_qty",
            "backlog_qty"
        ]
    ],
    on="order_id",
    how="left"
)

# Determine whether shipment was in full

otif_investigation["in_full_flag"] = np.where(
    otif_investigation["shipped_qty"]
    >= otif_investigation["ordered_qty"],
    "Y",
    "N"
)

# Profile OTIF against On-Time and In-Full conditions

otif_logic_profile = (
    otif_investigation
    .groupby(
        ["otif_flag", "on_time_flag", "in_full_flag"]
    )
    .size()
    .reset_index(name="Shipment_Count")
)

display(otif_logic_profile)

,otif_flag,on_time_flag,in_full_flag,Shipment_Count
0,N,N,N,3991
1,N,N,Y,1938
2,N,Y,N,6657
3,Y,Y,Y,12212


### OTIF Classification Validation

Based on the observed shipment patterns, OTIF is defined as:

**OTIF = On Time AND In Full**

Where:

- **On Time:** `on_time_flag = Y`
- **In Full:** `shipped_qty >= ordered_qty`

The inferred OTIF rule is compared with the existing `otif_flag` to verify classification consistency.

In [44]:
# Reconstruct expected OTIF flag

expected_otif_flag = np.where(
    (otif_investigation["on_time_flag"] == "Y")
    & (
        otif_investigation["shipped_qty"]
        >= otif_investigation["ordered_qty"]
    ),
    "Y",
    "N"
)

otif_invalid = (
    otif_investigation["otif_flag"]
    != expected_otif_flag
)

print(
    f"Total Shipment Records : "
    f"{len(otif_investigation):,}"
)

print(
    f"Invalid OTIF Flags     : "
    f"{otif_invalid.sum():,}"
)

print(
    f"Invalid Percentage     : "
    f"{otif_invalid.mean() * 100:.2f}%"
)

Total Shipment Records : 24,798
Invalid OTIF Flags     : 0
Invalid Percentage     : 0.00%


## 22. Purchase Order Quantity Consistency Audit

Evaluate whether purchase-order receipt quantities are consistent with ordered quantities.

The following core business rule is assessed:

**Received Quantity <= Ordered Quantity**

Purchase-order quantities are also profiled by PO status to understand the relationship between receipt completion and the existing status classification.

In [45]:
# Validate purchase order receipt quantities

over_received = (
    purchase_orders["received_qty"]
    > purchase_orders["ordered_qty"]
)

print(f"Total Purchase Orders   : {len(purchase_orders):,}")
print(f"Over-Received Orders    : {over_received.sum():,}")
print(
    f"Invalid Percentage      : "
    f"{over_received.mean() * 100:.2f}%"
)

# Profile quantities by PO status

po_quantity_profile = (
    purchase_orders
    .groupby("po_status")
    .agg(
        PO_Count=("po_id", "count"),
        Min_Ordered_Qty=("ordered_qty", "min"),
        Max_Ordered_Qty=("ordered_qty", "max"),
        Min_Received_Qty=("received_qty", "min"),
        Max_Received_Qty=("received_qty", "max"),
        Zero_Received_PO=(
            "received_qty",
            lambda x: (x == 0).sum()
        ),
        Fully_Received_PO=(
            "received_qty",
            lambda x: (
                x.index.to_series().map(
                    purchase_orders["received_qty"]
                )
                == x.index.to_series().map(
                    purchase_orders["ordered_qty"]
                )
            ).sum()
        )
    )
    .reset_index()
)

display(po_quantity_profile)

Total Purchase Orders   : 8,500
Over-Received Orders    : 0
Invalid Percentage      : 0.00%


,po_status,PO_Count,Min_Ordered_Qty,Max_Ordered_Qty,Min_Received_Qty,Max_Received_Qty,Zero_Received_PO,Fully_Received_PO
0,Open,30,27,447,0,0,30,0
1,Partially Received,8470,20,449,17,448,0,0


### Purchase Order Status Validation

Validate whether purchase-order status is consistent with receipt quantities.

Observed business logic:

- **Open:** Received Quantity = 0
- **Partially Received:** Received Quantity > 0 and Received Quantity < Ordered Quantity

The existing PO status is compared with these quantity conditions.

In [46]:
# Validate purchase order status against receipt quantities

open_invalid = (
    (purchase_orders["po_status"] == "Open")
    & (purchase_orders["received_qty"] != 0)
)

partial_invalid = (
    (purchase_orders["po_status"] == "Partially Received")
    & (
        (purchase_orders["received_qty"] <= 0)
        | (
            purchase_orders["received_qty"]
            >= purchase_orders["ordered_qty"]
        )
    )
)

po_status_validation = pd.DataFrame([
    {
        "Status": "Open",
        "Invalid_Rows": open_invalid.sum()
    },
    {
        "Status": "Partially Received",
        "Invalid_Rows": partial_invalid.sum()
    }
])

display(po_status_validation)

print(
    "\nTotal PO status inconsistencies:",
    (open_invalid | partial_invalid).sum()
)

,Status,Invalid_Rows
0,Open,0
1,Partially Received,0



Total PO status inconsistencies: 0


## 23. Supplier Delay Validation

Validate whether the pre-calculated `supplier_delay_days` field is consistent with expected and actual purchase-order receipt dates.

Expected business logic:

**Supplier Delay Days = Actual Receipt Date - Expected Date**

Interpretation:

- Negative value = Received early
- Zero = Received on expected date
- Positive value = Received late
- Missing actual receipt date = Supplier delay remains unavailable

In [47]:
# Convert dates temporarily for validation

po_expected_date = pd.to_datetime(
    purchase_orders["expected_date"],
    errors="coerce"
)

po_actual_receipt = pd.to_datetime(
    purchase_orders["actual_receipt_date"],
    errors="coerce"
)

# Calculate expected supplier delay

expected_supplier_delay = (
    po_actual_receipt - po_expected_date
).dt.days

# Compare only rows where actual receipt exists

valid_receipt_mask = po_actual_receipt.notna()

supplier_delay_invalid = (
    purchase_orders.loc[
        valid_receipt_mask,
        "supplier_delay_days"
    ]
    != expected_supplier_delay[valid_receipt_mask]
)

print(f"Total Purchase Orders       : {len(purchase_orders):,}")
print(f"POs With Actual Receipt     : {valid_receipt_mask.sum():,}")
print(f"POs Without Actual Receipt  : {(~valid_receipt_mask).sum():,}")
print(f"Invalid Supplier Delays     : {supplier_delay_invalid.sum():,}")

print(
    f"Invalid Percentage          : "
    f"{supplier_delay_invalid.mean() * 100:.2f}%"
)

print("\nSupplier Delay Range:")
print(
    purchase_orders.loc[
        valid_receipt_mask,
        "supplier_delay_days"
    ].min(),
    "to",
    purchase_orders.loc[
        valid_receipt_mask,
        "supplier_delay_days"
    ].max(),
    "days"
)

Total Purchase Orders       : 8,500
POs With Actual Receipt     : 8,470
POs Without Actual Receipt  : 30
Invalid Supplier Delays     : 0
Invalid Percentage          : 0.00%

Supplier Delay Range:
-2.0 to 17.0 days


## 24. Supplier Contract Lead-Time Consistency Audit

Validate whether the contract lead time recorded on each purchase order is consistent with the corresponding supplier master record.

Expected relationship:

**PO Contract Lead Time = Supplier Contract Lead Time**

This check identifies inconsistencies between transactional procurement data and supplier master data.

In [48]:
# Join purchase orders with supplier master lead time

po_supplier_leadtime = purchase_orders[
    [
        "po_id",
        "supplier_id",
        "contract_lead_time_days"
    ]
].merge(
    suppliers[
        [
            "supplier_id",
            "contract_lead_time_days"
        ]
    ],
    on="supplier_id",
    how="left",
    suffixes=("_po", "_supplier")
)

# Compare PO lead time with supplier master lead time

leadtime_invalid = (
    po_supplier_leadtime["contract_lead_time_days_po"]
    != po_supplier_leadtime["contract_lead_time_days_supplier"]
)

print(f"Total Purchase Orders       : {len(po_supplier_leadtime):,}")
print(f"Lead-Time Mismatches        : {leadtime_invalid.sum():,}")
print(
    f"Mismatch Percentage         : "
    f"{leadtime_invalid.mean() * 100:.2f}%"
)

Total Purchase Orders       : 8,500
Lead-Time Mismatches        : 0
Mismatch Percentage         : 0.00%


### Expected Receipt Date Validation

Validate whether the purchase-order expected receipt date is consistent with the PO date and contractual supplier lead time.

Expected business logic:

**Expected Receipt Date = PO Date + Contract Lead Time Days**

This check verifies that procurement planning dates are internally consistent with supplier lead-time agreements.

In [49]:
# Validate expected receipt date against contract lead time

po_date_temp = pd.to_datetime(
    purchase_orders["po_date"],
    errors="coerce"
)

expected_date_temp = pd.to_datetime(
    purchase_orders["expected_date"],
    errors="coerce"
)

calculated_expected_date = (
    po_date_temp
    + pd.to_timedelta(
        purchase_orders["contract_lead_time_days"],
        unit="D"
    )
)

expected_date_invalid = (
    expected_date_temp != calculated_expected_date
)

print(f"Total Purchase Orders       : {len(purchase_orders):,}")
print(f"Invalid Expected Dates      : {expected_date_invalid.sum():,}")
print(
    f"Invalid Percentage          : "
    f"{expected_date_invalid.mean() * 100:.2f}%"
)

print("\nContract Lead-Time Range:")
print(
    purchase_orders["contract_lead_time_days"].min(),
    "to",
    purchase_orders["contract_lead_time_days"].max(),
    "days"
)

Total Purchase Orders       : 8,500
Invalid Expected Dates      : 0
Invalid Percentage          : 0.00%

Contract Lead-Time Range:
5 to 24 days


## 25. Purchase Order Value Validation

Validate whether the pre-calculated purchase-order value is consistent with ordered quantity and unit cost.

Expected business logic:

**PO Value = Ordered Quantity × Unit Cost**

A small tolerance is applied to account for monetary rounding.

In [50]:
# Calculate expected purchase order value

expected_po_value = (
    purchase_orders["ordered_qty"]
    * purchase_orders["unit_cost"]
)

po_value_invalid = ~np.isclose(
    purchase_orders["po_value"],
    expected_po_value,
    atol=0.05
)

print(f"Total Purchase Orders    : {len(purchase_orders):,}")
print(f"Invalid PO Values        : {po_value_invalid.sum():,}")
print(
    f"Invalid Percentage       : "
    f"{po_value_invalid.mean() * 100:.2f}%"
)

Total Purchase Orders    : 8,500
Invalid PO Values        : 0
Invalid Percentage       : 0.00%


## 26. Product Cost Consistency Audit

Validate whether unit costs used in inventory and purchase-order records are consistent with the corresponding product master cost.

Expected relationships:

- **Inventory Unit Cost = Product Master Unit Cost**
- **Purchase Order Unit Cost = Product Master Unit Cost**

Any differences are identified for further investigation rather than automatically treated as errors, since historical purchase costs may legitimately differ from current product master costs.

In [51]:
# Inventory vs Product master unit cost

inventory_cost_check = inventory[
    ["product_id", "warehouse_id", "unit_cost"]
].merge(
    products[
        ["product_id", "unit_cost"]
    ],
    on="product_id",
    how="left",
    suffixes=("_inventory", "_product")
)

inventory_cost_invalid = ~np.isclose(
    inventory_cost_check["unit_cost_inventory"],
    inventory_cost_check["unit_cost_product"],
    atol=0.01
)


# Purchase Orders vs Product master unit cost

po_cost_check = purchase_orders[
    ["po_id", "product_id", "unit_cost"]
].merge(
    products[
        ["product_id", "unit_cost"]
    ],
    on="product_id",
    how="left",
    suffixes=("_po", "_product")
)

po_cost_invalid = ~np.isclose(
    po_cost_check["unit_cost_po"],
    po_cost_check["unit_cost_product"],
    atol=0.01
)


cost_consistency_summary = pd.DataFrame([
    {
        "Comparison": "Inventory vs Product Master",
        "Total_Rows": len(inventory_cost_check),
        "Cost_Mismatches": inventory_cost_invalid.sum(),
        "Mismatch_Percentage": round(
            inventory_cost_invalid.mean() * 100, 2
        )
    },
    {
        "Comparison": "Purchase Orders vs Product Master",
        "Total_Rows": len(po_cost_check),
        "Cost_Mismatches": po_cost_invalid.sum(),
        "Mismatch_Percentage": round(
            po_cost_invalid.mean() * 100, 2
        )
    }
])

display(cost_consistency_summary)

,Comparison,Total_Rows,Cost_Mismatches,Mismatch_Percentage
0,Inventory vs Product Master,3000,0,0.0
1,Purchase Orders vs Product Master,8500,0,0.0


## 27. Product Selling Price Consistency Audit

Compare sales-order-item unit prices with the corresponding product master unit price.

Differences are identified for investigation rather than automatically treated as errors, since transactional selling prices may legitimately vary due to historical pricing or other commercial conditions.

In [52]:
# Compare Sales Order Item unit price with Product master unit price

item_price_check = sales_order_items[
    [
        "order_item_id",
        "product_id",
        "unit_price"
    ]
].merge(
    products[
        [
            "product_id",
            "unit_price"
        ]
    ],
    on="product_id",
    how="left",
    suffixes=("_order_item", "_product")
)

price_mismatch = ~np.isclose(
    item_price_check["unit_price_order_item"],
    item_price_check["unit_price_product"],
    atol=0.01
)

print(f"Total Sales Order Items : {len(item_price_check):,}")
print(f"Price Mismatches        : {price_mismatch.sum():,}")
print(
    f"Mismatch Percentage     : "
    f"{price_mismatch.mean() * 100:.2f}%"
)

Total Sales Order Items : 61,397
Price Mismatches        : 0
Mismatch Percentage     : 0.00%


## 28. Sales Order Item Value Validation

Validate the pre-calculated gross and net sales-order-item values.

Expected business logic:

**Gross Value = Ordered Quantity × Unit Price**

**Net Order Value = Gross Value × (1 - Discount Rate)**

A small monetary tolerance is applied to account for rounding.

In [53]:
# Calculate expected gross value

expected_gross_value = (
    sales_order_items["ordered_qty"]
    * sales_order_items["unit_price"]
)

gross_value_invalid = ~np.isclose(
    sales_order_items["gross_value"],
    expected_gross_value,
    atol=0.05
)


# Calculate expected net order value

expected_net_order_value = (
    sales_order_items["gross_value"]
    * (1 - sales_order_items["discount_pct"])
)

net_value_invalid = ~np.isclose(
    sales_order_items["net_order_value"],
    expected_net_order_value,
    atol=0.05
)


value_validation_summary = pd.DataFrame([
    {
        "Field": "gross_value",
        "Total_Rows": len(sales_order_items),
        "Invalid_Rows": gross_value_invalid.sum(),
        "Invalid_Percentage": round(
            gross_value_invalid.mean() * 100, 2
        )
    },
    {
        "Field": "net_order_value",
        "Total_Rows": len(sales_order_items),
        "Invalid_Rows": net_value_invalid.sum(),
        "Invalid_Percentage": round(
            net_value_invalid.mean() * 100, 2
        )
    }
])

display(value_validation_summary)

,Field,Total_Rows,Invalid_Rows,Invalid_Percentage
0,gross_value,61397,0,0.0
1,net_order_value,61397,0,0.0


## 29. Supplier Quality Rating Audit

Evaluate supplier quality ratings for missing values, valid numerical range, and potential out-of-range records.

The missing rating is documented during the audit phase and will be handled later during data cleaning.

In [54]:
# Audit supplier quality ratings

print(f"Total Suppliers          : {len(suppliers):,}")
print(
    f"Missing Quality Ratings  : "
    f"{suppliers['quality_rating'].isna().sum():,}"
)

print(
    f"Minimum Quality Rating   : "
    f"{suppliers['quality_rating'].min()}"
)

print(
    f"Maximum Quality Rating   : "
    f"{suppliers['quality_rating'].max()}"
)

print("\nQuality Rating Summary:")
display(
    suppliers["quality_rating"].describe().to_frame().T
)

Total Suppliers          : 100
Missing Quality Ratings  : 1
Minimum Quality Rating   : 3.14
Maximum Quality Rating   : 4.99

Quality Rating Summary:


,count,mean,std,min,25%,50%,75%,max
quality_rating,99.0,4.045657,0.554113,3.14,3.595,4.03,4.53,4.99


## 30. Supplier Reliability Factor Audit

Evaluate supplier reliability factors for missing values, numerical range, and potential out-of-range observations.

The observed distribution is reviewed before defining any formal validation boundaries.

In [55]:
# Audit supplier reliability factor

print(f"Total Suppliers             : {len(suppliers):,}")

print(
    f"Missing Reliability Factors : "
    f"{suppliers['reliability_factor'].isna().sum():,}"
)

print(
    f"Minimum Reliability Factor  : "
    f"{suppliers['reliability_factor'].min()}"
)

print(
    f"Maximum Reliability Factor  : "
    f"{suppliers['reliability_factor'].max()}"
)

print("\nReliability Factor Summary:")

display(
    suppliers["reliability_factor"]
    .describe()
    .to_frame()
    .T
)

Total Suppliers             : 100
Missing Reliability Factors : 0
Minimum Reliability Factor  : 0.57
Maximum Reliability Factor  : 0.98

Reliability Factor Summary:


,count,mean,std,min,25%,50%,75%,max
reliability_factor,100.0,0.86459,0.09108,0.57,0.8255,0.8805,0.928,0.98


## 31. Supplier Categorical Field Audit

Review supplier categorical fields for unexpected values and category consistency.

Fields evaluated:

- Supplier Region
- Preferred Supplier Flag

Preferred-supplier status is treated as a source business classification and is not reconstructed from quality or reliability measures without a documented business rule.

In [56]:
# Audit supplier categorical fields

print("Supplier Region Distribution")
print("-" * 40)
print(
    suppliers["supplier_region"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPreferred Supplier Flag Distribution")
print("-" * 40)
print(
    suppliers["preferred_supplier_flag"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nUnique Supplier Regions:")
print(
    sorted(
        suppliers["supplier_region"]
        .dropna()
        .unique()
    )
)

print("\nUnique Preferred Supplier Flags:")
print(
    sorted(
        suppliers["preferred_supplier_flag"]
        .dropna()
        .unique()
    )
)

Supplier Region Distribution
----------------------------------------
supplier_region
Central    19
East       22
North      18
South      15
West       26
Name: count, dtype: int64

Preferred Supplier Flag Distribution
----------------------------------------
preferred_supplier_flag
N    60
Y    40
Name: count, dtype: int64

Unique Supplier Regions:
['Central', 'East', 'North', 'South', 'West']

Unique Preferred Supplier Flags:
['N', 'Y']


## 32. Customer Categorical Field Audit

Review key customer categorical fields for missing values, unexpected categories, and inconsistent labels.

Fields evaluated:

- Customer Segment
- Region
- Credit Tier

In [57]:
# Audit customer categorical fields

customer_categorical_columns = [
    "customer_segment",
    "region",
    "credit_tier"
]

customer_category_results = []

for column in customer_categorical_columns:

    customer_category_results.append({
        "Column": column,
        "Unique_Non_Null_Values":
            customers[column].nunique(dropna=True),
        "Missing_Count":
            customers[column].isna().sum(),
        "Missing_Percentage":
            round(customers[column].isna().mean() * 100, 2)
    })

customer_category_summary = pd.DataFrame(
    customer_category_results
)

display(customer_category_summary)

for column in customer_categorical_columns:

    print(f"\n{column}")
    print("-" * 40)

    print(
        customers[column]
        .value_counts(dropna=False)
        .sort_index()
    )

,Column,Unique_Non_Null_Values,Missing_Count,Missing_Percentage
0,customer_segment,4,0,0.0
1,region,5,0,0.0
2,credit_tier,3,20,0.8



customer_segment
----------------------------------------
customer_segment
Distributor    363
Enterprise     420
Retail         931
SMB            786
Name: count, dtype: int64

region
----------------------------------------
region
Central    316
East       386
North      470
South      707
West       621
Name: count, dtype: int64

credit_tier
----------------------------------------
credit_tier
A      1022
B      1048
C       410
NaN      20
Name: count, dtype: int64


## 33. Product Categorical Field Audit

Review key product categorical fields for missing values, unexpected categories, and inconsistent labels.

Fields evaluated:

- Product Category
- Product Subcategory
- Active Flag

Missing or inconsistent values identified during this audit will be handled in the data-cleaning phase.

In [58]:
# Audit product categorical fields

product_categorical_columns = [
    "category",
    "subcategory",
    "active_flag"
]

product_category_results = []

for column in product_categorical_columns:

    product_category_results.append({
        "Column": column,
        "Unique_Non_Null_Values":
            products[column].nunique(dropna=True),
        "Missing_Count":
            products[column].isna().sum(),
        "Missing_Percentage":
            round(products[column].isna().mean() * 100, 2)
    })

product_category_summary = pd.DataFrame(
    product_category_results
)

display(product_category_summary)


# Display category distributions

for column in product_categorical_columns:

    print(f"\n{column}")
    print("-" * 40)

    print(
        products[column]
        .value_counts(dropna=False)
        .sort_index()
    )

,Column,Unique_Non_Null_Values,Missing_Count,Missing_Percentage
0,category,5,0,0.0
1,subcategory,14,3,0.6
2,active_flag,2,0,0.0



category
----------------------------------------
category
Automotive                 76
Electronics               122
Health & Personal Care     96
Home & Kitchen            109
Office Supplies            97
Name: count, dtype: int64

subcategory
----------------------------------------
subcategory
Accessories      60
Appliances       43
Audio            52
Cookware         36
Electrical       31
Furniture        36
Grooming         39
Maintenance      20
Personal Care    29
Printing         34
Smart Devices    35
Stationery       26
Storage          28
Wellness         28
NaN               3
Name: count, dtype: int64

active_flag
----------------------------------------
active_flag
N     12
Y    488
Name: count, dtype: int64


## 34. Sales Order Categorical Field Audit

Review key sales-order categorical fields for missing values, unexpected categories, and inconsistent labels.

Fields evaluated:

- Order Priority
- Sales Channel
- Order Status

This audit verifies categorical standardization without modifying the source data.

In [59]:
# Audit sales order categorical fields

sales_order_categorical_columns = [
    "priority",
    "sales_channel",
    "order_status"
]

sales_order_category_results = []

for column in sales_order_categorical_columns:

    sales_order_category_results.append({
        "Column": column,
        "Unique_Non_Null_Values":
            sales_orders[column].nunique(dropna=True),

        "Missing_Count":
            sales_orders[column].isna().sum(),

        "Missing_Percentage":
            round(
                sales_orders[column].isna().mean() * 100,
                2
            )
    })

sales_order_category_summary = pd.DataFrame(
    sales_order_category_results
)

display(sales_order_category_summary)


# Display category distributions

for column in sales_order_categorical_columns:

    print(f"\n{column}")
    print("-" * 40)

    print(
        sales_orders[column]
        .value_counts(dropna=False)
        .sort_index()
    )

,Column,Unique_Non_Null_Values,Missing_Count,Missing_Percentage
0,priority,4,0,0.0
1,sales_channel,4,0,0.0
2,order_status,4,0,0.0



priority
----------------------------------------
priority
High       6359
Low        2035
Normal    14557
Urgent     2049
Name: count, dtype: int64

sales_channel
----------------------------------------
sales_channel
Direct         7195
Distributor    5469
Marketplace    4485
Online         7851
Name: count, dtype: int64

order_status
----------------------------------------
order_status
Backordered              198
Cancelled                275
Fulfilled              14150
Partially Fulfilled    10377
Name: count, dtype: int64


## 35. Sales Order Item Categorical Field Audit

Review key sales-order-item categorical fields for missing values, unexpected categories, and inconsistent labels.

Fields evaluated:

- Stockout Flag
- Backlog Status

The underlying business behavior of these fields was evaluated previously; this section focuses on categorical data quality.

In [60]:
# Audit Sales Order Item categorical fields

sales_item_categorical_columns = [
    "stockout_flag",
    "backlog_status"
]

sales_item_category_results = []

for column in sales_item_categorical_columns:

    sales_item_category_results.append({
        "Column": column,
        "Unique_Non_Null_Values":
            sales_order_items[column].nunique(dropna=True),

        "Missing_Count":
            sales_order_items[column].isna().sum(),

        "Missing_Percentage":
            round(
                sales_order_items[column].isna().mean() * 100,
                2
            )
    })

sales_item_category_summary = pd.DataFrame(
    sales_item_category_results
)

display(sales_item_category_summary)


# Display category distributions

for column in sales_item_categorical_columns:

    print(f"\n{column}")
    print("-" * 40)

    print(
        sales_order_items[column]
        .value_counts(dropna=False)
        .sort_index()
    )

,Column,Unique_Non_Null_Values,Missing_Count,Missing_Percentage
0,stockout_flag,2,0,0.0
1,backlog_status,4,0,0.0



stockout_flag
----------------------------------------
stockout_flag
N    51151
Y    10246
Name: count, dtype: int64

backlog_status
----------------------------------------
backlog_status
Cancelled           1349
Later Fulfilled     6365
No Backlog         48119
Open Backlog        5564
Name: count, dtype: int64


## 36. Inventory Categorical Field Audit

Review the inventory `stock_status` field for missing values, unexpected categories, and inconsistent labels.

The underlying stock-status classification logic was validated previously; this section focuses on categorical data quality.

In [61]:
# Audit Inventory stock status field

inventory_status_summary = pd.DataFrame([
    {
        "Column": "stock_status",
        "Unique_Non_Null_Values":
            inventory["stock_status"].nunique(dropna=True),

        "Missing_Count":
            inventory["stock_status"].isna().sum(),

        "Missing_Percentage":
            round(
                inventory["stock_status"].isna().mean() * 100,
                2
            )
    }
])

display(inventory_status_summary)


print("\nStock Status Distribution")
print("-" * 40)

print(
    inventory["stock_status"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nUnique Stock Status Values:")

print(
    sorted(
        inventory["stock_status"]
        .dropna()
        .unique()
    )
)

,Column,Unique_Non_Null_Values,Missing_Count,Missing_Percentage
0,stock_status,4,0,0.0



Stock Status Distribution
----------------------------------------
stock_status
Healthy      2077
Low Stock     205
Overstock     478
Stockout      240
Name: count, dtype: int64

Unique Stock Status Values:
['Healthy', 'Low Stock', 'Overstock', 'Stockout']


## 37. Inventory Transaction Type Audit

Review the inventory transaction type field for missing values, unexpected categories, and inconsistent labels.

Transaction quantity sign conventions were validated previously; this section focuses on categorical data quality.

In [62]:
# Audit Inventory Transaction types

transaction_type_summary = pd.DataFrame([
    {
        "Column": "transaction_type",
        "Unique_Non_Null_Values":
            inventory_transactions["transaction_type"]
            .nunique(dropna=True),

        "Missing_Count":
            inventory_transactions["transaction_type"]
            .isna()
            .sum(),

        "Missing_Percentage":
            round(
                inventory_transactions["transaction_type"]
                .isna()
                .mean() * 100,
                2
            )
    }
])

display(transaction_type_summary)


print("\nTransaction Type Distribution")
print("-" * 45)

print(
    inventory_transactions["transaction_type"]
    .value_counts(dropna=False)
    .sort_index()
)


print("\nUnique Transaction Types:")

print(
    sorted(
        inventory_transactions["transaction_type"]
        .dropna()
        .unique()
    )
)

,Column,Unique_Non_Null_Values,Missing_Count,Missing_Percentage
0,transaction_type,5,0,0.0



Transaction Type Distribution
---------------------------------------------
transaction_type
ADJUSTMENT_IN      5511
ADJUSTMENT_OUT     5504
PO_RECEIPT        27423
RETURN_IN          8759
SALE_ISSUE        62848
Name: count, dtype: int64

Unique Transaction Types:
['ADJUSTMENT_IN', 'ADJUSTMENT_OUT', 'PO_RECEIPT', 'RETURN_IN', 'SALE_ISSUE']


## 38. Shipment Categorical Field Audit

Review shipment categorical fields for missing values, unexpected categories, and inconsistent labels.

Fields evaluated:

- On-Time Flag
- OTIF Flag

The underlying business logic for these fields was validated previously; this section focuses on categorical data quality.

In [63]:
# Audit Shipment categorical fields

shipment_categorical_columns = [
    "on_time_flag",
    "otif_flag"
]

shipment_category_results = []

for column in shipment_categorical_columns:

    shipment_category_results.append({
        "Column": column,

        "Unique_Non_Null_Values":
            shipments[column].nunique(dropna=True),

        "Missing_Count":
            shipments[column].isna().sum(),

        "Missing_Percentage":
            round(
                shipments[column].isna().mean() * 100,
                2
            )
    })

shipment_category_summary = pd.DataFrame(
    shipment_category_results
)

display(shipment_category_summary)


# Display distributions

for column in shipment_categorical_columns:

    print(f"\n{column}")
    print("-" * 40)

    print(
        shipments[column]
        .value_counts(dropna=False)
        .sort_index()
    )

,Column,Unique_Non_Null_Values,Missing_Count,Missing_Percentage
0,on_time_flag,2,0,0.0
1,otif_flag,2,0,0.0



on_time_flag
----------------------------------------
on_time_flag
N     5929
Y    18869
Name: count, dtype: int64

otif_flag
----------------------------------------
otif_flag
N    12586
Y    12212
Name: count, dtype: int64


## 39. Purchase Order Categorical Field Audit

Review the purchase-order status field for missing values, unexpected categories, and inconsistent labels.

The relationship between PO status and receipt quantity was validated previously; this section focuses on categorical data quality.

In [64]:
# Audit Purchase Order status field

po_status_summary = pd.DataFrame([
    {
        "Column": "po_status",

        "Unique_Non_Null_Values":
            purchase_orders["po_status"].nunique(dropna=True),

        "Missing_Count":
            purchase_orders["po_status"].isna().sum(),

        "Missing_Percentage":
            round(
                purchase_orders["po_status"].isna().mean() * 100,
                2
            )
    }
])

display(po_status_summary)


print("\nPurchase Order Status Distribution")
print("-" * 45)

print(
    purchase_orders["po_status"]
    .value_counts(dropna=False)
    .sort_index()
)


print("\nUnique PO Status Values:")

print(
    sorted(
        purchase_orders["po_status"]
        .dropna()
        .unique()
    )
)

,Column,Unique_Non_Null_Values,Missing_Count,Missing_Percentage
0,po_status,2,0,0.0



Purchase Order Status Distribution
---------------------------------------------
po_status
Open                    30
Partially Received    8470
Name: count, dtype: int64

Unique PO Status Values:
['Open', 'Partially Received']


## 40. Warehouse Master Data Audit

Evaluate warehouse master data for completeness, uniqueness, and valid capacity values.

Checks include:

- Missing warehouse attributes
- Duplicate warehouse names
- Non-positive warehouse capacity
- City and region completeness

In [65]:
# Audit warehouse master data

warehouse_audit = pd.DataFrame([
    {
        "Check": "Missing Warehouse Names",
        "Invalid_Rows":
            warehouses["warehouse_name"].isna().sum()
    },
    {
        "Check": "Missing Cities",
        "Invalid_Rows":
            warehouses["city"].isna().sum()
    },
    {
        "Check": "Missing Regions",
        "Invalid_Rows":
            warehouses["region"].isna().sum()
    },
    {
        "Check": "Duplicate Warehouse Names",
        "Invalid_Rows":
            warehouses["warehouse_name"].duplicated().sum()
    },
    {
        "Check": "Non-Positive Capacity",
        "Invalid_Rows":
            (warehouses["capacity_units"] <= 0).sum()
    }
])

display(warehouse_audit)

print("\nWarehouse Capacity Summary")
print("-" * 40)

print(
    warehouses["capacity_units"]
    .describe()
)

,Check,Invalid_Rows
0,Missing Warehouse Names,0
1,Missing Cities,0
2,Missing Regions,0
3,Duplicate Warehouse Names,0
4,Non-Positive Capacity,0



Warehouse Capacity Summary
----------------------------------------
count        6.000000
mean     53833.333333
std       7626.707459
min      45000.000000
25%      48500.000000
50%      52500.000000
75%      58750.000000
max      65000.000000
Name: capacity_units, dtype: float64


## 41. Warehouse Region Mapping Audit

Review the warehouse master mapping between warehouse, city, region, and capacity.

Since each warehouse represents a unique operational location, the warehouse-to-region relationship is inspected for consistency before downstream geographic analysis.

In [66]:
# Review warehouse master mapping

warehouse_region_mapping = (
    warehouses[
        [
            "warehouse_id",
            "warehouse_name",
            "city",
            "region",
            "capacity_units"
        ]
    ]
    .sort_values("warehouse_id")
    .reset_index(drop=True)
)

display(warehouse_region_mapping)


# Check whether a warehouse ID maps to more than one region

warehouse_region_count = (
    warehouses
    .groupby("warehouse_id")["region"]
    .nunique()
)

inconsistent_warehouse_regions = (
    warehouse_region_count > 1
).sum()

print(
    "\nWarehouse IDs mapped to multiple regions:",
    inconsistent_warehouse_regions
)

,warehouse_id,warehouse_name,city,region,capacity_units
0,WH01,Bengaluru Central,Bengaluru,South,65000
1,WH02,Mumbai West,Mumbai,West,55000
2,WH03,Delhi NCR,Gurugram,North,60000
3,WH04,Hyderabad South,Hyderabad,South,48000
4,WH05,Chennai Hub,Chennai,South,50000
5,WH06,Pune DC,Pune,West,45000



Warehouse IDs mapped to multiple regions: 0


## 42. Product–Supplier Relationship Audit

Review the relationship between products and suppliers to understand supplier coverage across the product portfolio.

This audit evaluates:

- Number of products assigned to each supplier
- Suppliers with no assigned products
- Minimum and maximum products per active supplier
- Overall supplier utilization within the product master

Foreign-key integrity was validated previously; this section focuses on relationship coverage.

In [67]:
# Count products assigned to each supplier

supplier_product_count = (
    products
    .groupby("supplier_id")
    .size()
    .reset_index(name="Product_Count")
)

# Include suppliers with no products

supplier_product_coverage = (
    suppliers[["supplier_id", "supplier_name"]]
    .merge(
        supplier_product_count,
        on="supplier_id",
        how="left"
    )
)

supplier_product_coverage["Product_Count"] = (
    supplier_product_coverage["Product_Count"]
    .fillna(0)
    .astype(int)
)

print(f"Total Suppliers              : {len(suppliers):,}")
print(
    f"Suppliers With Products      : "
    f"{(supplier_product_coverage['Product_Count'] > 0).sum():,}"
)
print(
    f"Suppliers Without Products   : "
    f"{(supplier_product_coverage['Product_Count'] == 0).sum():,}"
)

active_supplier_counts = supplier_product_coverage.loc[
    supplier_product_coverage["Product_Count"] > 0,
    "Product_Count"
]

print(
    f"Min Products / Active Supplier : "
    f"{active_supplier_counts.min():,}"
)

print(
    f"Max Products / Active Supplier : "
    f"{active_supplier_counts.max():,}"
)

print(
    f"Avg Products / Supplier        : "
    f"{supplier_product_coverage['Product_Count'].mean():.2f}"
)

Total Suppliers              : 100
Suppliers With Products      : 100
Suppliers Without Products   : 0
Min Products / Active Supplier : 2
Max Products / Active Supplier : 11
Avg Products / Supplier        : 5.00


## 43. Product–Warehouse Inventory Coverage Audit

Validate inventory coverage across the complete product and warehouse network.

With 500 products and 6 warehouses, complete coverage would produce:

**500 × 6 = 3,000 unique Product–Warehouse combinations**

This check verifies whether every product is represented exactly once at every warehouse.

In [68]:
# Expected number of Product-Warehouse combinations

expected_combinations = (
    products["product_id"].nunique()
    * warehouses["warehouse_id"].nunique()
)

actual_combinations = (
    inventory[
        ["product_id", "warehouse_id"]
    ]
    .drop_duplicates()
    .shape[0]
)


# Number of warehouse records per product

warehouses_per_product = (
    inventory
    .groupby("product_id")["warehouse_id"]
    .nunique()
)


# Number of product records per warehouse

products_per_warehouse = (
    inventory
    .groupby("warehouse_id")["product_id"]
    .nunique()
)


print(f"Expected Product-Warehouse Pairs : {expected_combinations:,}")
print(f"Actual Unique Pairs              : {actual_combinations:,}")

print(
    f"Missing Product-Warehouse Pairs  : "
    f"{expected_combinations - actual_combinations:,}"
)

print(
    f"\nWarehouses per Product           : "
    f"{warehouses_per_product.min()} to "
    f"{warehouses_per_product.max()}"
)

print(
    f"Products per Warehouse           : "
    f"{products_per_warehouse.min()} to "
    f"{products_per_warehouse.max()}"
)

Expected Product-Warehouse Pairs : 3,000
Actual Unique Pairs              : 3,000
Missing Product-Warehouse Pairs  : 0

Warehouses per Product           : 6 to 6
Products per Warehouse           : 500 to 500


## 44. Sales Order Header–Item Quantity Reconciliation

Validate whether order-level quantities in the Sales Orders table reconcile with the aggregated quantities from Sales Order Items.

Expected relationships:

- Sales Order Ordered Quantity = Sum of Item Ordered Quantities
- Sales Order Fulfilled Quantity = Sum of Item Fulfilled Quantities
- Sales Order Backlog Quantity = Sum of Item Backlog Quantities

This check verifies consistency between order-header and line-level transactional data.

In [69]:
# Aggregate Sales Order Item quantities to order level

item_qty_by_order = (
    sales_order_items
    .groupby("order_id")
    .agg(
        item_ordered_qty=("ordered_qty", "sum"),
        item_fulfilled_qty=("fulfilled_qty", "sum"),
        item_backlog_qty=("backlog_qty", "sum")
    )
    .reset_index()
)


# Join with Sales Order header quantities

order_qty_check = sales_orders[
    [
        "order_id",
        "ordered_qty",
        "fulfilled_qty",
        "backlog_qty"
    ]
].merge(
    item_qty_by_order,
    on="order_id",
    how="left"
)


# Compare header vs line-level quantities

ordered_qty_invalid = (
    order_qty_check["ordered_qty"]
    != order_qty_check["item_ordered_qty"]
)

fulfilled_qty_invalid = (
    order_qty_check["fulfilled_qty"]
    != order_qty_check["item_fulfilled_qty"]
)

backlog_qty_invalid = (
    order_qty_check["backlog_qty"]
    != order_qty_check["item_backlog_qty"]
)


quantity_reconciliation = pd.DataFrame([
    {
        "Measure": "Ordered Quantity",
        "Invalid_Orders": ordered_qty_invalid.sum()
    },
    {
        "Measure": "Fulfilled Quantity",
        "Invalid_Orders": fulfilled_qty_invalid.sum()
    },
    {
        "Measure": "Backlog Quantity",
        "Invalid_Orders": backlog_qty_invalid.sum()
    }
])

display(quantity_reconciliation)

print(
    "\nOrders with any quantity mismatch:",
    (
        ordered_qty_invalid
        | fulfilled_qty_invalid
        | backlog_qty_invalid
    ).sum()
)

,Measure,Invalid_Orders
0,Ordered Quantity,0
1,Fulfilled Quantity,0
2,Backlog Quantity,0



Orders with any quantity mismatch: 0


## 45. Sales Order Header–Item Backlog Value Reconciliation

Validate whether the order-level backlog value in the Sales Orders table reconciles with the aggregated backlog value from Sales Order Items.

Expected relationship:

**Sales Order Backlog Value = Sum of Sales Order Item Backlog Values**

A small monetary tolerance is applied to account for rounding.

In [70]:
# Aggregate item backlog value to order level

item_backlog_by_order = (
    sales_order_items
    .groupby("order_id")
    .agg(
        item_backlog_value=("backlog_value", "sum")
    )
    .reset_index()
)


# Join with Sales Order header backlog value

backlog_value_check = sales_orders[
    [
        "order_id",
        "backlog_value"
    ]
].merge(
    item_backlog_by_order,
    on="order_id",
    how="left"
)


# Compare header and item-level backlog values

backlog_value_mismatch = ~np.isclose(
    backlog_value_check["backlog_value"],
    backlog_value_check["item_backlog_value"],
    atol=0.05
)

print(
    f"Total Sales Orders        : "
    f"{len(backlog_value_check):,}"
)

print(
    f"Backlog Value Mismatches  : "
    f"{backlog_value_mismatch.sum():,}"
)

print(
    f"Mismatch Percentage       : "
    f"{backlog_value_mismatch.mean() * 100:.2f}%"
)

Total Sales Orders        : 25,000
Backlog Value Mismatches  : 0
Mismatch Percentage       : 0.00%


## 46. Shipment–Order Fulfillment Quantity Reconciliation

Validate whether shipment quantities reconcile with fulfilled quantities recorded in Sales Orders.

Since an order may potentially have multiple shipment records, shipped quantities are first aggregated to the order level.

Expected relationship:

**Total Shipped Quantity per Order = Fulfilled Quantity per Order**

This check evaluates consistency between order fulfillment and shipment execution data.

In [71]:
# Aggregate shipped quantity to order level

shipment_qty_by_order = (
    shipments
    .groupby("order_id")
    .agg(
        shipment_count=("shipment_id", "count"),
        total_shipped_qty=("shipped_qty", "sum")
    )
    .reset_index()
)


# Join with Sales Orders

shipment_order_check = sales_orders[
    [
        "order_id",
        "fulfilled_qty"
    ]
].merge(
    shipment_qty_by_order,
    on="order_id",
    how="left"
)


# Orders without shipment records

missing_shipment = (
    shipment_order_check["shipment_count"].isna()
)


# Fill shipment quantity only for comparison/reporting

shipment_order_check["total_shipped_qty"] = (
    shipment_order_check["total_shipped_qty"]
    .fillna(0)
)


# Compare shipped quantity with fulfilled quantity

shipment_qty_mismatch = (
    shipment_order_check["total_shipped_qty"]
    != shipment_order_check["fulfilled_qty"]
)


print(
    f"Total Sales Orders             : "
    f"{len(shipment_order_check):,}"
)

print(
    f"Orders Without Shipment Record : "
    f"{missing_shipment.sum():,}"
)

print(
    f"Shipment Quantity Mismatches   : "
    f"{shipment_qty_mismatch.sum():,}"
)

print(
    f"Mismatch Percentage            : "
    f"{shipment_qty_mismatch.mean() * 100:.2f}%"
)

print(
    f"\nMaximum Shipments per Order    : "
    f"{shipment_qty_by_order['shipment_count'].max():,}"
)

Total Sales Orders             : 25,000
Orders Without Shipment Record : 202
Shipment Quantity Mismatches   : 0
Mismatch Percentage            : 0.00%

Maximum Shipments per Order    : 1


### Orders Without Shipment Validation

Investigate sales orders without shipment records to determine whether the absence of shipment data is consistent with order fulfillment.

Expected relationship:

**Orders with no shipment record should have Fulfilled Quantity = 0.**

This distinguishes legitimate unfulfilled orders from potentially missing shipment records.

In [72]:
# Investigate orders without shipment records

orders_without_shipments = (
    shipment_order_check[
        shipment_order_check["shipment_count"].isna()
    ]
)

print(
    f"Orders Without Shipments        : "
    f"{len(orders_without_shipments):,}"
)

print(
    f"With Zero Fulfilled Quantity    : "
    f"{(orders_without_shipments['fulfilled_qty'] == 0).sum():,}"
)

print(
    f"With Positive Fulfilled Quantity: "
    f"{(orders_without_shipments['fulfilled_qty'] > 0).sum():,}"
)


# Validate reverse relationship:
# zero fulfillment should also imply no shipment

zero_fulfilled_orders = (
    shipment_order_check["fulfilled_qty"] == 0
)

zero_fulfilled_with_shipment = (
    zero_fulfilled_orders
    & shipment_order_check["shipment_count"].notna()
)

print(
    f"\nTotal Zero-Fulfilled Orders     : "
    f"{zero_fulfilled_orders.sum():,}"
)

print(
    f"Zero-Fulfilled With Shipment    : "
    f"{zero_fulfilled_with_shipment.sum():,}"
)

Orders Without Shipments        : 202
With Zero Fulfilled Quantity    : 202
With Positive Fulfilled Quantity: 0

Total Zero-Fulfilled Orders     : 202
Zero-Fulfilled With Shipment    : 0


### Shipment–Order Warehouse Consistency

Validate whether each shipment originates from the warehouse assigned to the corresponding sales order.

Expected relationship:

**Shipment Warehouse = Sales Order Warehouse**

This check verifies consistency between order allocation and shipment execution.

In [73]:
# Compare Shipment warehouse with Sales Order warehouse

shipment_warehouse_check = shipments[
    [
        "shipment_id",
        "order_id",
        "warehouse_id"
    ]
].merge(
    sales_orders[
        [
            "order_id",
            "warehouse_id"
        ]
    ],
    on="order_id",
    how="left",
    suffixes=("_shipment", "_order")
)


warehouse_mismatch = (
    shipment_warehouse_check["warehouse_id_shipment"]
    != shipment_warehouse_check["warehouse_id_order"]
)


print(
    f"Total Shipments            : "
    f"{len(shipment_warehouse_check):,}"
)

print(
    f"Warehouse Mismatches       : "
    f"{warehouse_mismatch.sum():,}"
)

print(
    f"Mismatch Percentage        : "
    f"{warehouse_mismatch.mean() * 100:.2f}%"
)

Total Shipments            : 24,798
Warehouse Mismatches       : 0
Mismatch Percentage        : 0.00%


### Shipment–Order Requested Delivery Date Consistency

Validate whether the requested delivery date carried into the shipment record matches the original requested delivery date recorded on the corresponding sales order.

Expected relationship:

**Shipment Requested Delivery Date = Sales Order Requested Delivery Date**

This check verifies consistency between order requirements and shipment execution records.

In [74]:
# Compare requested delivery dates between Shipments and Sales Orders

shipment_request_date_check = shipments[
    [
        "shipment_id",
        "order_id",
        "requested_delivery_date"
    ]
].merge(
    sales_orders[
        [
            "order_id",
            "requested_delivery_date"
        ]
    ],
    on="order_id",
    how="left",
    suffixes=("_shipment", "_order")
)


# Convert temporarily to datetime before comparison

shipment_requested_date = pd.to_datetime(
    shipment_request_date_check[
        "requested_delivery_date_shipment"
    ],
    errors="coerce"
)

order_requested_date = pd.to_datetime(
    shipment_request_date_check[
        "requested_delivery_date_order"
    ],
    errors="coerce"
)


requested_date_mismatch = (
    shipment_requested_date != order_requested_date
)


print(
    f"Total Shipments                 : "
    f"{len(shipment_request_date_check):,}"
)

print(
    f"Requested Date Mismatches       : "
    f"{requested_date_mismatch.sum():,}"
)

print(
    f"Mismatch Percentage             : "
    f"{requested_date_mismatch.mean() * 100:.2f}%"
)

Total Shipments                 : 24,798
Requested Date Mismatches       : 0
Mismatch Percentage             : 0.00%


### Sales Order Item–Order Warehouse Consistency

Validate whether each sales-order item is associated with the same warehouse as its parent sales order.

Expected relationship:

**Sales Order Item Warehouse = Sales Order Warehouse**

This check verifies consistency between order-header allocation and line-level order data.

In [75]:
# Compare Sales Order Item warehouse with Sales Order warehouse

item_warehouse_check = sales_order_items[
    [
        "order_item_id",
        "order_id",
        "warehouse_id"
    ]
].merge(
    sales_orders[
        [
            "order_id",
            "warehouse_id"
        ]
    ],
    on="order_id",
    how="left",
    suffixes=("_item", "_order")
)


item_warehouse_mismatch = (
    item_warehouse_check["warehouse_id_item"]
    != item_warehouse_check["warehouse_id_order"]
)


print(
    f"Total Sales Order Items : "
    f"{len(item_warehouse_check):,}"
)

print(
    f"Warehouse Mismatches    : "
    f"{item_warehouse_mismatch.sum():,}"
)

print(
    f"Mismatch Percentage     : "
    f"{item_warehouse_mismatch.mean() * 100:.2f}%"
)

Total Sales Order Items : 61,397
Warehouse Mismatches    : 0
Mismatch Percentage     : 0.00%


### Sales Order Item–Order Date Consistency

Validate whether the order date recorded at the sales-order-item level matches the order date of its parent sales order.

Expected relationship:

**Sales Order Item Order Date = Sales Order Order Date**

This check verifies temporal consistency between order-header and line-level transactional data.

In [76]:
# Compare Sales Order Item order date with Sales Order order date

item_order_date_check = sales_order_items[
    [
        "order_item_id",
        "order_id",
        "order_date"
    ]
].merge(
    sales_orders[
        [
            "order_id",
            "order_date"
        ]
    ],
    on="order_id",
    how="left",
    suffixes=("_item", "_order")
)


# Temporary datetime conversion for comparison

item_order_date = pd.to_datetime(
    item_order_date_check["order_date_item"],
    errors="coerce"
)

header_order_date = pd.to_datetime(
    item_order_date_check["order_date_order"],
    errors="coerce"
)


order_date_mismatch = (
    item_order_date != header_order_date
)


print(
    f"Total Sales Order Items : "
    f"{len(item_order_date_check):,}"
)

print(
    f"Order Date Mismatches   : "
    f"{order_date_mismatch.sum():,}"
)

print(
    f"Mismatch Percentage     : "
    f"{order_date_mismatch.mean() * 100:.2f}%"
)

Total Sales Order Items : 61,397
Order Date Mismatches   : 0
Mismatch Percentage     : 0.00%


### Purchase Order Product–Supplier Consistency

Validate whether the supplier recorded on each purchase order matches the supplier assigned to the corresponding product in the Products master table.

Expected relationship:

**Purchase Order Supplier = Product Master Supplier**

This check verifies consistency between procurement transactions and product-supplier master data.

In [77]:
# Compare Purchase Order supplier with Product master supplier

po_supplier_check = purchase_orders[
    [
        "po_id",
        "product_id",
        "supplier_id"
    ]
].merge(
    products[
        [
            "product_id",
            "supplier_id"
        ]
    ],
    on="product_id",
    how="left",
    suffixes=("_po", "_product")
)


supplier_mismatch = (
    po_supplier_check["supplier_id_po"]
    != po_supplier_check["supplier_id_product"]
)


print(
    f"Total Purchase Orders : "
    f"{len(po_supplier_check):,}"
)

print(
    f"Supplier Mismatches   : "
    f"{supplier_mismatch.sum():,}"
)

print(
    f"Mismatch Percentage   : "
    f"{supplier_mismatch.mean() * 100:.2f}%"
)

Total Purchase Orders : 8,500
Supplier Mismatches   : 0
Mismatch Percentage   : 0.00%


### Purchase Order Product–Warehouse Inventory Coverage

Validate whether every product and warehouse combination appearing in Purchase Orders exists in the Inventory master snapshot.

Expected relationship:

**Each Purchase Order Product–Warehouse combination should have a corresponding Inventory record.**

This check verifies consistency between procurement activity and inventory master coverage.

In [78]:
# Validate PO product-warehouse combinations against Inventory

po_inventory_check = purchase_orders[
    [
        "po_id",
        "product_id",
        "warehouse_id"
    ]
].merge(
    inventory[
        [
            "product_id",
            "warehouse_id"
        ]
    ].drop_duplicates(),
    on=["product_id", "warehouse_id"],
    how="left",
    indicator=True
)


missing_inventory_pair = (
    po_inventory_check["_merge"] == "left_only"
)


print(
    f"Total Purchase Orders          : "
    f"{len(po_inventory_check):,}"
)

print(
    f"Missing Inventory Combinations : "
    f"{missing_inventory_pair.sum():,}"
)

print(
    f"Missing Percentage             : "
    f"{missing_inventory_pair.mean() * 100:.2f}%"
)

Total Purchase Orders          : 8,500
Missing Inventory Combinations : 0
Missing Percentage             : 0.00%


### Inventory Transaction Product–Warehouse Coverage

Validate whether every product and warehouse combination appearing in Inventory Transactions exists in the Inventory master snapshot.

Expected relationship:

**Each Inventory Transaction Product–Warehouse combination should have a corresponding Inventory record.**

This check verifies that all inventory movements reference valid product–warehouse combinations.

In [79]:
# Validate Inventory Transaction product-warehouse combinations
# against Inventory master

transaction_inventory_check = inventory_transactions[
    [
        "transaction_id",
        "product_id",
        "warehouse_id"
    ]
].merge(
    inventory[
        [
            "product_id",
            "warehouse_id"
        ]
    ].drop_duplicates(),
    on=["product_id", "warehouse_id"],
    how="left",
    indicator=True
)


missing_inventory_pair = (
    transaction_inventory_check["_merge"] == "left_only"
)


print(
    f"Total Inventory Transactions   : "
    f"{len(transaction_inventory_check):,}"
)

print(
    f"Missing Inventory Combinations : "
    f"{missing_inventory_pair.sum():,}"
)

print(
    f"Missing Percentage             : "
    f"{missing_inventory_pair.mean() * 100:.2f}%"
)

Total Inventory Transactions   : 110,045
Missing Inventory Combinations : 0
Missing Percentage             : 0.00%


### Inventory Transaction Date Range Validation

Validate whether inventory transaction dates fall within the expected operational observation period.

Expected period:

**2024-01-01 to 2026-09-30**

Transactions outside this period may indicate incorrect dates or records outside the intended analytical scope.

In [80]:
# Temporarily convert transaction date for validation

transaction_dates = pd.to_datetime(
    inventory_transactions["transaction_date"],
    errors="coerce"
)

expected_start_date = pd.Timestamp("2024-01-01")
expected_end_date = pd.Timestamp("2026-09-30")


before_start = (
    transaction_dates < expected_start_date
)

after_end = (
    transaction_dates > expected_end_date
)

outside_period = (
    before_start | after_end
)


print(
    f"Total Inventory Transactions : "
    f"{len(inventory_transactions):,}"
)

print(
    f"Earliest Transaction Date    : "
    f"{transaction_dates.min().date()}"
)

print(
    f"Latest Transaction Date      : "
    f"{transaction_dates.max().date()}"
)

print(
    f"Transactions Before Start    : "
    f"{before_start.sum():,}"
)

print(
    f"Transactions After End       : "
    f"{after_end.sum():,}"
)

print(
    f"Outside Expected Period      : "
    f"{outside_period.sum():,}"
)

Total Inventory Transactions : 110,045
Earliest Transaction Date    : 2024-01-01
Latest Transaction Date      : 2026-09-30
Transactions Before Start    : 0
Transactions After End       : 0
Outside Expected Period      : 0


### Inventory Transaction Duplicate ID Investigation

Investigate duplicate transaction IDs to determine whether they represent exact duplicate records or conflicting transaction records.

This distinction is important for the cleaning phase:

- **Exact duplicates** can safely be deduplicated.
- **Conflicting duplicate IDs** require further investigation before removal.

In [81]:
# Identify all rows belonging to duplicate transaction IDs

duplicate_transaction_rows = inventory_transactions[
    inventory_transactions["transaction_id"].duplicated(keep=False)
].sort_values("transaction_id")


# Number of duplicated transaction IDs

duplicate_ids = duplicate_transaction_rows[
    "transaction_id"
].nunique()


# Check whether duplicate IDs contain conflicting data

conflicting_duplicate_ids = (
    duplicate_transaction_rows
    .groupby("transaction_id")
    .nunique(dropna=False)
    .gt(1)
    .any(axis=1)
)


print(
    f"Duplicate Transaction IDs      : "
    f"{duplicate_ids:,}"
)

print(
    f"Rows With Duplicate IDs        : "
    f"{len(duplicate_transaction_rows):,}"
)

print(
    f"Exact Duplicate Rows           : "
    f"{inventory_transactions.duplicated().sum():,}"
)

print(
    f"Conflicting Duplicate IDs      : "
    f"{conflicting_duplicate_ids.sum():,}"
)

Duplicate Transaction IDs      : 45
Rows With Duplicate IDs        : 90
Exact Duplicate Rows           : 45
Conflicting Duplicate IDs      : 0


### Customer Credit Tier Missing-Value Pattern

Investigate the 20 missing `credit_tier` values to determine whether they are concentrated within specific customer segments or regions.

This helps distinguish between:

- Random missing values
- Systematic missingness associated with particular customer groups

No imputation is performed during this audit step.

In [82]:
# Customers with missing credit tier

missing_credit_tier = customers[
    customers["credit_tier"].isna()
]


print(
    f"Total Customers              : "
    f"{len(customers):,}"
)

print(
    f"Missing Credit Tier          : "
    f"{len(missing_credit_tier):,}"
)

print(
    f"Missing Percentage           : "
    f"{len(missing_credit_tier) / len(customers) * 100:.2f}%"
)


print("\nMissing Credit Tier by Customer Segment:")
print(
    missing_credit_tier["customer_segment"]
    .value_counts(dropna=False)
)


print("\nMissing Credit Tier by Region:")
print(
    missing_credit_tier["region"]
    .value_counts(dropna=False)
)

Total Customers              : 2,500
Missing Credit Tier          : 20
Missing Percentage           : 0.80%

Missing Credit Tier by Customer Segment:
customer_segment
Distributor    6
Retail         6
SMB            6
Enterprise     2
Name: count, dtype: int64

Missing Credit Tier by Region:
region
South      6
West       5
Central    4
North      3
East       2
Name: count, dtype: int64


### Product Subcategory Missing-Value Investigation

Investigate products with missing `subcategory` values and examine their associated product categories.

The objective is to determine whether the missing subcategories are concentrated within a particular product category and whether they can potentially be handled using category-level business context during the cleaning phase.

No missing values are filled during this audit step.

In [83]:
# Products with missing subcategory

missing_subcategory = products[
    products["subcategory"].isna()
]


print(
    f"Total Products            : "
    f"{len(products):,}"
)

print(
    f"Missing Subcategory       : "
    f"{len(missing_subcategory):,}"
)

print(
    f"Missing Percentage        : "
    f"{len(missing_subcategory) / len(products) * 100:.2f}%"
)


print("\nMissing Subcategory by Category:")
print(
    missing_subcategory["category"]
    .value_counts(dropna=False)
)


print("\nProducts With Missing Subcategory:")
print(
    missing_subcategory[
        [
            "product_id",
            "product_name",
            "category",
            "supplier_id",
            "active_flag"
        ]
    ].to_string(index=False)
)

Total Products            : 500
Missing Subcategory       : 3
Missing Percentage        : 0.60%

Missing Subcategory by Category:
category
Home & Kitchen     2
Office Supplies    1
Name: count, dtype: int64

Products With Missing Subcategory:
product_id          product_name        category supplier_id active_flag
   SKU0184 Cookware Product 0184  Home & Kitchen      SUP007           Y
   SKU0281  Storage Product 0281  Home & Kitchen      SUP085           Y
   SKU0424 Printing Product 0424 Office Supplies      SUP068           Y


### Supplier Quality Rating Missing-Value Investigation

Investigate the supplier with a missing `quality_rating` and assess its presence in product and purchase-order data.

This helps determine the potential analytical impact of the missing supplier quality rating.

No imputation is performed during this audit step.

In [84]:
# Supplier with missing quality rating

missing_supplier_quality = suppliers[
    suppliers["quality_rating"].isna()
].copy()


# Count products associated with each supplier

product_counts = (
    products
    .groupby("supplier_id")
    .size()
    .rename("product_count")
)


# Count purchase orders associated with each supplier

po_counts = (
    purchase_orders
    .groupby("supplier_id")
    .size()
    .rename("purchase_order_count")
)


# Add impact counts

missing_supplier_quality = (
    missing_supplier_quality
    .merge(
        product_counts,
        on="supplier_id",
        how="left"
    )
    .merge(
        po_counts,
        on="supplier_id",
        how="left"
    )
)


missing_supplier_quality[
    ["product_count", "purchase_order_count"]
] = (
    missing_supplier_quality[
        ["product_count", "purchase_order_count"]
    ].fillna(0)
)


print(
    f"Total Suppliers                : "
    f"{len(suppliers):,}"
)

print(
    f"Missing Quality Rating         : "
    f"{len(missing_supplier_quality):,}"
)

print(
    f"Missing Percentage             : "
    f"{len(missing_supplier_quality) / len(suppliers) * 100:.2f}%"
)


print("\nSupplier With Missing Quality Rating:")

print(
    missing_supplier_quality[
        [
            "supplier_id",
            "supplier_name",
            "supplier_region",
            "preferred_supplier_flag",
            "reliability_factor",
            "product_count",
            "purchase_order_count"
        ]
    ].to_string(index=False)
)

Total Suppliers                : 100
Missing Quality Rating         : 1
Missing Percentage             : 1.00%

Supplier With Missing Quality Rating:
supplier_id supplier_name supplier_region preferred_supplier_flag  reliability_factor  product_count  purchase_order_count
     SUP051  Supplier 051            West                       N               0.808              4                    10


### Purchase Order Missing Receipt Data Investigation

Investigate missing `actual_receipt_date` and `supplier_delay_days` values to determine whether they are associated with open purchase orders that have not yet been received.

Expected business relationship:

**Open PO + Received Quantity = 0 → Actual Receipt Date and Supplier Delay Days may legitimately be missing.**

This distinguishes structurally missing operational data from true data-quality issues.

In [85]:
# Missing receipt-related fields

missing_receipt_date = purchase_orders["actual_receipt_date"].isna()
missing_supplier_delay = purchase_orders["supplier_delay_days"].isna()

open_po = purchase_orders["po_status"] == "Open"
zero_received = purchase_orders["received_qty"] == 0


print(
    f"Total Purchase Orders                 : "
    f"{len(purchase_orders):,}"
)

print(
    f"Missing Actual Receipt Date           : "
    f"{missing_receipt_date.sum():,}"
)

print(
    f"Missing Supplier Delay Days           : "
    f"{missing_supplier_delay.sum():,}"
)

print(
    f"Both Receipt Fields Missing           : "
    f"{(missing_receipt_date & missing_supplier_delay).sum():,}"
)


# Validate relationship with Open POs

missing_receipt_open_po = (
    missing_receipt_date
    & missing_supplier_delay
    & open_po
    & zero_received
)

print(
    f"\nMissing Fields + Open + Zero Received : "
    f"{missing_receipt_open_po.sum():,}"
)


# Check for problematic cases

missing_on_non_open_po = (
    (missing_receipt_date | missing_supplier_delay)
    & ~open_po
)

open_po_with_receipt_data = (
    open_po
    & (
        purchase_orders["actual_receipt_date"].notna()
        | purchase_orders["supplier_delay_days"].notna()
    )
)

print(
    f"Missing Receipt Data on Non-Open POs  : "
    f"{missing_on_non_open_po.sum():,}"
)

print(
    f"Open POs With Receipt Data            : "
    f"{open_po_with_receipt_data.sum():,}"
)

Total Purchase Orders                 : 8,500
Missing Actual Receipt Date           : 30
Missing Supplier Delay Days           : 30
Both Receipt Fields Missing           : 30

Missing Fields + Open + Zero Received : 30
Missing Receipt Data on Non-Open POs  : 0
Open POs With Receipt Data            : 0


### Shipment Chronology Anomaly Impact Analysis

Investigate shipment records where the actual delivery date occurs before the shipment date.

Invalid chronological relationship:

**Actual Delivery Date < Shipment Date**

These records represent a temporal inconsistency because an order cannot normally be delivered before it is shipped.

This step measures the scale and warehouse distribution of the anomaly before any corrective action is taken.

In [86]:
# Temporary datetime conversion

shipment_date_temp = pd.to_datetime(
    shipments["shipment_date"],
    errors="coerce"
)

actual_delivery_temp = pd.to_datetime(
    shipments["actual_delivery_date"],
    errors="coerce"
)


# Identify chronology anomalies

shipment_chronology_anomaly = (
    actual_delivery_temp < shipment_date_temp
)


anomalous_shipments = shipments[
    shipment_chronology_anomaly
].copy()


# Calculate how many days delivery precedes shipment

anomalous_shipments["days_delivery_before_shipment"] = (
    shipment_date_temp[shipment_chronology_anomaly]
    - actual_delivery_temp[shipment_chronology_anomaly]
).dt.days


print(
    f"Total Shipments                     : "
    f"{len(shipments):,}"
)

print(
    f"Chronology Anomalies                : "
    f"{shipment_chronology_anomaly.sum():,}"
)

print(
    f"Anomaly Percentage                  : "
    f"{shipment_chronology_anomaly.mean() * 100:.2f}%"
)


print("\nDays Delivery Occurs Before Shipment:")

print(
    anomalous_shipments[
        "days_delivery_before_shipment"
    ].describe()
)


print("\nChronology Anomalies by Warehouse:")

print(
    anomalous_shipments[
        "warehouse_id"
    ].value_counts().sort_index()
)

Total Shipments                     : 24,798
Chronology Anomalies                : 4,467
Anomaly Percentage                  : 18.01%

Days Delivery Occurs Before Shipment:
count    4467.000000
mean        2.366913
std         1.304135
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         6.000000
Name: days_delivery_before_shipment, dtype: float64

Chronology Anomalies by Warehouse:
warehouse_id
WH01    984
WH02    758
WH03    887
WH04    649
WH05    612
WH06    577
Name: count, dtype: int64


### Shipment Chronology Anomaly by Year

Analyze shipment chronology anomalies by year to determine whether the issue is concentrated within a specific period or persists throughout the dataset.

This helps identify whether the anomaly may have resulted from a period-specific data-generation, system, or recording issue.

In [87]:
# Create temporary audit dataset

shipment_chronology_year = pd.DataFrame({
    "shipment_date": shipment_date_temp,
    "is_anomaly": shipment_chronology_anomaly
})


# Extract shipment year

shipment_chronology_year["year"] = (
    shipment_chronology_year["shipment_date"].dt.year
)


# Summarize by year

chronology_by_year = (
    shipment_chronology_year
    .groupby("year")
    .agg(
        total_shipments=("is_anomaly", "size"),
        chronology_anomalies=("is_anomaly", "sum")
    )
)


chronology_by_year["anomaly_pct"] = (
    chronology_by_year["chronology_anomalies"]
    / chronology_by_year["total_shipments"]
    * 100
).round(2)


print(chronology_by_year)

      total_shipments  chronology_anomalies  anomaly_pct
year                                                    
2024             9059                  1624        17.93
2025             9296                  1698        18.27
2026             6443                  1145        17.77


### Shipment Chronology Anomaly vs Delivery Performance

Analyze whether shipment chronology anomalies are associated with delivery-performance classifications.

The analysis compares:

- Delivery delay
- On-time status
- OTIF status

between valid shipment records and records where the actual delivery date occurs before the shipment date.

This helps determine the analytical impact of the chronology issue on downstream delivery KPIs.

In [88]:
# Create comparison dataset

shipment_performance_audit = shipments[
    [
        "delivery_delay_days",
        "on_time_flag",
        "otif_flag"
    ]
].copy()


shipment_performance_audit["chronology_status"] = np.where(
    shipment_chronology_anomaly,
    "Anomaly",
    "Valid"
)


# Delivery delay summary

print("Delivery Delay Summary:")

print(
    shipment_performance_audit
    .groupby("chronology_status")["delivery_delay_days"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        min="min",
        max="max"
    )
    .round(2)
)


# On-time distribution

print("\nOn-Time Flag Distribution:")

print(
    pd.crosstab(
        shipment_performance_audit["chronology_status"],
        shipment_performance_audit["on_time_flag"],
        margins=True
    )
)


# OTIF distribution

print("\nOTIF Flag Distribution:")

print(
    pd.crosstab(
        shipment_performance_audit["chronology_status"],
        shipment_performance_audit["otif_flag"],
        margins=True
    )
)

Delivery Delay Summary:
                   count  mean  median  min  max
chronology_status                               
Anomaly             4467 -1.18    -1.0   -2    3
Valid              20331  0.93     0.0   -2   10

On-Time Flag Distribution:
on_time_flag          N      Y    All
chronology_status                    
Anomaly              78   4389   4467
Valid              5851  14480  20331
All                5929  18869  24798

OTIF Flag Distribution:
otif_flag              N      Y    All
chronology_status                     
Anomaly             1659   2808   4467
Valid              10927   9404  20331
All                12586  12212  24798


### Shipment Date Sequence Classification

Classify shipment records according to the chronological relationship between requested delivery, shipment, and actual delivery dates.

The purpose is to understand the broader temporal structure of shipment records and distinguish valid operational sequences from impossible delivery-before-shipment sequences.

No dates are corrected during this audit step.

In [89]:
# Temporary datetime versions

requested_temp = pd.to_datetime(
    shipments["requested_delivery_date"],
    errors="coerce"
)

shipment_temp = pd.to_datetime(
    shipments["shipment_date"],
    errors="coerce"
)

actual_temp = pd.to_datetime(
    shipments["actual_delivery_date"],
    errors="coerce"
)


# Classify shipment date sequence

conditions = [
    actual_temp < shipment_temp,

    (shipment_temp <= actual_temp)
    & (actual_temp <= requested_temp),

    (shipment_temp <= requested_temp)
    & (requested_temp < actual_temp),

    (requested_temp < shipment_temp)
    & (shipment_temp <= actual_temp)
]

labels = [
    "Invalid: Delivered Before Shipment",
    "Valid: Delivered On/Before Requested Date",
    "Valid: Shipped Before Request, Delivered Late",
    "Valid: Shipped After Requested Date"
]


shipments_date_sequence = np.select(
    conditions,
    labels,
    default="Other"
)


sequence_summary = (
    pd.Series(shipments_date_sequence)
    .value_counts()
    .rename_axis("date_sequence")
    .reset_index(name="shipment_count")
)


sequence_summary["percentage"] = (
    sequence_summary["shipment_count"]
    / len(shipments)
    * 100
).round(2)


print(sequence_summary.to_string(index=False))

                                date_sequence  shipment_count  percentage
    Valid: Delivered On/Before Requested Date           14480       58.39
Valid: Shipped Before Request, Delivered Late            5046       20.35
           Invalid: Delivered Before Shipment            4467       18.01
          Valid: Shipped After Requested Date             805        3.25


### Shipment Chronology Anomaly by Sales Channel

Analyze shipment chronology anomalies across sales channels.

This check determines whether delivery-before-shipment anomalies are concentrated within a particular sales channel or distributed throughout the business.

A similar anomaly rate across channels would provide further evidence that the issue is systematic rather than channel-specific.

In [90]:
# Join shipment chronology status with Sales Order channel

shipment_channel_audit = shipments[
    [
        "shipment_id",
        "order_id"
    ]
].copy()

shipment_channel_audit["is_chronology_anomaly"] = (
    shipment_chronology_anomaly.values
)


shipment_channel_audit = shipment_channel_audit.merge(
    sales_orders[
        [
            "order_id",
            "sales_channel"
        ]
    ],
    on="order_id",
    how="left"
)


# Summarize anomaly rate by sales channel

channel_anomaly_summary = (
    shipment_channel_audit
    .groupby("sales_channel")
    .agg(
        total_shipments=("shipment_id", "count"),
        chronology_anomalies=("is_chronology_anomaly", "sum")
    )
)


channel_anomaly_summary["anomaly_pct"] = (
    channel_anomaly_summary["chronology_anomalies"]
    / channel_anomaly_summary["total_shipments"]
    * 100
).round(2)


print(channel_anomaly_summary)

               total_shipments  chronology_anomalies  anomaly_pct
sales_channel                                                    
Direct                    7147                  1286        17.99
Distributor               5415                   977        18.04
Marketplace               4457                   837        18.78
Online                    7779                  1367        17.57


### Shipment Chronology Anomaly by Order Priority

Analyze shipment chronology anomalies across order priority levels.

This check determines whether delivery-before-shipment anomalies are concentrated among urgent or high-priority orders or are distributed consistently across all priority levels.

Similar anomaly rates across priority groups would provide additional evidence that the chronology issue is systematic.

In [91]:
# Join shipment chronology status with Sales Order priority

shipment_priority_audit = shipments[
    [
        "shipment_id",
        "order_id"
    ]
].copy()

shipment_priority_audit["is_chronology_anomaly"] = (
    shipment_chronology_anomaly.values
)


shipment_priority_audit = shipment_priority_audit.merge(
    sales_orders[
        [
            "order_id",
            "priority"
        ]
    ],
    on="order_id",
    how="left"
)


# Summarize anomaly rate by priority

priority_anomaly_summary = (
    shipment_priority_audit
    .groupby("priority")
    .agg(
        total_shipments=("shipment_id", "count"),
        chronology_anomalies=("is_chronology_anomaly", "sum")
    )
)


priority_anomaly_summary["anomaly_pct"] = (
    priority_anomaly_summary["chronology_anomalies"]
    / priority_anomaly_summary["total_shipments"]
    * 100
).round(2)


print(priority_anomaly_summary)

          total_shipments  chronology_anomalies  anomaly_pct
priority                                                    
High                 6299                  1035        16.43
Low                  2021                   384        19.00
Normal              14448                  2765        19.14
Urgent               2030                   283        13.94


### Shipment Chronology Anomaly Severity Distribution

Analyze the frequency of shipment chronology anomalies based on how many days the recorded actual delivery date precedes the shipment date.

This helps determine whether the issue follows a recurring date-offset pattern that may indicate a systematic data-generation or recording problem.

In [92]:
# Frequency of days delivery occurs before shipment

anomaly_day_distribution = (
    anomalous_shipments[
        "days_delivery_before_shipment"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("days_before_shipment")
    .reset_index(name="shipment_count")
)


anomaly_day_distribution["percentage"] = (
    anomaly_day_distribution["shipment_count"]
    / len(anomalous_shipments)
    * 100
).round(2)


print(
    anomaly_day_distribution.to_string(index=False)
)

 days_before_shipment  shipment_count  percentage
                    1            1456       32.59
                    2            1237       27.69
                    3             878       19.66
                    4             548       12.27
                    5             271        6.07
                    6              77        1.72


### Shipment Date vs Sales Order Date Validation

Validate whether shipments occur on or after the corresponding sales order date.

Expected chronological relationship:

**Shipment Date ≥ Order Date**

A shipment occurring before its associated order date would represent an invalid transactional sequence.

In [93]:
# Join shipment date with Sales Order date

shipment_order_date_audit = shipments[
    [
        "shipment_id",
        "order_id",
        "shipment_date"
    ]
].merge(
    sales_orders[
        [
            "order_id",
            "order_date"
        ]
    ],
    on="order_id",
    how="left"
)


# Temporary datetime conversion

shipment_date_check = pd.to_datetime(
    shipment_order_date_audit["shipment_date"],
    errors="coerce"
)

order_date_check = pd.to_datetime(
    shipment_order_date_audit["order_date"],
    errors="coerce"
)


# Identify impossible chronology

shipment_before_order = (
    shipment_date_check < order_date_check
)


print(
    f"Total Shipments              : "
    f"{len(shipment_order_date_audit):,}"
)

print(
    f"Shipments Before Order Date  : "
    f"{shipment_before_order.sum():,}"
)

print(
    f"Invalid Percentage           : "
    f"{shipment_before_order.mean() * 100:.2f}%"
)

Total Shipments              : 24,798
Shipments Before Order Date  : 0
Invalid Percentage           : 0.00%


### Actual Delivery Date vs Sales Order Date Validation

Validate whether the recorded actual delivery date occurs on or after the corresponding sales order date.

Expected chronological relationship:

**Actual Delivery Date ≥ Order Date**

A delivery occurring before the original order date would represent a severe transactional chronology error.

In [94]:
# Join Actual Delivery Date with Sales Order Date

delivery_order_date_audit = shipments[
    [
        "shipment_id",
        "order_id",
        "actual_delivery_date"
    ]
].merge(
    sales_orders[
        [
            "order_id",
            "order_date"
        ]
    ],
    on="order_id",
    how="left"
)


# Temporary datetime conversion

actual_delivery_check = pd.to_datetime(
    delivery_order_date_audit["actual_delivery_date"],
    errors="coerce"
)

order_date_check = pd.to_datetime(
    delivery_order_date_audit["order_date"],
    errors="coerce"
)


# Identify invalid chronology

delivery_before_order = (
    actual_delivery_check < order_date_check
)


print(
    f"Total Shipments                 : "
    f"{len(delivery_order_date_audit):,}"
)

print(
    f"Deliveries Before Order Date    : "
    f"{delivery_before_order.sum():,}"
)

print(
    f"Invalid Percentage              : "
    f"{delivery_before_order.mean() * 100:.2f}%"
)

Total Shipments                 : 24,798
Deliveries Before Order Date    : 0
Invalid Percentage              : 0.00%


### Requested Delivery Lead-Time Audit

Analyze the number of days between the sales order date and requested delivery date.

Expected relationship:

**Requested Delivery Date ≥ Order Date**

The lead-time distribution helps validate whether requested delivery dates fall within reasonable and consistent operational ranges.

In [95]:
# Temporary datetime conversion

order_date_temp = pd.to_datetime(
    sales_orders["order_date"],
    errors="coerce"
)

requested_delivery_temp = pd.to_datetime(
    sales_orders["requested_delivery_date"],
    errors="coerce"
)


# Calculate requested delivery lead time

requested_lead_days = (
    requested_delivery_temp - order_date_temp
).dt.days


print(
    f"Total Sales Orders        : "
    f"{len(sales_orders):,}"
)

print(
    f"Negative Lead Times       : "
    f"{(requested_lead_days < 0).sum():,}"
)

print(
    f"Same-Day Requested        : "
    f"{(requested_lead_days == 0).sum():,}"
)


print("\nRequested Delivery Lead-Time Summary:")

print(
    requested_lead_days.describe()
)


print("\nLead-Time Frequency:")

print(
    requested_lead_days
    .value_counts()
    .sort_index()
)

Total Sales Orders        : 25,000
Negative Lead Times       : 0
Same-Day Requested        : 0

Requested Delivery Lead-Time Summary:
count    25000.000000
mean         6.989000
std          3.145103
min          2.000000
25%          4.000000
50%          7.000000
75%         10.000000
max         12.000000
dtype: float64

Lead-Time Frequency:
2     2224
3     2293
4     2322
5     2171
6     2409
7     2243
8     2298
9     2351
10    2239
11    2233
12    2217
Name: count, dtype: int64


### Order-to-Shipment Processing Time Audit

Analyze the number of days between the sales order date and shipment date.

Expected relationship:

**Shipment Date ≥ Order Date**

This analysis evaluates order-processing time and identifies potentially unusual shipment delays before delivery.

In [96]:
# Join shipment date with order date

order_shipment_time = shipments[
    [
        "shipment_id",
        "order_id",
        "shipment_date"
    ]
].merge(
    sales_orders[
        [
            "order_id",
            "order_date"
        ]
    ],
    on="order_id",
    how="left"
)


# Convert temporarily to datetime

order_shipment_time["order_date"] = pd.to_datetime(
    order_shipment_time["order_date"],
    errors="coerce"
)

order_shipment_time["shipment_date"] = pd.to_datetime(
    order_shipment_time["shipment_date"],
    errors="coerce"
)


# Calculate processing time

order_shipment_time["processing_days"] = (
    order_shipment_time["shipment_date"]
    - order_shipment_time["order_date"]
).dt.days


print(
    f"Total Shipments           : "
    f"{len(order_shipment_time):,}"
)

print(
    f"Negative Processing Time  : "
    f"{(order_shipment_time['processing_days'] < 0).sum():,}"
)

print(
    f"Same-Day Shipments        : "
    f"{(order_shipment_time['processing_days'] == 0).sum():,}"
)


print("\nOrder-to-Shipment Processing Time Summary:")

print(
    order_shipment_time["processing_days"].describe()
)


print("\nProcessing-Time Frequency:")

print(
    order_shipment_time["processing_days"]
    .value_counts()
    .sort_index()
)

Total Shipments           : 24,798
Negative Processing Time  : 0
Same-Day Shipments        : 0

Order-to-Shipment Processing Time Summary:
count    24798.000000
mean         3.518227
std          1.707125
min          1.000000
25%          2.000000
50%          4.000000
75%          5.000000
max          6.000000
Name: processing_days, dtype: float64

Processing-Time Frequency:
processing_days
1    4039
2    4153
3    4102
4    4115
5    4200
6    4189
Name: count, dtype: int64


### Shipment-to-Delivery Transit Time Audit

Analyze the number of days between shipment and actual delivery.

Expected chronological relationship:

**Actual Delivery Date ≥ Shipment Date**

Negative transit times represent impossible delivery-before-shipment records and are treated as data-quality anomalies.

This analysis quantifies the overall transit-time distribution before data cleaning.

In [97]:
# Temporary datetime conversion

shipment_date_transit = pd.to_datetime(
    shipments["shipment_date"],
    errors="coerce"
)

actual_delivery_transit = pd.to_datetime(
    shipments["actual_delivery_date"],
    errors="coerce"
)


# Calculate shipment-to-delivery transit time

transit_days = (
    actual_delivery_transit - shipment_date_transit
).dt.days


print(
    f"Total Shipments          : "
    f"{len(shipments):,}"
)

print(
    f"Negative Transit Times   : "
    f"{(transit_days < 0).sum():,}"
)

print(
    f"Same-Day Deliveries      : "
    f"{(transit_days == 0).sum():,}"
)

print(
    f"Positive Transit Times   : "
    f"{(transit_days > 0).sum():,}"
)


print("\nShipment-to-Delivery Transit Time Summary:")

print(
    transit_days.describe()
)


print("\nTransit-Time Frequency:")

print(
    transit_days
    .value_counts()
    .sort_index()
)

Total Shipments          : 24,798
Negative Transit Times   : 4,467
Same-Day Deliveries      : 1,703
Positive Transit Times   : 18,628

Shipment-to-Delivery Transit Time Summary:
count    24798.000000
mean         4.025042
std          4.779543
min         -6.000000
25%          1.000000
50%          4.000000
75%          7.000000
max         21.000000
dtype: float64

Transit-Time Frequency:
-6       77
-5      271
-4      548
-3      878
-2     1237
-1     1456
 0     1703
 1     1923
 2     1940
 3     1964
 4     2050
 5     1993
 6     1765
 7     1536
 8     1295
 9     1103
 10     743
 11     555
 12     404
 13     319
 14     303
 15     246
 16     177
 17     143
 18      80
 19      56
 20      25
 21       8
Name: count, dtype: int64


### Valid Shipment Transit-Time Profile

Analyze shipment-to-delivery transit time after excluding records with impossible negative transit times.

This provides a reliable baseline for normal shipment transit behavior without modifying or deleting any source records.

Only records satisfying:

**Actual Delivery Date ≥ Shipment Date**

are included in this profile.

In [98]:
# Keep only chronologically valid transit times

valid_transit_days = transit_days[
    transit_days >= 0
]


print(
    f"Total Shipments              : "
    f"{len(shipments):,}"
)

print(
    f"Valid Transit Records        : "
    f"{len(valid_transit_days):,}"
)

print(
    f"Excluded Chronology Anomalies: "
    f"{(transit_days < 0).sum():,}"
)


print("\nValid Transit-Time Summary:")

print(
    valid_transit_days.describe()
)


print("\nValid Transit-Time Frequency:")

print(
    valid_transit_days
    .value_counts()
    .sort_index()
)

Total Shipments              : 24,798
Valid Transit Records        : 20,331
Excluded Chronology Anomalies: 4,467

Valid Transit-Time Summary:
count    20331.000000
mean         5.429443
std          4.066948
min          0.000000
25%          2.000000
50%          5.000000
75%          8.000000
max         21.000000
dtype: float64

Valid Transit-Time Frequency:
0     1703
1     1923
2     1940
3     1964
4     2050
5     1993
6     1765
7     1536
8     1295
9     1103
10     743
11     555
12     404
13     319
14     303
15     246
16     177
17     143
18      80
19      56
20      25
21       8
Name: count, dtype: int64


### Valid Shipment Transit-Time Outlier Audit

Evaluate chronologically valid shipment transit times for potential statistical outliers using the Interquartile Range (IQR) method.

Outlier boundaries are defined as:

**Lower Bound = Q1 − 1.5 × IQR**

**Upper Bound = Q3 + 1.5 × IQR**

This step identifies unusually long transit times for further assessment without modifying any records.

In [99]:
# Calculate IQR for valid transit times

q1 = valid_transit_days.quantile(0.25)
q3 = valid_transit_days.quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - (1.5 * iqr)
upper_bound = q3 + (1.5 * iqr)


# Identify statistical outliers

transit_outliers = valid_transit_days[
    (valid_transit_days < lower_bound)
    | (valid_transit_days > upper_bound)
]


print(f"Q1                     : {q1:.2f}")
print(f"Q3                     : {q3:.2f}")
print(f"IQR                    : {iqr:.2f}")
print(f"Lower Outlier Bound    : {lower_bound:.2f}")
print(f"Upper Outlier Bound    : {upper_bound:.2f}")

print(
    f"\nValid Transit Records  : "
    f"{len(valid_transit_days):,}"
)

print(
    f"Transit-Time Outliers  : "
    f"{len(transit_outliers):,}"
)

print(
    f"Outlier Percentage     : "
    f"{len(transit_outliers) / len(valid_transit_days) * 100:.2f}%"
)

if len(transit_outliers) > 0:
    print(
        f"\nOutlier Range          : "
        f"{transit_outliers.min()} to {transit_outliers.max()} days"
    )

Q1                     : 2.00
Q3                     : 8.00
IQR                    : 6.00
Lower Outlier Bound    : -7.00
Upper Outlier Bound    : 17.00

Valid Transit Records  : 20,331
Transit-Time Outliers  : 169
Outlier Percentage     : 0.83%

Outlier Range          : 18 to 21 days


### Long Transit Outliers vs Delivery Performance

Investigate chronologically valid shipment transit-time outliers and compare them with requested-delivery performance.

This determines whether unusually long transit times necessarily represent late deliveries or whether some shipments had sufficiently long requested delivery windows.

Transit-time outliers are defined using the previously calculated IQR upper bound.

In [100]:
# Build valid shipment transit audit dataset

valid_transit_audit = shipments[
    transit_days >= 0
].copy()


valid_transit_audit["transit_days"] = (
    transit_days[transit_days >= 0].values
)


# Identify long-transit outliers

long_transit_outliers = valid_transit_audit[
    valid_transit_audit["transit_days"] > upper_bound
]


print(
    f"Valid Shipment Records       : "
    f"{len(valid_transit_audit):,}"
)

print(
    f"Long Transit Outliers        : "
    f"{len(long_transit_outliers):,}"
)


print("\nDelivery Delay Summary for Long Transit Outliers:")

print(
    long_transit_outliers[
        "delivery_delay_days"
    ].describe()
)


print("\nOn-Time Status:")

print(
    long_transit_outliers[
        "on_time_flag"
    ].value_counts()
)


print("\nOTIF Status:")

print(
    long_transit_outliers[
        "otif_flag"
    ].value_counts()
)

Valid Shipment Records       : 20,331
Long Transit Outliers        : 169

Delivery Delay Summary for Long Transit Outliers:
count    169.000000
mean       9.295858
std        0.903719
min        7.000000
25%        9.000000
50%       10.000000
75%       10.000000
max       10.000000
Name: delivery_delay_days, dtype: float64

On-Time Status:
on_time_flag
N    169
Name: count, dtype: int64

OTIF Status:
otif_flag
N    169
Name: count, dtype: int64


### Delivery Delay Distribution Audit

Analyze the distribution of shipment delivery delays relative to the requested delivery date.

Shipments are classified as:

- **Early Delivery:** Delivery Delay < 0
- **On Requested Date:** Delivery Delay = 0
- **Late Delivery:** Delivery Delay > 0

This provides a baseline view of delivery performance before downstream analysis.

In [101]:
# Classify delivery timing

delivery_timing = np.select(
    [
        shipments["delivery_delay_days"] < 0,
        shipments["delivery_delay_days"] == 0,
        shipments["delivery_delay_days"] > 0
    ],
    [
        "Early Delivery",
        "On Requested Date",
        "Late Delivery"
    ],
    default="Unknown"
)


delivery_timing_summary = (
    pd.Series(delivery_timing)
    .value_counts()
    .rename_axis("delivery_timing")
    .reset_index(name="shipment_count")
)


delivery_timing_summary["percentage"] = (
    delivery_timing_summary["shipment_count"]
    / len(shipments)
    * 100
).round(2)


print(
    delivery_timing_summary.to_string(index=False)
)


print("\nDelivery Delay Days Summary:")

print(
    shipments["delivery_delay_days"].describe()
)

  delivery_timing  shipment_count  percentage
   Early Delivery           12600       50.81
On Requested Date            6269       25.28
    Late Delivery            5929       23.91

Delivery Delay Days Summary:
count    24798.000000
mean         0.553270
std          3.189889
min         -2.000000
25%         -2.000000
50%         -1.000000
75%          0.000000
max         10.000000
Name: delivery_delay_days, dtype: float64


### Delivery Performance for Chronologically Valid Shipments

Evaluate delivery performance after excluding shipment records with impossible delivery-before-shipment chronology.

This provides a more reliable delivery-performance baseline using only records satisfying:

**Actual Delivery Date ≥ Shipment Date**

No source records are deleted or modified during this audit.

In [102]:
# Keep only chronologically valid shipments

valid_shipments = shipments[
    ~shipment_chronology_anomaly
].copy()


# Classify valid shipments

valid_delivery_timing = np.select(
    [
        valid_shipments["delivery_delay_days"] < 0,
        valid_shipments["delivery_delay_days"] == 0,
        valid_shipments["delivery_delay_days"] > 0
    ],
    [
        "Early Delivery",
        "On Requested Date",
        "Late Delivery"
    ],
    default="Unknown"
)


valid_delivery_summary = (
    pd.Series(valid_delivery_timing)
    .value_counts()
    .rename_axis("delivery_timing")
    .reset_index(name="shipment_count")
)


valid_delivery_summary["percentage"] = (
    valid_delivery_summary["shipment_count"]
    / len(valid_shipments)
    * 100
).round(2)


print(
    f"Chronologically Valid Shipments : "
    f"{len(valid_shipments):,}"
)

print(
    f"Excluded Chronology Anomalies   : "
    f"{shipment_chronology_anomaly.sum():,}"
)


print("\nValid Shipment Delivery Performance:")

print(
    valid_delivery_summary.to_string(index=False)
)

Chronologically Valid Shipments : 20,331
Excluded Chronology Anomalies   : 4,467

Valid Shipment Delivery Performance:
  delivery_timing  shipment_count  percentage
   Early Delivery            9182       45.16
    Late Delivery            5851       28.78
On Requested Date            5298       26.06


### Delivery KPI Impact of Shipment Chronology Anomalies

Compare delivery-performance KPIs between the complete shipment dataset and the chronologically valid subset.

This quantifies how delivery-before-shipment anomalies influence reported operational performance.

Metrics compared:

- On-Time Delivery Rate
- Late Delivery Rate
- OTIF Rate
- Average Delivery Delay

No source records are modified during this audit.

In [103]:
# Calculate KPIs for complete shipment dataset

all_on_time_rate = (
    (shipments["on_time_flag"] == "Y").mean() * 100
)

all_late_rate = (
    (shipments["delivery_delay_days"] > 0).mean() * 100
)

all_otif_rate = (
    (shipments["otif_flag"] == "Y").mean() * 100
)

all_avg_delay = (
    shipments["delivery_delay_days"].mean()
)


# Calculate KPIs for chronologically valid shipments

valid_on_time_rate = (
    (valid_shipments["on_time_flag"] == "Y").mean() * 100
)

valid_late_rate = (
    (valid_shipments["delivery_delay_days"] > 0).mean() * 100
)

valid_otif_rate = (
    (valid_shipments["otif_flag"] == "Y").mean() * 100
)

valid_avg_delay = (
    valid_shipments["delivery_delay_days"].mean()
)


# Create comparison table

kpi_comparison = pd.DataFrame({
    "Metric": [
        "On-Time Delivery Rate (%)",
        "Late Delivery Rate (%)",
        "OTIF Rate (%)",
        "Average Delivery Delay (Days)"
    ],

    "All Shipments": [
        all_on_time_rate,
        all_late_rate,
        all_otif_rate,
        all_avg_delay
    ],

    "Valid Shipments Only": [
        valid_on_time_rate,
        valid_late_rate,
        valid_otif_rate,
        valid_avg_delay
    ]
})


kpi_comparison["Difference"] = (
    kpi_comparison["Valid Shipments Only"]
    - kpi_comparison["All Shipments"]
)


print(
    kpi_comparison.round(2).to_string(index=False)
)

                       Metric  All Shipments  Valid Shipments Only  Difference
    On-Time Delivery Rate (%)          76.09                 71.22       -4.87
       Late Delivery Rate (%)          23.91                 28.78        4.87
                OTIF Rate (%)          49.25                 46.25       -2.99
Average Delivery Delay (Days)           0.55                  0.93        0.38


### Purchase Order Receipt Cycle-Time Audit

Analyze the number of days between purchase order creation and actual receipt for received purchase orders.

Only purchase orders with a recorded `actual_receipt_date` are included.

This provides a baseline of procurement cycle time and helps identify unusually long supplier fulfillment periods.

In [104]:
# Temporary datetime conversion

po_date_temp = pd.to_datetime(
    purchase_orders["po_date"],
    errors="coerce"
)

receipt_date_temp = pd.to_datetime(
    purchase_orders["actual_receipt_date"],
    errors="coerce"
)


# Calculate PO-to-receipt cycle time

po_receipt_days = (
    receipt_date_temp - po_date_temp
).dt.days


# Keep only received POs

received_po_cycle = po_receipt_days[
    receipt_date_temp.notna()
]


print(
    f"Total Purchase Orders      : "
    f"{len(purchase_orders):,}"
)

print(
    f"Received Purchase Orders   : "
    f"{len(received_po_cycle):,}"
)

print(
    f"Open / Not Yet Received    : "
    f"{receipt_date_temp.isna().sum():,}"
)

print(
    f"Negative Receipt Cycles    : "
    f"{(received_po_cycle < 0).sum():,}"
)


print("\nPO-to-Receipt Cycle-Time Summary:")

print(
    received_po_cycle.describe()
)


print("\nReceipt Cycle-Time Frequency:")

print(
    received_po_cycle
    .value_counts()
    .sort_index()
)

Total Purchase Orders      : 8,500
Received Purchase Orders   : 8,470
Open / Not Yet Received    : 30
Negative Receipt Cycles    : 0

PO-to-Receipt Cycle-Time Summary:
count    8470.000000
mean       15.525030
std         6.885911
min         3.000000
25%        10.000000
50%        16.000000
75%        20.000000
max        41.000000
dtype: float64

Receipt Cycle-Time Frequency:
3.0     108
4.0     177
5.0     283
6.0     342
7.0     338
8.0     299
9.0     332
10.0    332
11.0    325
12.0    419
13.0    359
14.0    447
15.0    413
16.0    445
17.0    467
18.0    527
19.0    562
20.0    418
21.0    412
22.0    302
23.0    264
24.0    240
25.0    132
26.0     72
27.0     54
28.0     56
29.0     54
30.0     47
31.0     38
32.0     47
33.0     33
34.0     30
35.0     32
36.0     18
37.0     15
38.0     13
39.0      5
40.0     12
41.0      1
Name: count, dtype: int64


### Supplier Delivery Performance Distribution

Analyze supplier delivery performance relative to the expected purchase-order receipt date.

Received purchase orders are classified as:

- **Early Receipt:** Supplier Delay Days < 0
- **On-Time Receipt:** Supplier Delay Days = 0
- **Late Receipt:** Supplier Delay Days > 0

Open purchase orders without an actual receipt date are excluded because their delivery performance cannot yet be evaluated.

In [105]:
# Keep only received Purchase Orders

received_pos = purchase_orders[
    purchase_orders["actual_receipt_date"].notna()
].copy()


# Classify supplier delivery performance

supplier_delivery_timing = np.select(
    [
        received_pos["supplier_delay_days"] < 0,
        received_pos["supplier_delay_days"] == 0,
        received_pos["supplier_delay_days"] > 0
    ],
    [
        "Early Receipt",
        "On-Time Receipt",
        "Late Receipt"
    ],
    default="Unknown"
)


supplier_delivery_summary = (
    pd.Series(supplier_delivery_timing)
    .value_counts()
    .rename_axis("delivery_timing")
    .reset_index(name="po_count")
)


supplier_delivery_summary["percentage"] = (
    supplier_delivery_summary["po_count"]
    / len(received_pos)
    * 100
).round(2)


print(
    f"Received Purchase Orders : "
    f"{len(received_pos):,}"
)

print("\nSupplier Delivery Performance:")

print(
    supplier_delivery_summary.to_string(index=False)
)


print("\nSupplier Delay Days Summary:")

print(
    received_pos["supplier_delay_days"].describe()
)

Received Purchase Orders : 8,470

Supplier Delivery Performance:
delivery_timing  po_count  percentage
  Early Receipt      3759       44.38
   Late Receipt      2935       34.65
On-Time Receipt      1776       20.97

Supplier Delay Days Summary:
count    8470.000000
mean        0.839197
std         4.012423
min        -2.000000
25%        -1.000000
50%         0.000000
75%         1.000000
max        17.000000
Name: supplier_delay_days, dtype: float64


### Supplier Delay Outlier Audit

Evaluate supplier delay days for statistical outliers using the Interquartile Range (IQR) method.

Outlier boundaries are defined as:

**Lower Bound = Q1 − 1.5 × IQR**

**Upper Bound = Q3 + 1.5 × IQR**

Statistical outliers are not automatically considered data-quality errors. They are identified for further business assessment.

In [106]:
# Supplier delay values for received POs

supplier_delay = received_pos["supplier_delay_days"]


# Calculate IQR

q1_supplier = supplier_delay.quantile(0.25)
q3_supplier = supplier_delay.quantile(0.75)

iqr_supplier = q3_supplier - q1_supplier

lower_supplier_bound = (
    q1_supplier - 1.5 * iqr_supplier
)

upper_supplier_bound = (
    q3_supplier + 1.5 * iqr_supplier
)


# Identify statistical outliers

supplier_delay_outliers = supplier_delay[
    (supplier_delay < lower_supplier_bound)
    | (supplier_delay > upper_supplier_bound)
]


print(f"Q1                     : {q1_supplier:.2f}")
print(f"Q3                     : {q3_supplier:.2f}")
print(f"IQR                    : {iqr_supplier:.2f}")
print(f"Lower Outlier Bound    : {lower_supplier_bound:.2f}")
print(f"Upper Outlier Bound    : {upper_supplier_bound:.2f}")

print(
    f"\nReceived Purchase Orders : "
    f"{len(received_pos):,}"
)

print(
    f"Supplier Delay Outliers   : "
    f"{len(supplier_delay_outliers):,}"
)

print(
    f"Outlier Percentage        : "
    f"{len(supplier_delay_outliers) / len(received_pos) * 100:.2f}%"
)

if len(supplier_delay_outliers) > 0:
    print(
        f"Outlier Delay Range       : "
        f"{supplier_delay_outliers.min():.0f} "
        f"to {supplier_delay_outliers.max():.0f} days"
    )

Q1                     : -1.00
Q3                     : 1.00
IQR                    : 2.00
Lower Outlier Bound    : -4.00
Upper Outlier Bound    : 4.00

Received Purchase Orders : 8,470
Supplier Delay Outliers   : 902
Outlier Percentage        : 10.65%
Outlier Delay Range       : 5 to 17 days


### Supplier Delay Outliers by Supplier

Analyze unusually late purchase-order receipts by supplier.

Supplier-delay outliers are defined as received purchase orders with:

**Supplier Delay Days > 4 days**

The objective is to determine whether severe procurement delays are concentrated among specific suppliers or broadly distributed across the supplier network.

No records are removed during this audit.

In [107]:
# Identify severe supplier-delay records

supplier_delay_outlier_pos = received_pos[
    received_pos["supplier_delay_days"] > upper_supplier_bound
].copy()


# Summarize by supplier

supplier_outlier_summary = (
    supplier_delay_outlier_pos
    .groupby("supplier_id")
    .agg(
        outlier_po_count=("po_id", "count"),
        avg_delay_days=("supplier_delay_days", "mean"),
        max_delay_days=("supplier_delay_days", "max")
    )
    .sort_values(
        ["outlier_po_count", "avg_delay_days"],
        ascending=False
    )
)


print(
    f"Total Delay Outlier POs       : "
    f"{len(supplier_delay_outlier_pos):,}"
)

print(
    f"Suppliers With Delay Outliers : "
    f"{supplier_outlier_summary.shape[0]:,}"
)


print("\nTop 10 Suppliers by Delay-Outlier Count:")

print(
    supplier_outlier_summary
    .head(10)
    .round(2)
)

Total Delay Outlier POs       : 902
Suppliers With Delay Outliers : 96

Top 10 Suppliers by Delay-Outlier Count:
             outlier_po_count  avg_delay_days  max_delay_days
supplier_id                                                  
SUP089                     89           11.44            17.0
SUP100                     41           10.29            16.0
SUP034                     39           12.03            17.0
SUP084                     25           11.44            17.0
SUP045                     25           11.32            17.0
SUP049                     21           11.19            17.0
SUP073                     21           11.10            17.0
SUP062                     19           11.32            17.0
SUP099                     18           11.56            17.0
SUP001                     17           12.00            17.0


### Supplier Delay-Outlier Rate by Supplier

Evaluate severe supplier delays relative to each supplier's total number of received purchase orders.

Using an outlier rate rather than only the raw outlier count prevents high-volume suppliers from appearing worse simply because they process more purchase orders.

**Delay Outlier Rate = Delay Outlier POs / Total Received POs × 100**

In [108]:
# Total received POs by supplier

supplier_received_volume = (
    received_pos
    .groupby("supplier_id")
    .agg(
        received_po_count=("po_id", "count")
    )
)


# Outlier POs by supplier

supplier_outlier_volume = (
    supplier_delay_outlier_pos
    .groupby("supplier_id")
    .agg(
        outlier_po_count=("po_id", "count")
    )
)


# Combine supplier volumes

supplier_delay_rate = (
    supplier_received_volume
    .join(
        supplier_outlier_volume,
        how="left"
    )
)


supplier_delay_rate["outlier_po_count"] = (
    supplier_delay_rate["outlier_po_count"]
    .fillna(0)
    .astype(int)
)


# Calculate normalized outlier rate

supplier_delay_rate["outlier_rate_pct"] = (
    supplier_delay_rate["outlier_po_count"]
    / supplier_delay_rate["received_po_count"]
    * 100
).round(2)


# Sort highest outlier rate first

supplier_delay_rate = (
    supplier_delay_rate
    .sort_values(
        ["outlier_rate_pct", "received_po_count"],
        ascending=[False, False]
    )
)


print(
    f"Suppliers Evaluated       : "
    f"{len(supplier_delay_rate):,}"
)

print(
    f"Suppliers With No Outliers: "
    f"{(supplier_delay_rate['outlier_po_count'] == 0).sum():,}"
)


print("\nTop 10 Suppliers by Delay-Outlier Rate:")

print(
    supplier_delay_rate
    .head(10)
)

Suppliers Evaluated       : 100
Suppliers With No Outliers: 4

Top 10 Suppliers by Delay-Outlier Rate:
             received_po_count  outlier_po_count  outlier_rate_pct
supplier_id                                                       
SUP089                     257                89             34.63
SUP012                      44                14             31.82
SUP100                     129                41             31.78
SUP034                     135                39             28.89
SUP023                      40                11             27.50
SUP067                      22                 5             22.73
SUP051                       9                 2             22.22
SUP078                      28                 6             21.43
SUP015                      43                 9             20.93
SUP074                      25                 5             20.00


### Supplier Reliability Factor vs Delivery Performance

Evaluate whether the supplier `reliability_factor` in the supplier master data aligns with observed purchase-order delivery performance.

For each supplier, compare:

- Reliability Factor
- Total Received Purchase Orders
- Average Supplier Delay
- Late PO Rate
- Severe Delay-Outlier Rate

This is an audit of business consistency between supplier master attributes and observed procurement performance. No supplier ratings or reliability values are modified.

In [109]:
# Build supplier-level observed delivery performance

supplier_delivery_performance = (
    received_pos
    .groupby("supplier_id")
    .agg(
        received_po_count=("po_id", "count"),
        avg_delay_days=("supplier_delay_days", "mean"),
        late_po_count=(
            "supplier_delay_days",
            lambda x: (x > 0).sum()
        ),
        severe_delay_count=(
            "supplier_delay_days",
            lambda x: (x > upper_supplier_bound).sum()
        )
    )
    .reset_index()
)


# Calculate rates

supplier_delivery_performance["late_po_rate_pct"] = (
    supplier_delivery_performance["late_po_count"]
    / supplier_delivery_performance["received_po_count"]
    * 100
)

supplier_delivery_performance["severe_delay_rate_pct"] = (
    supplier_delivery_performance["severe_delay_count"]
    / supplier_delivery_performance["received_po_count"]
    * 100
)


# Add supplier reliability factor

supplier_reliability_audit = (
    suppliers[
        [
            "supplier_id",
            "reliability_factor"
        ]
    ]
    .merge(
        supplier_delivery_performance,
        on="supplier_id",
        how="left"
    )
)


# Correlation with observed performance

corr_avg_delay = (
    supplier_reliability_audit[
        ["reliability_factor", "avg_delay_days"]
    ]
    .corr()
    .iloc[0, 1]
)

corr_late_rate = (
    supplier_reliability_audit[
        ["reliability_factor", "late_po_rate_pct"]
    ]
    .corr()
    .iloc[0, 1]
)

corr_severe_rate = (
    supplier_reliability_audit[
        ["reliability_factor", "severe_delay_rate_pct"]
    ]
    .corr()
    .iloc[0, 1]
)


print(
    f"Suppliers Evaluated                    : "
    f"{len(supplier_reliability_audit):,}"
)

print(
    f"Reliability vs Average Delay Correlation: "
    f"{corr_avg_delay:.4f}"
)

print(
    f"Reliability vs Late PO Rate Correlation : "
    f"{corr_late_rate:.4f}"
)

print(
    f"Reliability vs Severe Delay Correlation : "
    f"{corr_severe_rate:.4f}"
)


print("\nSupplier Performance Summary:")

print(
    supplier_reliability_audit[
        [
            "reliability_factor",
            "avg_delay_days",
            "late_po_rate_pct",
            "severe_delay_rate_pct"
        ]
    ]
    .describe()
    .round(2)
)

Suppliers Evaluated                    : 100
Reliability vs Average Delay Correlation: -0.8513
Reliability vs Late PO Rate Correlation : -0.7553
Reliability vs Severe Delay Correlation : -0.8756

Supplier Performance Summary:
       reliability_factor  avg_delay_days  late_po_rate_pct  \
count              100.00          100.00            100.00   
mean                 0.86            0.82             34.09   
std                  0.09            0.97              9.30   
min                  0.57           -0.83             12.77   
25%                  0.83            0.14             28.16   
50%                  0.88            0.62             33.33   
75%                  0.93            1.34             38.83   
max                  0.98            4.05             61.36   

       severe_delay_rate_pct  
count                 100.00  
mean                   10.52  
std                     7.39  
min                     0.00  
25%                     5.08  
50%                 

### Preferred Supplier Status vs Delivery Performance

Compare observed procurement delivery performance between preferred and non-preferred suppliers.

The analysis evaluates:

- Number of suppliers
- Average Reliability Factor
- Average Supplier Delay
- Average Late PO Rate
- Average Severe Delay Rate

This helps validate whether the preferred-supplier designation is consistent with observed supplier performance.

In [110]:
# Add preferred supplier status to supplier performance data

preferred_supplier_audit = (
    supplier_reliability_audit
    .merge(
        suppliers[
            [
                "supplier_id",
                "preferred_supplier_flag"
            ]
        ],
        on="supplier_id",
        how="left"
    )
)


# Compare preferred vs non-preferred suppliers

preferred_performance_summary = (
    preferred_supplier_audit
    .groupby("preferred_supplier_flag")
    .agg(
        supplier_count=("supplier_id", "count"),
        avg_reliability_factor=("reliability_factor", "mean"),
        avg_delay_days=("avg_delay_days", "mean"),
        avg_late_po_rate_pct=("late_po_rate_pct", "mean"),
        avg_severe_delay_rate_pct=("severe_delay_rate_pct", "mean")
    )
    .round(2)
)


print("Preferred Supplier Performance Comparison:")

print(
    preferred_performance_summary
)

Preferred Supplier Performance Comparison:
                         supplier_count  avg_reliability_factor  \
preferred_supplier_flag                                           
N                                    60                    0.85   
Y                                    40                    0.88   

                         avg_delay_days  avg_late_po_rate_pct  \
preferred_supplier_flag                                         
N                                  0.96                 35.10   
Y                                  0.61                 32.58   

                         avg_severe_delay_rate_pct  
preferred_supplier_flag                             
N                                            11.63  
Y                                             8.86  


### Supplier Quality Rating vs Delivery Performance

Evaluate whether supplier `quality_rating` is associated with observed procurement delivery performance.

The analysis compares quality rating against:

- Average Supplier Delay
- Late PO Rate
- Severe Delay Rate
- Reliability Factor

The supplier with a missing quality rating is excluded from correlation calculations.

This analysis validates the business consistency of supplier master attributes without modifying any values.

In [111]:
# Add quality rating to supplier performance audit

supplier_quality_audit = (
    supplier_reliability_audit
    .merge(
        suppliers[
            [
                "supplier_id",
                "quality_rating"
            ]
        ],
        on="supplier_id",
        how="left"
    )
)


# Keep suppliers with known quality rating

supplier_quality_valid = supplier_quality_audit[
    supplier_quality_audit["quality_rating"].notna()
].copy()


# Calculate correlations

quality_vs_reliability = (
    supplier_quality_valid[
        ["quality_rating", "reliability_factor"]
    ]
    .corr()
    .iloc[0, 1]
)

quality_vs_avg_delay = (
    supplier_quality_valid[
        ["quality_rating", "avg_delay_days"]
    ]
    .corr()
    .iloc[0, 1]
)

quality_vs_late_rate = (
    supplier_quality_valid[
        ["quality_rating", "late_po_rate_pct"]
    ]
    .corr()
    .iloc[0, 1]
)

quality_vs_severe_delay = (
    supplier_quality_valid[
        ["quality_rating", "severe_delay_rate_pct"]
    ]
    .corr()
    .iloc[0, 1]
)


print(
    f"Total Suppliers                  : "
    f"{len(supplier_quality_audit):,}"
)

print(
    f"Suppliers With Quality Rating    : "
    f"{len(supplier_quality_valid):,}"
)

print(
    f"Missing Quality Rating           : "
    f"{supplier_quality_audit['quality_rating'].isna().sum():,}"
)


print(
    f"\nQuality vs Reliability Correlation : "
    f"{quality_vs_reliability:.4f}"
)

print(
    f"Quality vs Average Delay Correlation: "
    f"{quality_vs_avg_delay:.4f}"
)

print(
    f"Quality vs Late PO Rate Correlation : "
    f"{quality_vs_late_rate:.4f}"
)

print(
    f"Quality vs Severe Delay Correlation : "
    f"{quality_vs_severe_delay:.4f}"
)

Total Suppliers                  : 100
Suppliers With Quality Rating    : 99
Missing Quality Rating           : 1

Quality vs Reliability Correlation : -0.0122
Quality vs Average Delay Correlation: 0.0401
Quality vs Late PO Rate Correlation : 0.0381
Quality vs Severe Delay Correlation : 0.0357


### Product Price–Cost Margin Validity

Validate whether product selling prices are greater than their unit costs and evaluate the implied gross margin.

Expected relationship:

**Unit Price > Unit Cost**

Gross Margin % is calculated as:

**(Unit Price − Unit Cost) / Unit Price × 100**

Products with zero or negative margins may indicate pricing or master-data issues.

In [112]:
# Calculate product gross margin

product_margin = products[
    [
        "product_id",
        "unit_cost",
        "unit_price"
    ]
].copy()


product_margin["gross_margin"] = (
    product_margin["unit_price"]
    - product_margin["unit_cost"]
)


product_margin["gross_margin_pct"] = (
    product_margin["gross_margin"]
    / product_margin["unit_price"]
    * 100
)


# Validate pricing relationship

invalid_margin = (
    product_margin["unit_price"]
    <= product_margin["unit_cost"]
)


print(
    f"Total Products                : "
    f"{len(product_margin):,}"
)

print(
    f"Price <= Cost                 : "
    f"{invalid_margin.sum():,}"
)

print(
    f"Invalid Margin Percentage     : "
    f"{invalid_margin.mean() * 100:.2f}%"
)


print("\nGross Margin Percentage Summary:")

print(
    product_margin["gross_margin_pct"]
    .describe()
    .round(2)
)

Total Products                : 500
Price <= Cost                 : 0
Invalid Margin Percentage     : 0.00%

Gross Margin Percentage Summary:
count    500.00
mean      22.32
std        6.27
min       10.73
25%       17.26
50%       22.44
75%       28.01
max       32.43
Name: gross_margin_pct, dtype: float64


### Product Gross Margin Outlier Audit

Evaluate product gross margin percentages for potential statistical outliers using the Interquartile Range (IQR) method.

Outlier boundaries are defined as:

**Lower Bound = Q1 − 1.5 × IQR**

**Upper Bound = Q3 + 1.5 × IQR**

Statistical margin outliers are not automatically considered data-quality errors and should be assessed in business context.

In [113]:
# Calculate IQR for product gross margin percentage

margin_q1 = product_margin["gross_margin_pct"].quantile(0.25)
margin_q3 = product_margin["gross_margin_pct"].quantile(0.75)

margin_iqr = margin_q3 - margin_q1

margin_lower_bound = (
    margin_q1 - 1.5 * margin_iqr
)

margin_upper_bound = (
    margin_q3 + 1.5 * margin_iqr
)


# Identify margin outliers

margin_outliers = product_margin[
    (product_margin["gross_margin_pct"] < margin_lower_bound)
    |
    (product_margin["gross_margin_pct"] > margin_upper_bound)
]


print(f"Q1                    : {margin_q1:.2f}%")
print(f"Q3                    : {margin_q3:.2f}%")
print(f"IQR                   : {margin_iqr:.2f}")
print(f"Lower Outlier Bound   : {margin_lower_bound:.2f}%")
print(f"Upper Outlier Bound   : {margin_upper_bound:.2f}%")

print(
    f"\nTotal Products        : "
    f"{len(product_margin):,}"
)

print(
    f"Margin Outliers       : "
    f"{len(margin_outliers):,}"
)

print(
    f"Outlier Percentage    : "
    f"{len(margin_outliers) / len(product_margin) * 100:.2f}%"
)

if len(margin_outliers) > 0:
    print(
        f"Observed Outlier Range: "
        f"{margin_outliers['gross_margin_pct'].min():.2f}% "
        f"to {margin_outliers['gross_margin_pct'].max():.2f}%"
    )

Q1                    : 17.26%
Q3                    : 28.01%
IQR                   : 10.75
Lower Outlier Bound   : 1.14%
Upper Outlier Bound   : 44.13%

Total Products        : 500
Margin Outliers       : 0
Outlier Percentage    : 0.00%


### Discount-Adjusted Sales Margin Validation

Validate whether sales-order-item discounts result in transaction prices falling below product unit cost.

The effective selling price after discount is calculated as:

**Effective Unit Price = Unit Price × (1 − Discount %)**

The effective gross margin is then evaluated against the product unit cost.

This check identifies any sales lines that become loss-making after discounts.

In [114]:
# Add product unit cost to Sales Order Items

item_margin_audit = sales_order_items[
    [
        "order_item_id",
        "product_id",
        "unit_price",
        "discount_pct"
    ]
].merge(
    products[
        [
            "product_id",
            "unit_cost"
        ]
    ],
    on="product_id",
    how="left"
)


# Calculate effective selling price after discount

item_margin_audit["effective_unit_price"] = (
    item_margin_audit["unit_price"]
    * (1 - item_margin_audit["discount_pct"])
)


# Calculate effective unit margin

item_margin_audit["effective_unit_margin"] = (
    item_margin_audit["effective_unit_price"]
    - item_margin_audit["unit_cost"]
)


item_margin_audit["effective_margin_pct"] = (
    item_margin_audit["effective_unit_margin"]
    / item_margin_audit["effective_unit_price"]
    * 100
)


# Identify loss-making lines

loss_making_lines = (
    item_margin_audit["effective_unit_price"]
    < item_margin_audit["unit_cost"]
)

zero_margin_lines = (
    np.isclose(
        item_margin_audit["effective_unit_price"],
        item_margin_audit["unit_cost"],
        atol=0.01
    )
)


print(
    f"Total Sales Order Items : "
    f"{len(item_margin_audit):,}"
)

print(
    f"Loss-Making Lines       : "
    f"{loss_making_lines.sum():,}"
)

print(
    f"Zero-Margin Lines       : "
    f"{zero_margin_lines.sum():,}"
)

print(
    f"Loss-Making Percentage  : "
    f"{loss_making_lines.mean() * 100:.2f}%"
)


print("\nEffective Margin Percentage Summary:")

print(
    item_margin_audit["effective_margin_pct"]
    .describe()
    .round(2)
)

Total Sales Order Items : 61,397
Loss-Making Lines       : 500
Zero-Margin Lines       : 0
Loss-Making Percentage  : 0.81%

Effective Margin Percentage Summary:
count    61397.00
mean        18.47
std          7.75
min         -5.02
25%         12.82
50%         18.90
75%         24.54
max         32.43
Name: effective_margin_pct, dtype: float64


### Loss-Making Sales Lines by Discount Level

Analyze whether loss-making sales-order-item transactions are concentrated at specific discount levels.

This helps determine whether negative margins result from unusually aggressive discounting rather than incorrect product cost or price data.

No sales records or discount values are modified during this audit.

In [115]:
# Add loss-making indicator

item_margin_audit["loss_making_flag"] = np.where(
    item_margin_audit["effective_unit_price"]
    < item_margin_audit["unit_cost"],
    "Y",
    "N"
)


# Summarize by discount level

discount_margin_summary = (
    item_margin_audit
    .groupby("discount_pct")
    .agg(
        total_lines=("order_item_id", "count"),
        loss_making_lines=(
            "loss_making_flag",
            lambda x: (x == "Y").sum()
        ),
        avg_effective_margin_pct=(
            "effective_margin_pct",
            "mean"
        )
    )
    .reset_index()
)


discount_margin_summary["loss_making_rate_pct"] = (
    discount_margin_summary["loss_making_lines"]
    / discount_margin_summary["total_lines"]
    * 100
)


# Display discount as percentage

discount_margin_summary["discount_pct"] = (
    discount_margin_summary["discount_pct"] * 100
)


print(
    discount_margin_summary
    .round(2)
    .to_string(index=False)
)

 discount_pct  total_lines  loss_making_lines  avg_effective_margin_pct  loss_making_rate_pct
          0.0        21405                  0                     22.39                  0.00
          2.0         6083                  0                     20.95                  0.00
          5.0        13566                  0                     18.30                  0.00
          8.0         7377                  0                     15.78                  0.00
         10.0         9857                  0                     13.72                  0.00
         15.0         3109                500                      8.83                 16.08


### Products Responsible for Loss-Making Sales

Analyze which products generate loss-making sales lines after discounts.

This determines whether negative transaction margins are concentrated among products with relatively low base gross margins.

The analysis evaluates:

- Number of affected products
- Loss-making transaction count
- Product base gross margin
- Effective transaction margin after discount

No product prices, costs, or discounts are modified.

In [116]:
# Keep only loss-making sales lines

loss_lines = item_margin_audit[
    item_margin_audit["loss_making_flag"] == "Y"
].copy()


# Add product information

loss_lines = loss_lines.merge(
    products[
        [
            "product_id",
            "product_name",
            "category",
            "unit_cost",
            "unit_price"
        ]
    ],
    on="product_id",
    how="left",
    suffixes=("_item", "_product")
)


# Product base gross margin percentage

loss_lines["base_margin_pct"] = (
    (
        loss_lines["unit_price_product"]
        - loss_lines["unit_cost_product"]
    )
    / loss_lines["unit_price_product"]
    * 100
)


# Summarize affected products

loss_product_summary = (
    loss_lines
    .groupby(
        [
            "product_id",
            "product_name",
            "category"
        ]
    )
    .agg(
        loss_making_lines=("order_item_id", "count"),
        base_margin_pct=("base_margin_pct", "first"),
        avg_effective_margin_pct=("effective_margin_pct", "mean"),
        min_effective_margin_pct=("effective_margin_pct", "min")
    )
    .reset_index()
    .sort_values(
        "loss_making_lines",
        ascending=False
    )
)


print(
    f"Total Loss-Making Lines    : "
    f"{len(loss_lines):,}"
)

print(
    f"Affected Products          : "
    f"{loss_product_summary['product_id'].nunique():,}"
)


print("\nTop 10 Products by Loss-Making Line Count:")

print(
    loss_product_summary
    .head(10)
    .round(2)
    .to_string(index=False)
)

Total Loss-Making Lines    : 500
Affected Products          : 80

Top 10 Products by Loss-Making Line Count:
product_id               product_name               category  loss_making_lines  base_margin_pct  avg_effective_margin_pct  min_effective_margin_pct
   SKU0152      Wellness Product 0152 Health & Personal Care                 21            11.33                     -4.31                     -4.31
   SKU0492   Accessories Product 0492             Automotive                 17            12.42                     -3.03                     -3.03
   SKU0407     Furniture Product 0407        Office Supplies                 17            13.44                     -1.84                     -1.84
   SKU0057       Storage Product 0057         Home & Kitchen                 13            12.79                     -2.60                     -2.60
   SKU0240       Storage Product 0240         Home & Kitchen                 13            14.84                     -0.19                     -0.

### Loss-Making Sales by Product Category

Analyze the distribution of loss-making sales-order-item transactions across product categories.

This helps determine whether discount-driven negative margins are concentrated within specific product categories or distributed across the product portfolio.

No records are modified during this audit.

In [117]:
# Total sales lines by category

all_lines_category = (
    sales_order_items[
        ["order_item_id", "product_id"]
    ]
    .merge(
        products[
            ["product_id", "category"]
        ],
        on="product_id",
        how="left"
    )
    .groupby("category")
    .agg(
        total_sales_lines=("order_item_id", "count")
    )
)


# Loss-making lines by category

loss_category = (
    loss_lines
    .groupby("category")
    .agg(
        loss_making_lines=("order_item_id", "count")
    )
)


# Combine results

loss_category_summary = (
    all_lines_category
    .join(
        loss_category,
        how="left"
    )
)


loss_category_summary["loss_making_lines"] = (
    loss_category_summary["loss_making_lines"]
    .fillna(0)
    .astype(int)
)


# Calculate category-level loss-making rate

loss_category_summary["loss_making_rate_pct"] = (
    loss_category_summary["loss_making_lines"]
    / loss_category_summary["total_sales_lines"]
    * 100
).round(2)


loss_category_summary = (
    loss_category_summary
    .sort_values(
        "loss_making_rate_pct",
        ascending=False
    )
)


print(loss_category_summary)

                        total_sales_lines  loss_making_lines  \
category                                                       
Home & Kitchen                      11584                153   
Office Supplies                     11401                104   
Automotive                          10904                 77   
Electronics                         16347                110   
Health & Personal Care              11161                 56   

                        loss_making_rate_pct  
category                                      
Home & Kitchen                          1.32  
Office Supplies                         0.91  
Automotive                              0.71  
Electronics                             0.67  
Health & Personal Care                  0.50  


### Inactive Product Transaction Audit

Investigate whether products currently marked as inactive appear in historical sales transactions.

An inactive product may legitimately have historical sales activity if it was discontinued after those transactions.

This audit measures the transaction exposure of inactive products without treating historical usage as a data-quality error.

In [118]:
# Identify inactive products

inactive_products = products[
    products["active_flag"] == "N"
][
    [
        "product_id",
        "product_name",
        "category"
    ]
].copy()


# Find sales lines involving inactive products

inactive_product_sales = (
    sales_order_items[
        [
            "order_item_id",
            "order_id",
            "product_id",
            "order_date",
            "ordered_qty",
            "net_order_value"
        ]
    ]
    .merge(
        inactive_products,
        on="product_id",
        how="inner"
    )
)


print(
    f"Total Products                  : "
    f"{len(products):,}"
)

print(
    f"Inactive Products               : "
    f"{len(inactive_products):,}"
)

print(
    f"Inactive Products With Sales    : "
    f"{inactive_product_sales['product_id'].nunique():,}"
)

print(
    f"Sales Lines for Inactive Products: "
    f"{len(inactive_product_sales):,}"
)


if len(inactive_product_sales) > 0:

    inactive_sales_summary = (
        inactive_product_sales
        .groupby("product_id")
        .agg(
            sales_lines=("order_item_id", "count"),
            total_ordered_qty=("ordered_qty", "sum"),
            total_net_order_value=("net_order_value", "sum"),
            first_sale_date=("order_date", "min"),
            last_sale_date=("order_date", "max")
        )
        .sort_values(
            "sales_lines",
            ascending=False
        )
    )

    print("\nInactive Product Sales Summary:")

    print(
        inactive_sales_summary
        .round(2)
    )

Total Products                  : 500
Inactive Products               : 12
Inactive Products With Sales    : 12
Sales Lines for Inactive Products: 1,330

Inactive Product Sales Summary:
            sales_lines  total_ordered_qty  total_net_order_value  \
product_id                                                          
SKU0054             187               1521             3446175.08   
SKU0463             136               1353            11819475.24   
SKU0298             132               1308             8106086.68   
SKU0183             126                988              403350.66   
SKU0143             125                935              972576.93   
SKU0264             124               1089             6488236.93   
SKU0263             114                947             4282403.47   
SKU0205             112                853             2731748.55   
SKU0315              84                700             4344868.12   
SKU0346              81                584             

### Inactive Product Inventory Audit

Evaluate whether products currently marked as inactive still hold inventory across warehouse locations.

Inactive products with positive available inventory may represent discontinued, obsolete, or residual stock.

This is treated as an operational inventory observation rather than an automatic data-quality error because product deactivation dates are not available.

In [119]:
# Inventory records for inactive products

inactive_inventory = (
    inventory
    .merge(
        inactive_products[
            [
                "product_id",
                "product_name",
                "category"
            ]
        ],
        on="product_id",
        how="inner"
    )
)


# Product-level inventory summary

inactive_inventory_summary = (
    inactive_inventory
    .groupby("product_id")
    .agg(
        warehouse_count=("warehouse_id", "nunique"),
        total_on_hand_qty=("on_hand_qty", "sum"),
        total_allocated_qty=("allocated_qty", "sum"),
        total_available_qty=("available_qty", "sum"),
        total_inventory_value=("inventory_value", "sum")
    )
    .reset_index()
)


print(
    f"Inactive Products                     : "
    f"{len(inactive_products):,}"
)

print(
    f"Inactive Products With Inventory Rows : "
    f"{inactive_inventory['product_id'].nunique():,}"
)

print(
    f"Inactive Products With Available Stock: "
    f"{(inactive_inventory_summary['total_available_qty'] > 0).sum():,}"
)

print(
    f"Total Available Inactive Stock        : "
    f"{inactive_inventory_summary['total_available_qty'].sum():,.0f}"
)

print(
    f"Total Inactive Inventory Value        : "
    f"{inactive_inventory_summary['total_inventory_value'].sum():,.2f}"
)


print("\nInactive Product Inventory Summary:")

print(
    inactive_inventory_summary
    .sort_values(
        "total_inventory_value",
        ascending=False
    )
    .round(2)
    .to_string(index=False)
)

Inactive Products                     : 12
Inactive Products With Inventory Rows : 12
Inactive Products With Available Stock: 12
Total Available Inactive Stock        : 1,301
Total Inactive Inventory Value        : 4,659,916.55

Inactive Product Inventory Summary:
product_id  warehouse_count  total_on_hand_qty  total_allocated_qty  total_available_qty  total_inventory_value
   SKU0263                6                174                   22                  152              714134.28
   SKU0298                6                138                   19                  119              677087.34
   SKU0463                6                 91                   11                   80              579802.86
   SKU0264                6                118                   11                  107              575541.46
   SKU0315                6                 98                    9                   89              505789.76
   SKU0346                6                 92                 

### Inactive Products with Open Purchase Orders

Investigate whether products currently marked as inactive still have open purchase orders.

An inactive product with an open purchase order may indicate ongoing procurement for a discontinued product and therefore represents a potential operational control issue.

Because product deactivation dates are unavailable, such records are flagged for review rather than automatically treated as data-quality errors.

In [120]:
# Purchase Orders involving inactive products

inactive_product_pos = (
    purchase_orders
    .merge(
        inactive_products[
            [
                "product_id",
                "product_name",
                "category"
            ]
        ],
        on="product_id",
        how="inner"
    )
)


# Open POs for inactive products

inactive_open_pos = inactive_product_pos[
    inactive_product_pos["po_status"] == "Open"
].copy()


print(
    f"Inactive Products                     : "
    f"{len(inactive_products):,}"
)

print(
    f"Inactive Products With Any POs        : "
    f"{inactive_product_pos['product_id'].nunique():,}"
)

print(
    f"Total POs for Inactive Products       : "
    f"{len(inactive_product_pos):,}"
)

print(
    f"Open POs for Inactive Products        : "
    f"{len(inactive_open_pos):,}"
)

print(
    f"Inactive Products With Open POs       : "
    f"{inactive_open_pos['product_id'].nunique():,}"
)


if len(inactive_open_pos) > 0:

    print(
        f"Open PO Ordered Quantity              : "
        f"{inactive_open_pos['ordered_qty'].sum():,.0f}"
    )

    print(
        f"Open PO Value                         : "
        f"{inactive_open_pos['po_value'].sum():,.2f}"
    )


    print("\nOpen Purchase Orders for Inactive Products:")

    print(
        inactive_open_pos[
            [
                "po_id",
                "product_id",
                "warehouse_id",
                "po_date",
                "ordered_qty",
                "po_value",
                "expected_date",
                "po_status"
            ]
        ]
        .sort_values("po_date")
        .to_string(index=False)
    )

Inactive Products                     : 12
Inactive Products With Any POs        : 12
Total POs for Inactive Products       : 180
Open POs for Inactive Products        : 2
Inactive Products With Open POs       : 1
Open PO Ordered Quantity              : 798
Open PO Value                         : 1,382,654.70

Open Purchase Orders for Inactive Products:
   po_id product_id warehouse_id    po_date  ordered_qty  po_value expected_date po_status
PO000113    SKU0054         WH04 2026-09-12          383 663604.95    2026-09-30      Open
PO001281    SKU0054         WH01 2026-09-12          415 719049.75    2026-09-30      Open


### Inactive Product SKU0054 Exposure Review

Perform a focused review of SKU0054 because it is currently marked inactive while still having open purchase orders.

The review summarizes:

- Current inventory
- Available inventory
- Inventory value
- Open purchase-order quantity and value
- Historical sales quantity and value

This record is flagged for business review rather than automatically treated as a data-quality error because the product deactivation date is unavailable.

In [121]:
focus_product = "SKU0054"


# Current inventory exposure

focus_inventory = inventory[
    inventory["product_id"] == focus_product
]

total_on_hand = focus_inventory["on_hand_qty"].sum()
total_allocated = focus_inventory["allocated_qty"].sum()
total_available = focus_inventory["available_qty"].sum()
total_inventory_value = focus_inventory["inventory_value"].sum()


# Open procurement exposure

focus_open_po = purchase_orders[
    (purchase_orders["product_id"] == focus_product)
    & (purchase_orders["po_status"] == "Open")
]

open_po_qty = focus_open_po["ordered_qty"].sum()
open_po_value = focus_open_po["po_value"].sum()


# Historical sales exposure

focus_sales = sales_order_items[
    sales_order_items["product_id"] == focus_product
]

historical_sales_qty = focus_sales["ordered_qty"].sum()
historical_sales_value = focus_sales["net_order_value"].sum()


print(f"Product ID                    : {focus_product}")

print(
    f"Current On-Hand Quantity      : "
    f"{total_on_hand:,.0f}"
)

print(
    f"Current Allocated Quantity    : "
    f"{total_allocated:,.0f}"
)

print(
    f"Current Available Quantity    : "
    f"{total_available:,.0f}"
)

print(
    f"Current Inventory Value       : "
    f"{total_inventory_value:,.2f}"
)

print(
    f"\nOpen PO Quantity              : "
    f"{open_po_qty:,.0f}"
)

print(
    f"Open PO Value                 : "
    f"{open_po_value:,.2f}"
)

print(
    f"\nHistorical Sales Lines        : "
    f"{len(focus_sales):,}"
)

print(
    f"Historical Ordered Quantity   : "
    f"{historical_sales_qty:,.0f}"
)

print(
    f"Historical Net Order Value    : "
    f"{historical_sales_value:,.2f}"
)

Product ID                    : SKU0054
Current On-Hand Quantity      : 165
Current Allocated Quantity    : 14
Current Available Quantity    : 151
Current Inventory Value       : 285,887.25

Open PO Quantity              : 798
Open PO Value                 : 1,382,654.70

Historical Sales Lines        : 187
Historical Ordered Quantity   : 1,521
Historical Net Order Value    : 3,446,175.08


### Inventory Allocation Validity Audit

Validate whether allocated inventory exceeds the physical on-hand quantity.

Expected relationship:

**Allocated Quantity ≤ On-Hand Quantity**

Allocated quantity greater than on-hand inventory would indicate an inconsistent inventory state.

The relationship between on-hand, allocated, and available inventory is also reviewed for zero-stock records.

In [122]:
# Validate inventory allocation

allocation_exceeds_onhand = (
    inventory["allocated_qty"] > inventory["on_hand_qty"]
)


print(
    f"Total Inventory Records        : "
    f"{len(inventory):,}"
)

print(
    f"Allocation > On-Hand           : "
    f"{allocation_exceeds_onhand.sum():,}"
)

print(
    f"Invalid Percentage             : "
    f"{allocation_exceeds_onhand.mean() * 100:.2f}%"
)


# Additional zero-stock consistency check

zero_onhand = inventory["on_hand_qty"] == 0

zero_onhand_with_allocation = (
    zero_onhand
    & (inventory["allocated_qty"] > 0)
)

zero_onhand_with_available = (
    zero_onhand
    & (inventory["available_qty"] > 0)
)


print(
    f"\nZero On-Hand Records           : "
    f"{zero_onhand.sum():,}"
)

print(
    f"Zero On-Hand With Allocation   : "
    f"{zero_onhand_with_allocation.sum():,}"
)

print(
    f"Zero On-Hand With Availability : "
    f"{zero_onhand_with_available.sum():,}"
)

Total Inventory Records        : 3,000
Allocation > On-Hand           : 0
Invalid Percentage             : 0.00%

Zero On-Hand Records           : 219
Zero On-Hand With Allocation   : 0
Zero On-Hand With Availability : 0


### Inventory Stockout Status vs Zero Availability

Validate the consistency between zero available inventory and the `Stockout` inventory status.

Expected relationships:

- **Available Quantity = 0 → Stock Status = Stockout**
- **Stock Status = Stockout → Available Quantity = 0**

This ensures that stockout classification accurately represents the current inventory state.

In [123]:
# Identify zero-availability inventory records

zero_available = (
    inventory["available_qty"] == 0
)

stockout_status = (
    inventory["stock_status"] == "Stockout"
)


# Zero available but not classified as Stockout

zero_available_not_stockout = (
    zero_available
    & ~stockout_status
)


# Classified as Stockout but still has available inventory

stockout_with_available = (
    stockout_status
    & (inventory["available_qty"] > 0)
)


print(
    f"Total Inventory Records          : "
    f"{len(inventory):,}"
)

print(
    f"Zero Available Records           : "
    f"{zero_available.sum():,}"
)

print(
    f"Stockout Status Records          : "
    f"{stockout_status.sum():,}"
)

print(
    f"Zero Available but Not Stockout  : "
    f"{zero_available_not_stockout.sum():,}"
)

print(
    f"Stockout With Available Quantity : "
    f"{stockout_with_available.sum():,}"
)

Total Inventory Records          : 3,000
Zero Available Records           : 240
Stockout Status Records          : 240
Zero Available but Not Stockout  : 0
Stockout With Available Quantity : 0


### Warehouse Inventory Capacity Utilization Audit

Validate whether total physical on-hand inventory exceeds the stated capacity of each warehouse.

Expected relationship:

**Total Warehouse On-Hand Quantity ≤ Warehouse Capacity**

Capacity utilization is calculated as:

**Capacity Utilization % = Total On-Hand Quantity / Capacity Units × 100**

This check identifies potential warehouse-capacity inconsistencies.

In [124]:
# Aggregate current inventory by warehouse

warehouse_inventory = (
    inventory
    .groupby("warehouse_id")
    .agg(
        total_on_hand_qty=("on_hand_qty", "sum"),
        total_allocated_qty=("allocated_qty", "sum"),
        total_available_qty=("available_qty", "sum")
    )
    .reset_index()
)


# Add warehouse capacity

warehouse_capacity_audit = (
    warehouse_inventory
    .merge(
        warehouses[
            [
                "warehouse_id",
                "warehouse_name",
                "capacity_units"
            ]
        ],
        on="warehouse_id",
        how="left"
    )
)


# Calculate capacity utilization

warehouse_capacity_audit["capacity_utilization_pct"] = (
    warehouse_capacity_audit["total_on_hand_qty"]
    / warehouse_capacity_audit["capacity_units"]
    * 100
)


# Identify capacity violations

warehouse_capacity_audit["capacity_exceeded"] = (
    warehouse_capacity_audit["total_on_hand_qty"]
    > warehouse_capacity_audit["capacity_units"]
)


print(
    f"Total Warehouses          : "
    f"{len(warehouse_capacity_audit):,}"
)

print(
    f"Warehouses Over Capacity  : "
    f"{warehouse_capacity_audit['capacity_exceeded'].sum():,}"
)


print("\nWarehouse Capacity Summary:")

print(
    warehouse_capacity_audit[
        [
            "warehouse_id",
            "warehouse_name",
            "capacity_units",
            "total_on_hand_qty",
            "capacity_utilization_pct",
            "capacity_exceeded"
        ]
    ]
    .round(2)
    .to_string(index=False)
)

Total Warehouses          : 6
Warehouses Over Capacity  : 0

Warehouse Capacity Summary:
warehouse_id    warehouse_name  capacity_units  total_on_hand_qty  capacity_utilization_pct  capacity_exceeded
        WH01 Bengaluru Central           65000              14433                     22.20              False
        WH02       Mumbai West           55000              14356                     26.10              False
        WH03         Delhi NCR           60000              13810                     23.02              False
        WH04   Hyderabad South           48000              13952                     29.07              False
        WH05       Chennai Hub           50000              15121                     30.24              False
        WH06           Pune DC           45000              13633                     30.30              False


### Warehouse-Level Inventory Quantity Reconciliation

Validate inventory quantities after aggregation to the warehouse level.

Expected relationship:

**Total On-Hand Quantity = Total Allocated Quantity + Total Available Quantity**

This check confirms that inventory quantity consistency is preserved after warehouse-level aggregation.

In [125]:
# Warehouse-level inventory reconciliation

warehouse_qty_check = (
    inventory
    .groupby("warehouse_id")
    .agg(
        total_on_hand_qty=("on_hand_qty", "sum"),
        total_allocated_qty=("allocated_qty", "sum"),
        total_available_qty=("available_qty", "sum")
    )
    .reset_index()
)


# Reconstruct on-hand inventory

warehouse_qty_check["reconstructed_on_hand"] = (
    warehouse_qty_check["total_allocated_qty"]
    + warehouse_qty_check["total_available_qty"]
)


# Calculate difference

warehouse_qty_check["difference"] = (
    warehouse_qty_check["total_on_hand_qty"]
    - warehouse_qty_check["reconstructed_on_hand"]
)


# Identify mismatches

warehouse_qty_check["mismatch"] = (
    warehouse_qty_check["difference"] != 0
)


print(
    f"Total Warehouses              : "
    f"{len(warehouse_qty_check):,}"
)

print(
    f"Warehouse Quantity Mismatches : "
    f"{warehouse_qty_check['mismatch'].sum():,}"
)


print("\nWarehouse Quantity Reconciliation:")

print(
    warehouse_qty_check.to_string(index=False)
)

Total Warehouses              : 6
Warehouse Quantity Mismatches : 0

Warehouse Quantity Reconciliation:
warehouse_id  total_on_hand_qty  total_allocated_qty  total_available_qty  reconstructed_on_hand  difference  mismatch
        WH01              14433                 1173                13260                  14433           0     False
        WH02              14356                 1194                13162                  14356           0     False
        WH03              13810                 1198                12612                  13810           0     False
        WH04              13952                 1164                12788                  13952           0     False
        WH05              15121                 1177                13944                  15121           0     False
        WH06              13633                 1180                12453                  13633           0     False


### Warehouse-Level Inventory Value Audit

Review the total inventory value held at each warehouse.

This check summarizes current inventory value by warehouse and validates whether any positive on-hand inventory records have zero or negative inventory value.

The results provide a warehouse-level view of inventory financial exposure while also confirming value consistency.

In [126]:
# Warehouse-level inventory value summary

warehouse_value_summary = (
    inventory
    .groupby("warehouse_id")
    .agg(
        total_on_hand_qty=("on_hand_qty", "sum"),
        total_inventory_value=("inventory_value", "sum")
    )
    .reset_index()
)


# Add warehouse information

warehouse_value_summary = (
    warehouse_value_summary
    .merge(
        warehouses[
            [
                "warehouse_id",
                "warehouse_name"
            ]
        ],
        on="warehouse_id",
        how="left"
    )
)


# Check invalid inventory value conditions

positive_stock_invalid_value = (
    (inventory["on_hand_qty"] > 0)
    & (inventory["inventory_value"] <= 0)
)


print(
    f"Total Inventory Records                    : "
    f"{len(inventory):,}"
)

print(
    f"Positive Stock With Zero/Negative Value    : "
    f"{positive_stock_invalid_value.sum():,}"
)

print(
    f"Total Inventory Value                      : "
    f"{inventory['inventory_value'].sum():,.2f}"
)


print("\nWarehouse Inventory Value Summary:")

print(
    warehouse_value_summary[
        [
            "warehouse_id",
            "warehouse_name",
            "total_on_hand_qty",
            "total_inventory_value"
        ]
    ]
    .sort_values(
        "total_inventory_value",
        ascending=False
    )
    .round(2)
    .to_string(index=False)
)

Total Inventory Records                    : 3,000
Positive Stock With Zero/Negative Value    : 0
Total Inventory Value                      : 285,985,182.04

Warehouse Inventory Value Summary:
warehouse_id    warehouse_name  total_on_hand_qty  total_inventory_value
        WH05       Chennai Hub              15121            51086797.70
        WH01 Bengaluru Central              14433            48514594.37
        WH02       Mumbai West              14356            47404063.86
        WH04   Hyderabad South              13952            47126003.30
        WH03         Delhi NCR              13810            46495691.15
        WH06           Pune DC              13633            45358031.66


### Missing Customer Credit Tier — Sales Impact Audit

Evaluate the business impact of customers with missing credit-tier values.

The analysis measures:

- Number of affected customers
- Number of sales orders
- Total order value
- Share of total orders
- Share of total sales value

This helps determine whether the missing credit-tier records are financially material before defining the cleaning strategy.

In [127]:
# Customers with missing credit tier

missing_credit_customers = customers[
    customers["credit_tier"].isna()
].copy()


# Sales orders belonging to those customers

missing_credit_sales = sales_orders[
    sales_orders["customer_id"].isin(
        missing_credit_customers["customer_id"]
    )
]


# Calculate impact

affected_customers = len(missing_credit_customers)

affected_orders = len(missing_credit_sales)

affected_order_value = (
    missing_credit_sales["order_value"].sum()
)

order_share_pct = (
    affected_orders
    / len(sales_orders)
    * 100
)

sales_value_share_pct = (
    affected_order_value
    / sales_orders["order_value"].sum()
    * 100
)


print(
    f"Customers Missing Credit Tier : "
    f"{affected_customers:,}"
)

print(
    f"Orders From These Customers   : "
    f"{affected_orders:,}"
)

print(
    f"Total Order Value             : "
    f"{affected_order_value:,.2f}"
)

print(
    f"Share of Total Orders         : "
    f"{order_share_pct:.2f}%"
)

print(
    f"Share of Total Order Value    : "
    f"{sales_value_share_pct:.2f}%"
)

Customers Missing Credit Tier : 20
Orders From These Customers   : 174
Total Order Value             : 14,179,704.33
Share of Total Orders         : 0.70%
Share of Total Order Value    : 0.69%


### Missing Product Subcategory — Sales Impact Audit

Evaluate the business impact of products with missing `subcategory` values.

The analysis measures:

- Number of affected products
- Sales lines involving those products
- Ordered quantity
- Net order value
- Share of total sales lines
- Share of total order value

This helps determine the analytical impact of the missing product classification before defining the cleaning strategy.

In [128]:
# Products with missing subcategory

missing_subcategory_products = products[
    products["subcategory"].isna()
]["product_id"]


# Sales lines involving missing-subcategory products

missing_subcategory_sales = sales_order_items[
    sales_order_items["product_id"].isin(
        missing_subcategory_products
    )
]


# Calculate business impact

affected_products = (
    len(missing_subcategory_products)
)

affected_sales_lines = (
    len(missing_subcategory_sales)
)

affected_quantity = (
    missing_subcategory_sales["ordered_qty"].sum()
)

affected_sales_value = (
    missing_subcategory_sales["net_order_value"].sum()
)


sales_line_share_pct = (
    affected_sales_lines
    / len(sales_order_items)
    * 100
)

sales_value_share_pct = (
    affected_sales_value
    / sales_order_items["net_order_value"].sum()
    * 100
)


print(
    f"Products Missing Subcategory : "
    f"{affected_products:,}"
)

print(
    f"Sales Lines Affected         : "
    f"{affected_sales_lines:,}"
)

print(
    f"Ordered Quantity             : "
    f"{affected_quantity:,.0f}"
)

print(
    f"Net Order Value              : "
    f"{affected_sales_value:,.2f}"
)

print(
    f"Share of Sales Lines         : "
    f"{sales_line_share_pct:.2f}%"
)

print(
    f"Share of Sales Value         : "
    f"{sales_value_share_pct:.2f}%"
)

Products Missing Subcategory : 3
Sales Lines Affected         : 353
Ordered Quantity             : 3,044
Net Order Value              : 5,315,335.55
Share of Sales Lines         : 0.57%
Share of Sales Value         : 0.26%


### Missing Supplier Quality Rating — Procurement Impact Audit

Evaluate the procurement exposure associated with the supplier whose `quality_rating` is missing.

The analysis measures:

- Number of affected suppliers
- Purchase orders involving the supplier
- Share of total purchase orders
- Total PO value
- Share of total PO value
- Received and open PO exposure

This helps determine the business significance of the missing supplier attribute before defining the cleaning strategy.

In [129]:
# Supplier with missing quality rating

missing_quality_supplier_ids = suppliers[
    suppliers["quality_rating"].isna()
]["supplier_id"]


# Purchase orders involving the supplier

missing_quality_pos = purchase_orders[
    purchase_orders["supplier_id"].isin(
        missing_quality_supplier_ids
    )
]


# Procurement impact

affected_suppliers = (
    len(missing_quality_supplier_ids)
)

affected_pos = (
    len(missing_quality_pos)
)

affected_po_value = (
    missing_quality_pos["po_value"].sum()
)

po_share_pct = (
    affected_pos
    / len(purchase_orders)
    * 100
)

po_value_share_pct = (
    affected_po_value
    / purchase_orders["po_value"].sum()
    * 100
)


# Receipt status

received_missing_quality = (
    missing_quality_pos["actual_receipt_date"].notna()
)

open_missing_quality = (
    missing_quality_pos["po_status"] == "Open"
)


print(
    f"Suppliers Missing Quality Rating : "
    f"{affected_suppliers:,}"
)

print(
    f"Purchase Orders Affected         : "
    f"{affected_pos:,}"
)

print(
    f"Share of Purchase Orders         : "
    f"{po_share_pct:.2f}%"
)

print(
    f"Total PO Value                   : "
    f"{affected_po_value:,.2f}"
)

print(
    f"Share of Total PO Value          : "
    f"{po_value_share_pct:.2f}%"
)

print(
    f"Received POs                     : "
    f"{received_missing_quality.sum():,}"
)

print(
    f"Open POs                         : "
    f"{open_missing_quality.sum():,}"
)

Suppliers Missing Quality Rating : 1
Purchase Orders Affected         : 10
Share of Purchase Orders         : 0.12%
Total PO Value                   : 10,490,155.90
Share of Total PO Value          : 0.16%
Received POs                     : 9
Open POs                         : 1


### Purchase Order Supplier Quality Rating Coverage

Validate whether Purchase Orders can be linked to a supplier with a populated quality rating.

This checks the downstream analytical coverage of the supplier `quality_rating` attribute.

The one supplier with a missing quality rating is expected to produce affected Purchase Orders.

In [130]:
# Join Purchase Orders with supplier quality rating

po_quality_coverage = purchase_orders[
    [
        "po_id",
        "supplier_id",
        "po_value"
    ]
].merge(
    suppliers[
        [
            "supplier_id",
            "quality_rating"
        ]
    ],
    on="supplier_id",
    how="left"
)


# Identify POs without quality rating

missing_po_quality = (
    po_quality_coverage["quality_rating"].isna()
)


print(
    f"Total Purchase Orders              : "
    f"{len(po_quality_coverage):,}"
)

print(
    f"POs Without Supplier Quality Rating: "
    f"{missing_po_quality.sum():,}"
)

print(
    f"Missing Percentage                  : "
    f"{missing_po_quality.mean() * 100:.2f}%"
)

print(
    f"PO Value Without Quality Rating    : "
    f"{po_quality_coverage.loc[missing_po_quality, 'po_value'].sum():,.2f}"
)

Total Purchase Orders              : 8,500
POs Without Supplier Quality Rating: 10
Missing Percentage                  : 0.12%
PO Value Without Quality Rating    : 10,490,155.90


### Sales Order Customer Credit Tier Coverage

Validate whether Sales Orders can be linked to a customer with a populated credit tier.

This checks the downstream analytical coverage of the customer `credit_tier` attribute and confirms how many Sales Orders are affected by the 20 customers with missing credit tiers.

In [131]:
# Join Sales Orders with customer credit tier

so_credit_coverage = sales_orders[
    [
        "order_id",
        "customer_id",
        "order_value"
    ]
].merge(
    customers[
        [
            "customer_id",
            "credit_tier"
        ]
    ],
    on="customer_id",
    how="left"
)


# Identify orders without credit tier

missing_so_credit = (
    so_credit_coverage["credit_tier"].isna()
)


print(
    f"Total Sales Orders             : "
    f"{len(so_credit_coverage):,}"
)

print(
    f"Orders Without Credit Tier     : "
    f"{missing_so_credit.sum():,}"
)

print(
    f"Missing Percentage             : "
    f"{missing_so_credit.mean() * 100:.2f}%"
)

print(
    f"Order Value Without Credit Tier: "
    f"{so_credit_coverage.loc[missing_so_credit, 'order_value'].sum():,.2f}"
)

Total Sales Orders             : 25,000
Orders Without Credit Tier     : 174
Missing Percentage             : 0.70%
Order Value Without Credit Tier: 14,179,704.33


### Sales Order Product Subcategory Coverage

Validate whether Sales Order Items can be linked to products with a populated subcategory.

This checks the downstream analytical coverage of the product `subcategory` attribute and quantifies Sales Order Items affected by missing product classification.

In [132]:
# Join Sales Order Items with product subcategory

so_subcategory_coverage = sales_order_items[
    [
        "order_item_id",
        "product_id",
        "net_order_value"
    ]
].merge(
    products[
        [
            "product_id",
            "subcategory"
        ]
    ],
    on="product_id",
    how="left"
)


# Identify sales lines without subcategory

missing_so_subcategory = (
    so_subcategory_coverage["subcategory"].isna()
)


print(
    f"Total Sales Order Items          : "
    f"{len(so_subcategory_coverage):,}"
)

print(
    f"Items Without Product Subcategory: "
    f"{missing_so_subcategory.sum():,}"
)

print(
    f"Missing Percentage                : "
    f"{missing_so_subcategory.mean() * 100:.2f}%"
)

print(
    f"Net Order Value Without Subcategory: "
    f"{so_subcategory_coverage.loc[
        missing_so_subcategory,
        'net_order_value'
    ].sum():,.2f}"
)

Total Sales Order Items          : 61,397
Items Without Product Subcategory: 353
Missing Percentage                : 0.57%
Net Order Value Without Subcategory: 5,315,335.55


### Sales Order Product Active Status Coverage

Validate whether Sales Order Items can be linked to products with a populated active-status flag.

This checks downstream coverage of the product `active_flag` attribute and identifies any Sales Order Items associated with products having missing or invalid active-status information.

In [133]:
# Join Sales Order Items with product active status

so_active_coverage = sales_order_items[
    [
        "order_item_id",
        "product_id",
        "net_order_value"
    ]
].merge(
    products[
        [
            "product_id",
            "active_flag"
        ]
    ],
    on="product_id",
    how="left"
)


# Identify missing or invalid active flags

missing_active_flag = (
    so_active_coverage["active_flag"].isna()
)

invalid_active_flag = ~(
    so_active_coverage["active_flag"].isin(["Y", "N"])
)


print(
    f"Total Sales Order Items       : "
    f"{len(so_active_coverage):,}"
)

print(
    f"Items Without Active Flag     : "
    f"{missing_active_flag.sum():,}"
)

print(
    f"Items With Invalid Active Flag: "
    f"{invalid_active_flag.sum():,}"
)

print(
    f"Missing Active Flag Percentage: "
    f"{missing_active_flag.mean() * 100:.2f}%"
)

print(
    f"Net Order Value Without Flag  : "
    f"{so_active_coverage.loc[
        missing_active_flag,
        'net_order_value'
    ].sum():,.2f}"
)

Total Sales Order Items       : 61,397
Items Without Active Flag     : 0
Items With Invalid Active Flag: 0
Missing Active Flag Percentage: 0.00%
Net Order Value Without Flag  : 0.00


### Purchase Order Product Active Status Coverage

Validate whether Purchase Orders can be linked to products with a populated and valid active-status flag.

This checks downstream coverage of the product `active_flag` attribute within procurement transactions.

In [134]:
# Join Purchase Orders with product active status

po_active_coverage = purchase_orders[
    [
        "po_id",
        "product_id",
        "po_value"
    ]
].merge(
    products[
        [
            "product_id",
            "active_flag"
        ]
    ],
    on="product_id",
    how="left"
)


# Identify missing or invalid active flags

missing_po_active_flag = (
    po_active_coverage["active_flag"].isna()
)

invalid_po_active_flag = ~(
    po_active_coverage["active_flag"].isin(["Y", "N"])
)


print(
    f"Total Purchase Orders        : "
    f"{len(po_active_coverage):,}"
)

print(
    f"POs Without Active Flag      : "
    f"{missing_po_active_flag.sum():,}"
)

print(
    f"POs With Invalid Active Flag : "
    f"{invalid_po_active_flag.sum():,}"
)

print(
    f"Missing Active Flag Percentage: "
    f"{missing_po_active_flag.mean() * 100:.2f}%"
)

print(
    f"PO Value Without Active Flag : "
    f"{po_active_coverage.loc[
        missing_po_active_flag,
        'po_value'
    ].sum():,.2f}"
)

Total Purchase Orders        : 8,500
POs Without Active Flag      : 0
POs With Invalid Active Flag : 0
Missing Active Flag Percentage: 0.00%
PO Value Without Active Flag : 0.00


### Inventory Product Active Status Coverage

Validate whether Inventory records can be linked to products with a populated and valid active-status flag.

This checks downstream coverage of the product `active_flag` attribute within the inventory dataset.

In [137]:
# Join Inventory with product active status

inventory_active_coverage = inventory[
    [
        "product_id",
        "warehouse_id",
        "inventory_value"
    ]
].merge(
    products[
        [
            "product_id",
            "active_flag"
        ]
    ],
    on="product_id",
    how="left"
)


# Identify missing or invalid active flags

missing_inventory_active_flag = (
    inventory_active_coverage["active_flag"].isna()
)

invalid_inventory_active_flag = ~(
    inventory_active_coverage["active_flag"].isin(["Y", "N"])
)


print(
    f"Total Inventory Records         : "
    f"{len(inventory_active_coverage):,}"
)

print(
    f"Records Without Active Flag     : "
    f"{missing_inventory_active_flag.sum():,}"
)

print(
    f"Records With Invalid Active Flag: "
    f"{invalid_inventory_active_flag.sum():,}"
)

print(
    f"Missing Active Flag Percentage  : "
    f"{missing_inventory_active_flag.mean() * 100:.2f}%"
)

print(
    f"Inventory Value Without Flag    : "
    f"{inventory_active_coverage.loc[
        missing_inventory_active_flag,
        'inventory_value'
    ].sum():,.2f}"
)

Total Inventory Records         : 3,000
Records Without Active Flag     : 0
Records With Invalid Active Flag: 0
Missing Active Flag Percentage  : 0.00%
Inventory Value Without Flag    : 0.00


### Inventory Transaction Type Domain Validation

Validate whether all inventory transaction records contain an approved transaction type.

Expected transaction types:

- SALE_ISSUE
- PO_RECEIPT
- RETURN_IN
- ADJUSTMENT_IN
- ADJUSTMENT_OUT

Any missing or unexpected transaction type would represent a categorical data-quality issue.

In [138]:
# Expected inventory transaction types

valid_transaction_types = [
    "SALE_ISSUE",
    "PO_RECEIPT",
    "RETURN_IN",
    "ADJUSTMENT_IN",
    "ADJUSTMENT_OUT"
]


# Missing transaction types

missing_transaction_type = (
    inventory_transactions["transaction_type"].isna()
)


# Invalid / unexpected transaction types

invalid_transaction_type = (
    ~inventory_transactions["transaction_type"].isin(
        valid_transaction_types
    )
    & inventory_transactions["transaction_type"].notna()
)


print(
    f"Total Inventory Transactions : "
    f"{len(inventory_transactions):,}"
)

print(
    f"Missing Transaction Types     : "
    f"{missing_transaction_type.sum():,}"
)

print(
    f"Invalid Transaction Types     : "
    f"{invalid_transaction_type.sum():,}"
)


print("\nTransaction Type Distribution:")

print(
    inventory_transactions["transaction_type"]
    .value_counts(dropna=False)
    .to_string()
)

Total Inventory Transactions : 110,045
Missing Transaction Types     : 0
Invalid Transaction Types     : 0

Transaction Type Distribution:
transaction_type
SALE_ISSUE        62848
PO_RECEIPT        27423
RETURN_IN          8759
ADJUSTMENT_IN      5511
ADJUSTMENT_OUT     5504


### Sales Order Priority Domain Validation

Validate whether all Sales Orders contain an approved priority classification.

Expected priority values:

- Low
- Normal
- High
- Urgent

Any missing or unexpected priority value would represent a categorical data-quality issue.

In [139]:
# Expected Sales Order priority values

valid_priorities = [
    "Low",
    "Normal",
    "High",
    "Urgent"
]


# Missing priority values

missing_priority = (
    sales_orders["priority"].isna()
)


# Invalid priority values

invalid_priority = (
    ~sales_orders["priority"].isin(valid_priorities)
    & sales_orders["priority"].notna()
)


print(
    f"Total Sales Orders      : "
    f"{len(sales_orders):,}"
)

print(
    f"Missing Priority Values : "
    f"{missing_priority.sum():,}"
)

print(
    f"Invalid Priority Values : "
    f"{invalid_priority.sum():,}"
)


print("\nPriority Distribution:")

print(
    sales_orders["priority"]
    .value_counts(dropna=False)
    .to_string()
)

Total Sales Orders      : 25,000
Missing Priority Values : 0
Invalid Priority Values : 0

Priority Distribution:
priority
Normal    14557
High       6359
Urgent     2049
Low        2035


### Sales Channel Domain Validation

Validate whether all Sales Orders contain an approved sales-channel classification.

Expected sales channels:

- Direct
- Distributor
- Marketplace
- Online

Any missing or unexpected sales-channel value would represent a categorical data-quality issue.

In [140]:
# Expected Sales Order channels

valid_sales_channels = [
    "Direct",
    "Distributor",
    "Marketplace",
    "Online"
]


# Missing channel values

missing_sales_channel = (
    sales_orders["sales_channel"].isna()
)


# Invalid channel values

invalid_sales_channel = (
    ~sales_orders["sales_channel"].isin(valid_sales_channels)
    & sales_orders["sales_channel"].notna()
)


print(
    f"Total Sales Orders          : "
    f"{len(sales_orders):,}"
)

print(
    f"Missing Sales Channel Values: "
    f"{missing_sales_channel.sum():,}"
)

print(
    f"Invalid Sales Channel Values: "
    f"{invalid_sales_channel.sum():,}"
)


print("\nSales Channel Distribution:")

print(
    sales_orders["sales_channel"]
    .value_counts(dropna=False)
    .to_string()
)

Total Sales Orders          : 25,000
Missing Sales Channel Values: 0
Invalid Sales Channel Values: 0

Sales Channel Distribution:
sales_channel
Online         7851
Direct         7195
Distributor    5469
Marketplace    4485


### Customer Segment Domain Validation

Validate whether all customers contain an approved customer-segment classification.

Expected customer segments:

- Distributor
- Enterprise
- Retail
- SMB

Any missing or unexpected customer-segment value would represent a categorical data-quality issue.

In [141]:
# Expected customer segments

valid_customer_segments = [
    "Distributor",
    "Enterprise",
    "Retail",
    "SMB"
]


# Missing segment values

missing_customer_segment = (
    customers["customer_segment"].isna()
)


# Invalid segment values

invalid_customer_segment = (
    ~customers["customer_segment"].isin(valid_customer_segments)
    & customers["customer_segment"].notna()
)


print(
    f"Total Customers                 : "
    f"{len(customers):,}"
)

print(
    f"Missing Customer Segment Values : "
    f"{missing_customer_segment.sum():,}"
)

print(
    f"Invalid Customer Segment Values : "
    f"{invalid_customer_segment.sum():,}"
)


print("\nCustomer Segment Distribution:")

print(
    customers["customer_segment"]
    .value_counts(dropna=False)
    .to_string()
)

Total Customers                 : 2,500
Missing Customer Segment Values : 0
Invalid Customer Segment Values : 0

Customer Segment Distribution:
customer_segment
Retail         931
SMB            786
Enterprise     420
Distributor    363


### Customer Region Domain Validation

Validate whether all customers contain an approved regional classification.

Expected regions:

- Central
- East
- North
- South
- West

Any missing or unexpected region value would represent a categorical data-quality issue.

In [142]:
# Expected customer regions

valid_customer_regions = [
    "Central",
    "East",
    "North",
    "South",
    "West"
]


# Missing region values

missing_customer_region = (
    customers["region"].isna()
)


# Invalid region values

invalid_customer_region = (
    ~customers["region"].isin(valid_customer_regions)
    & customers["region"].notna()
)


print(
    f"Total Customers               : "
    f"{len(customers):,}"
)

print(
    f"Missing Customer Region Values: "
    f"{missing_customer_region.sum():,}"
)

print(
    f"Invalid Customer Region Values: "
    f"{invalid_customer_region.sum():,}"
)


print("\nCustomer Region Distribution:")

print(
    customers["region"]
    .value_counts(dropna=False)
    .to_string()
)

Total Customers               : 2,500
Missing Customer Region Values: 0
Invalid Customer Region Values: 0

Customer Region Distribution:
region
South      707
West       621
North      470
East       386
Central    316


### Customer Credit Tier Domain Validation

Validate whether all non-missing customer credit-tier values belong to the approved classification.

Expected credit tiers:

- A
- B
- C

Missing credit-tier values have already been identified separately and are not treated as invalid categorical values in this check.

In [143]:
# Expected credit tiers

valid_credit_tiers = [
    "A",
    "B",
    "C"
]


# Missing credit tiers

missing_credit_tier = (
    customers["credit_tier"].isna()
)


# Invalid non-missing credit tiers

invalid_credit_tier = (
    customers["credit_tier"].notna()
    & ~customers["credit_tier"].isin(valid_credit_tiers)
)


print(
    f"Total Customers             : "
    f"{len(customers):,}"
)

print(
    f"Missing Credit Tier Values  : "
    f"{missing_credit_tier.sum():,}"
)

print(
    f"Invalid Credit Tier Values  : "
    f"{invalid_credit_tier.sum():,}"
)


print("\nCredit Tier Distribution:")

print(
    customers["credit_tier"]
    .value_counts(dropna=False)
    .to_string()
)

Total Customers             : 2,500
Missing Credit Tier Values  : 20
Invalid Credit Tier Values  : 0

Credit Tier Distribution:
credit_tier
B      1048
A      1022
C       410
NaN      20


### Product Category Domain Validation

Validate whether all products belong to an approved product category.

Expected categories:

- Automotive
- Electronics
- Health & Personal Care
- Home & Kitchen
- Office Supplies

Any missing or unexpected category value would represent a categorical data-quality issue.

In [144]:
# Expected product categories

valid_product_categories = [
    "Automotive",
    "Electronics",
    "Health & Personal Care",
    "Home & Kitchen",
    "Office Supplies"
]


# Missing categories

missing_product_category = (
    products["category"].isna()
)


# Invalid categories

invalid_product_category = (
    products["category"].notna()
    & ~products["category"].isin(valid_product_categories)
)


print(
    f"Total Products          : "
    f"{len(products):,}"
)

print(
    f"Missing Category Values : "
    f"{missing_product_category.sum():,}"
)

print(
    f"Invalid Category Values : "
    f"{invalid_product_category.sum():,}"
)


print("\nProduct Category Distribution:")

print(
    products["category"]
    .value_counts(dropna=False)
    .to_string()
)

Total Products          : 500
Missing Category Values : 0
Invalid Category Values : 0

Product Category Distribution:
category
Electronics               122
Home & Kitchen            109
Office Supplies            97
Health & Personal Care     96
Automotive                 76


### Product Subcategory-to-Category Consistency Audit

Validate the hierarchical consistency between product category and subcategory.

Each non-missing subcategory is expected to belong consistently to a single parent category.

A subcategory appearing under multiple categories could indicate inconsistent product classification.

In [145]:
# Exclude missing subcategories for hierarchy validation

subcategory_mapping = (
    products[
        products["subcategory"].notna()
    ]
    .groupby("subcategory")["category"]
    .nunique()
    .reset_index(name="category_count")
)


# Subcategories mapped to multiple parent categories

inconsistent_subcategories = (
    subcategory_mapping[
        subcategory_mapping["category_count"] > 1
    ]
)


print(
    f"Unique Non-Missing Subcategories : "
    f"{products['subcategory'].dropna().nunique():,}"
)

print(
    f"Subcategories With Multiple Categories: "
    f"{len(inconsistent_subcategories):,}"
)


if len(inconsistent_subcategories) > 0:

    print("\nInconsistent Subcategories:")

    print(
        inconsistent_subcategories
        .sort_values(
            "category_count",
            ascending=False
        )
        .to_string(index=False)
    )


print("\nSubcategory → Category Mapping:")

print(
    products[
        products["subcategory"].notna()
    ][
        [
            "subcategory",
            "category"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "category",
            "subcategory"
        ]
    )
    .to_string(index=False)
)

Unique Non-Missing Subcategories : 14
Subcategories With Multiple Categories: 1

Inconsistent Subcategories:
subcategory  category_count
Accessories               2

Subcategory → Category Mapping:
  subcategory               category
  Accessories             Automotive
   Electrical             Automotive
  Maintenance             Automotive
  Accessories            Electronics
        Audio            Electronics
Smart Devices            Electronics
     Grooming Health & Personal Care
Personal Care Health & Personal Care
     Wellness Health & Personal Care
   Appliances         Home & Kitchen
     Cookware         Home & Kitchen
      Storage         Home & Kitchen
    Furniture        Office Supplies
     Printing        Office Supplies
   Stationery        Office Supplies


### Product Category–Subcategory Combination Audit

Review the observed category and subcategory combinations across the product master.

This audit identifies:

- Valid category–subcategory combinations
- Number of products within each combination
- Missing subcategory records

Shared subcategory names across different parent categories are retained unless there is clear evidence of incorrect classification.

In [146]:
# Category-subcategory combination summary

category_subcategory_summary = (
    products
    .groupby(
        ["category", "subcategory"],
        dropna=False
    )
    .size()
    .reset_index(name="product_count")
    .sort_values(
        ["category", "product_count"],
        ascending=[True, False]
    )
)


print(
    f"Total Products                     : "
    f"{len(products):,}"
)

print(
    f"Unique Category-Subcategory Pairs  : "
    f"{len(category_subcategory_summary):,}"
)

print(
    f"Products With Missing Subcategory  : "
    f"{products['subcategory'].isna().sum():,}"
)


print("\nCategory-Subcategory Distribution:")

print(
    category_subcategory_summary
    .to_string(index=False)
)

Total Products                     : 500
Unique Category-Subcategory Pairs  : 17
Products With Missing Subcategory  : 3

Category-Subcategory Distribution:
              category   subcategory  product_count
            Automotive    Electrical             31
            Automotive   Accessories             25
            Automotive   Maintenance             20
           Electronics         Audio             52
           Electronics   Accessories             35
           Electronics Smart Devices             35
Health & Personal Care      Grooming             39
Health & Personal Care Personal Care             29
Health & Personal Care      Wellness             28
        Home & Kitchen    Appliances             43
        Home & Kitchen      Cookware             36
        Home & Kitchen       Storage             28
        Home & Kitchen           NaN              2
       Office Supplies     Furniture             36
       Office Supplies      Printing             34
       Offic

### Missing Subcategory Products — Recent Sales Activity Audit

Review the sales activity of products with missing subcategory values.

This check determines whether the affected products are active in the product master and whether they have recent transactional activity.

Products with missing classification but valid business activity should be retained and handled through an appropriate missing-value strategy rather than removed.

In [147]:
# Products with missing subcategory

missing_subcategory_detail = products[
    products["subcategory"].isna()
][
    [
        "product_id",
        "product_name",
        "category",
        "active_flag"
    ]
].copy()


# Sales activity for affected products

missing_subcategory_activity = (
    sales_order_items[
        sales_order_items["product_id"].isin(
            missing_subcategory_detail["product_id"]
        )
    ]
    .groupby("product_id")
    .agg(
        sales_lines=("order_item_id", "count"),
        total_ordered_qty=("ordered_qty", "sum"),
        total_net_order_value=("net_order_value", "sum"),
        first_sale_date=("order_date", "min"),
        last_sale_date=("order_date", "max")
    )
    .reset_index()
)


# Combine product details with sales activity

missing_subcategory_review = (
    missing_subcategory_detail
    .merge(
        missing_subcategory_activity,
        on="product_id",
        how="left"
    )
)


print(
    f"Products Missing Subcategory : "
    f"{len(missing_subcategory_review):,}"
)

print(
    f"Active Products              : "
    f"{(missing_subcategory_review['active_flag'] == 'Y').sum():,}"
)

print(
    f"Products With Sales Activity : "
    f"{missing_subcategory_review['sales_lines'].notna().sum():,}"
)


print("\nMissing Subcategory Product Activity:")

print(
    missing_subcategory_review
    .round(2)
    .to_string(index=False)
)

Products Missing Subcategory : 3
Active Products              : 3
Products With Sales Activity : 3

Missing Subcategory Product Activity:
product_id          product_name        category active_flag  sales_lines  total_ordered_qty  total_net_order_value first_sale_date last_sale_date
   SKU0184 Cookware Product 0184  Home & Kitchen           Y          162               1201             1770041.05      2024-01-04     2026-09-28
   SKU0281  Storage Product 0281  Home & Kitchen           Y          130               1396             3097400.88      2024-01-08     2026-09-24
   SKU0424 Printing Product 0424 Office Supplies           Y           61                447              447893.62      2024-01-02     2026-09-28


### Product Active Flag Domain Validation

Validate whether all products contain a valid active-status classification.

Expected values:

- Y — Active
- N — Inactive

Any missing or unexpected value would represent a categorical data-quality issue.

In [148]:
# Expected product active flag values

valid_active_flags = [
    "Y",
    "N"
]


# Missing active flags

missing_active_flag_master = (
    products["active_flag"].isna()
)


# Invalid active flags

invalid_active_flag_master = (
    products["active_flag"].notna()
    & ~products["active_flag"].isin(valid_active_flags)
)


print(
    f"Total Products             : "
    f"{len(products):,}"
)

print(
    f"Missing Active Flag Values : "
    f"{missing_active_flag_master.sum():,}"
)

print(
    f"Invalid Active Flag Values : "
    f"{invalid_active_flag_master.sum():,}"
)


print("\nActive Flag Distribution:")

print(
    products["active_flag"]
    .value_counts(dropna=False)
    .to_string()
)

Total Products             : 500
Missing Active Flag Values : 0
Invalid Active Flag Values : 0

Active Flag Distribution:
active_flag
Y    488
N     12


### Sales Order Status Domain Validation

Validate whether all Sales Orders contain an approved order-status classification.

Expected values:

- Fulfilled
- Partially Fulfilled
- Backordered
- Cancelled

Any missing or unexpected status value would represent a categorical data-quality issue.

In [149]:
# Expected Sales Order statuses

valid_order_statuses = [
    "Fulfilled",
    "Partially Fulfilled",
    "Backordered",
    "Cancelled"
]


# Missing statuses

missing_order_status = (
    sales_orders["order_status"].isna()
)


# Invalid statuses

invalid_order_status = (
    sales_orders["order_status"].notna()
    & ~sales_orders["order_status"].isin(valid_order_statuses)
)


print(
    f"Total Sales Orders          : "
    f"{len(sales_orders):,}"
)

print(
    f"Missing Order Status Values : "
    f"{missing_order_status.sum():,}"
)

print(
    f"Invalid Order Status Values : "
    f"{invalid_order_status.sum():,}"
)


print("\nOrder Status Distribution:")

print(
    sales_orders["order_status"]
    .value_counts(dropna=False)
    .to_string()
)

Total Sales Orders          : 25,000
Missing Order Status Values : 0
Invalid Order Status Values : 0

Order Status Distribution:
order_status
Fulfilled              14150
Partially Fulfilled    10377
Cancelled                275
Backordered              198


### Sales Order Item Backlog Status Domain Validation

Validate whether all Sales Order Item records contain an approved backlog-status classification.

Expected values:

- No Backlog
- Later Fulfilled
- Open Backlog
- Cancelled

Any missing or unexpected value would represent a categorical data-quality issue.

In [150]:
# Expected backlog status values

valid_backlog_statuses = [
    "No Backlog",
    "Later Fulfilled",
    "Open Backlog",
    "Cancelled"
]


# Missing backlog statuses

missing_backlog_status = (
    sales_order_items["backlog_status"].isna()
)


# Invalid backlog statuses

invalid_backlog_status = (
    sales_order_items["backlog_status"].notna()
    & ~sales_order_items["backlog_status"].isin(
        valid_backlog_statuses
    )
)


print(
    f"Total Sales Order Items       : "
    f"{len(sales_order_items):,}"
)

print(
    f"Missing Backlog Status Values : "
    f"{missing_backlog_status.sum():,}"
)

print(
    f"Invalid Backlog Status Values : "
    f"{invalid_backlog_status.sum():,}"
)


print("\nBacklog Status Distribution:")

print(
    sales_order_items["backlog_status"]
    .value_counts(dropna=False)
    .to_string()
)

Total Sales Order Items       : 61,397
Missing Backlog Status Values : 0
Invalid Backlog Status Values : 0

Backlog Status Distribution:
backlog_status
No Backlog         48119
Later Fulfilled     6365
Open Backlog        5564
Cancelled           1349


### Sales Order Item Stockout Flag Domain Validation

Validate whether all Sales Order Item records contain a valid stockout indicator.

Expected values:

- Y — Stockout occurred
- N — No stockout

Any missing or unexpected value would represent a categorical data-quality issue.

In [151]:
# Expected stockout flag values

valid_stockout_flags = [
    "Y",
    "N"
]


# Missing stockout flags

missing_stockout_flag = (
    sales_order_items["stockout_flag"].isna()
)


# Invalid stockout flags

invalid_stockout_flag = (
    sales_order_items["stockout_flag"].notna()
    & ~sales_order_items["stockout_flag"].isin(
        valid_stockout_flags
    )
)


print(
    f"Total Sales Order Items      : "
    f"{len(sales_order_items):,}"
)

print(
    f"Missing Stockout Flag Values : "
    f"{missing_stockout_flag.sum():,}"
)

print(
    f"Invalid Stockout Flag Values : "
    f"{invalid_stockout_flag.sum():,}"
)


print("\nStockout Flag Distribution:")

print(
    sales_order_items["stockout_flag"]
    .value_counts(dropna=False)
    .to_string()
)

Total Sales Order Items      : 61,397
Missing Stockout Flag Values : 0
Invalid Stockout Flag Values : 0

Stockout Flag Distribution:
stockout_flag
N    51151
Y    10246


### Inventory Stock Status Domain Validation

Validate whether all Inventory records contain an approved stock-status classification.

Expected values:

- Healthy
- Low Stock
- Overstock
- Stockout

Any missing or unexpected value would represent a categorical data-quality issue.

In [152]:
# Expected inventory stock status values

valid_stock_statuses = [
    "Healthy",
    "Low Stock",
    "Overstock",
    "Stockout"
]


# Missing stock statuses

missing_stock_status = (
    inventory["stock_status"].isna()
)


# Invalid stock statuses

invalid_stock_status = (
    inventory["stock_status"].notna()
    & ~inventory["stock_status"].isin(
        valid_stock_statuses
    )
)


print(
    f"Total Inventory Records       : "
    f"{len(inventory):,}"
)

print(
    f"Missing Stock Status Values   : "
    f"{missing_stock_status.sum():,}"
)

print(
    f"Invalid Stock Status Values   : "
    f"{invalid_stock_status.sum():,}"
)


print("\nStock Status Distribution:")

print(
    inventory["stock_status"]
    .value_counts(dropna=False)
    .to_string()
)

Total Inventory Records       : 3,000
Missing Stock Status Values   : 0
Invalid Stock Status Values   : 0

Stock Status Distribution:
stock_status
Healthy      2077
Overstock     478
Stockout      240
Low Stock     205


### Shipment On-Time Flag Domain Validation

Validate whether all shipment records contain a valid on-time delivery indicator.

Expected values:

- Y — Delivered on or before the requested delivery date
- N — Delivered after the requested delivery date

This step validates the categorical domain only. The relationship between the flag and delivery-delay calculation has already been validated separately.

In [153]:
# Expected on-time flag values

valid_on_time_flags = [
    "Y",
    "N"
]


# Missing on-time flags

missing_on_time_flag = (
    shipments["on_time_flag"].isna()
)


# Invalid on-time flags

invalid_on_time_flag = (
    shipments["on_time_flag"].notna()
    & ~shipments["on_time_flag"].isin(
        valid_on_time_flags
    )
)


print(
    f"Total Shipments             : "
    f"{len(shipments):,}"
)

print(
    f"Missing On-Time Flag Values : "
    f"{missing_on_time_flag.sum():,}"
)

print(
    f"Invalid On-Time Flag Values : "
    f"{invalid_on_time_flag.sum():,}"
)


print("\nOn-Time Flag Distribution:")

print(
    shipments["on_time_flag"]
    .value_counts(dropna=False)
    .to_string()
)

Total Shipments             : 24,798
Missing On-Time Flag Values : 0
Invalid On-Time Flag Values : 0

On-Time Flag Distribution:
on_time_flag
Y    18869
N     5929


### Shipment OTIF Flag Domain Validation

Validate whether all shipment records contain a valid OTIF (On Time In Full) indicator.

Expected values:

- Y — Shipment met OTIF requirements
- N — Shipment did not meet OTIF requirements

This step validates the categorical domain only. OTIF business-rule consistency has already been validated separately.

In [154]:
# Expected OTIF flag values

valid_otif_flags = [
    "Y",
    "N"
]


# Missing OTIF flags

missing_otif_flag = (
    shipments["otif_flag"].isna()
)


# Invalid OTIF flags

invalid_otif_flag = (
    shipments["otif_flag"].notna()
    & ~shipments["otif_flag"].isin(
        valid_otif_flags
    )
)


print(
    f"Total Shipments          : "
    f"{len(shipments):,}"
)

print(
    f"Missing OTIF Flag Values : "
    f"{missing_otif_flag.sum():,}"
)

print(
    f"Invalid OTIF Flag Values : "
    f"{invalid_otif_flag.sum():,}"
)


print("\nOTIF Flag Distribution:")

print(
    shipments["otif_flag"]
    .value_counts(dropna=False)
    .to_string()
)

Total Shipments          : 24,798
Missing OTIF Flag Values : 0
Invalid OTIF Flag Values : 0

OTIF Flag Distribution:
otif_flag
N    12586
Y    12212


### Purchase Order Status Domain Validation

Validate whether all Purchase Order records contain an approved PO status.

Expected values:

- Open
- Partially Received

Any missing or unexpected status value would represent a categorical data-quality issue.

The relationship between PO status, received quantity, and receipt dates has already been validated separately.

In [155]:
# Expected Purchase Order statuses

valid_po_statuses = [
    "Open",
    "Partially Received"
]


# Missing PO statuses

missing_po_status = (
    purchase_orders["po_status"].isna()
)


# Invalid PO statuses

invalid_po_status = (
    purchase_orders["po_status"].notna()
    & ~purchase_orders["po_status"].isin(
        valid_po_statuses
    )
)


print(
    f"Total Purchase Orders      : "
    f"{len(purchase_orders):,}"
)

print(
    f"Missing PO Status Values   : "
    f"{missing_po_status.sum():,}"
)

print(
    f"Invalid PO Status Values   : "
    f"{invalid_po_status.sum():,}"
)


print("\nPO Status Distribution:")

print(
    purchase_orders["po_status"]
    .value_counts(dropna=False)
    .to_string()
)

Total Purchase Orders      : 8,500
Missing PO Status Values   : 0
Invalid PO Status Values   : 0

PO Status Distribution:
po_status
Partially Received    8470
Open                    30


### Supplier Preferred Flag Domain Validation

Validate whether every supplier contains a valid preferred-supplier indicator.

Expected values:

- Y — Preferred supplier
- N — Non-preferred supplier

Any missing or unexpected value would represent a categorical data-quality issue.

In [156]:
# Expected preferred supplier flag values

valid_preferred_flags = [
    "Y",
    "N"
]


# Missing preferred supplier flags

missing_preferred_flag = (
    suppliers["preferred_supplier_flag"].isna()
)


# Invalid preferred supplier flags

invalid_preferred_flag = (
    suppliers["preferred_supplier_flag"].notna()
    & ~suppliers["preferred_supplier_flag"].isin(
        valid_preferred_flags
    )
)


print(
    f"Total Suppliers                 : "
    f"{len(suppliers):,}"
)

print(
    f"Missing Preferred Flag Values   : "
    f"{missing_preferred_flag.sum():,}"
)

print(
    f"Invalid Preferred Flag Values   : "
    f"{invalid_preferred_flag.sum():,}"
)


print("\nPreferred Supplier Flag Distribution:")

print(
    suppliers["preferred_supplier_flag"]
    .value_counts(dropna=False)
    .to_string()
)

Total Suppliers                 : 100
Missing Preferred Flag Values   : 0
Invalid Preferred Flag Values   : 0

Preferred Supplier Flag Distribution:
preferred_supplier_flag
N    60
Y    40


### Supplier Region Domain Validation

Validate whether every supplier belongs to an approved geographic region.

Expected regions:

- Central
- East
- North
- South
- West

Any missing or unexpected region would represent a categorical data-quality issue.

In [157]:
# Expected supplier regions

valid_supplier_regions = [
    "Central",
    "East",
    "North",
    "South",
    "West"
]


# Missing supplier regions

missing_supplier_region = (
    suppliers["supplier_region"].isna()
)


# Invalid supplier regions

invalid_supplier_region = (
    suppliers["supplier_region"].notna()
    & ~suppliers["supplier_region"].isin(
        valid_supplier_regions
    )
)


print(
    f"Total Suppliers               : "
    f"{len(suppliers):,}"
)

print(
    f"Missing Supplier Region Values: "
    f"{missing_supplier_region.sum():,}"
)

print(
    f"Invalid Supplier Region Values: "
    f"{invalid_supplier_region.sum():,}"
)


print("\nSupplier Region Distribution:")

print(
    suppliers["supplier_region"]
    .value_counts(dropna=False)
    .to_string()
)

Total Suppliers               : 100
Missing Supplier Region Values: 0
Invalid Supplier Region Values: 0

Supplier Region Distribution:
supplier_region
West       26
East       22
Central    19
North      18
South      15


### Warehouse Region Domain Validation

Validate whether every warehouse belongs to an approved geographic region.

Expected regions:

- Central
- East
- North
- South
- West

Any missing or unexpected value would represent a categorical data-quality issue.

In [158]:
# Expected warehouse regions

valid_warehouse_regions = [
    "Central",
    "East",
    "North",
    "South",
    "West"
]


# Missing warehouse regions

missing_warehouse_region = (
    warehouses["region"].isna()
)


# Invalid warehouse regions

invalid_warehouse_region = (
    warehouses["region"].notna()
    & ~warehouses["region"].isin(
        valid_warehouse_regions
    )
)


print(
    f"Total Warehouses               : "
    f"{len(warehouses):,}"
)

print(
    f"Missing Warehouse Region Values: "
    f"{missing_warehouse_region.sum():,}"
)

print(
    f"Invalid Warehouse Region Values: "
    f"{invalid_warehouse_region.sum():,}"
)


print("\nWarehouse Region Distribution:")

print(
    warehouses["region"]
    .value_counts(dropna=False)
    .to_string()
)

Total Warehouses               : 6
Missing Warehouse Region Values: 0
Invalid Warehouse Region Values: 0

Warehouse Region Distribution:
region
South    3
West     2
North    1


### Warehouse Name and City Integrity Validation

Validate the descriptive attributes of the Warehouse master.

The audit checks:

- Missing warehouse names
- Missing city values
- Duplicate warehouse names

Warehouse names should uniquely identify each warehouse, while every warehouse should have an associated city.

In [159]:
# Warehouse descriptive field validation

missing_warehouse_name = (
    warehouses["warehouse_name"].isna()
)

missing_warehouse_city = (
    warehouses["city"].isna()
)

duplicate_warehouse_name = (
    warehouses["warehouse_name"].duplicated(
        keep=False
    )
)


print(
    f"Total Warehouses          : "
    f"{len(warehouses):,}"
)

print(
    f"Missing Warehouse Names   : "
    f"{missing_warehouse_name.sum():,}"
)

print(
    f"Missing Warehouse Cities  : "
    f"{missing_warehouse_city.sum():,}"
)

print(
    f"Duplicate Warehouse Names : "
    f"{duplicate_warehouse_name.sum():,}"
)


print("\nWarehouse Master:")

print(
    warehouses[
        [
            "warehouse_id",
            "warehouse_name",
            "city",
            "region",
            "capacity_units"
        ]
    ]
    .sort_values("warehouse_id")
    .to_string(index=False)
)

Total Warehouses          : 6
Missing Warehouse Names   : 0
Missing Warehouse Cities  : 0
Duplicate Warehouse Names : 0

Warehouse Master:
warehouse_id    warehouse_name      city region  capacity_units
        WH01 Bengaluru Central Bengaluru  South           65000
        WH02       Mumbai West    Mumbai   West           55000
        WH03         Delhi NCR  Gurugram  North           60000
        WH04   Hyderabad South Hyderabad  South           48000
        WH05       Chennai Hub   Chennai  South           50000
        WH06           Pune DC      Pune   West           45000


### Supplier Name Completeness and Uniqueness

Validate the integrity of supplier names in the Supplier master.

The audit checks:

- Missing supplier names
- Blank supplier names
- Duplicate supplier names

Each supplier is expected to have a valid and unique descriptive name.

In [160]:
# Missing supplier names

missing_supplier_name = (
    suppliers["supplier_name"].isna()
)


# Blank supplier names

blank_supplier_name = (
    suppliers["supplier_name"]
    .fillna("")
    .str.strip()
    .eq("")
)


# Duplicate supplier names

duplicate_supplier_name = (
    suppliers["supplier_name"]
    .duplicated(keep=False)
)


print(
    f"Total Suppliers           : "
    f"{len(suppliers):,}"
)

print(
    f"Missing Supplier Names    : "
    f"{missing_supplier_name.sum():,}"
)

print(
    f"Blank Supplier Names      : "
    f"{blank_supplier_name.sum():,}"
)

print(
    f"Duplicate Supplier Names  : "
    f"{duplicate_supplier_name.sum():,}"
)


if duplicate_supplier_name.any():

    print("\nDuplicate Supplier Names:")

    print(
        suppliers.loc[
            duplicate_supplier_name,
            [
                "supplier_id",
                "supplier_name",
                "supplier_region"
            ]
        ]
        .sort_values("supplier_name")
        .to_string(index=False)
    )

Total Suppliers           : 100
Missing Supplier Names    : 0
Blank Supplier Names      : 0
Duplicate Supplier Names  : 0


### Customer Name Completeness and Blank-Value Audit

Validate whether every customer has a populated customer name.

The audit checks:

- Missing customer names
- Blank or whitespace-only customer names
- Number of unique customer names
- Duplicate-name records

Duplicate customer names are not automatically treated as errors because multiple customers may legitimately share the same name.

In [161]:
# Missing customer names

missing_customer_name = (
    customers["customer_name"].isna()
)


# Blank or whitespace-only customer names

blank_customer_name = (
    customers["customer_name"]
    .fillna("")
    .str.strip()
    .eq("")
)


# Duplicate customer names

duplicate_customer_name = (
    customers["customer_name"]
    .duplicated(keep=False)
)


print(
    f"Total Customers            : "
    f"{len(customers):,}"
)

print(
    f"Missing Customer Names     : "
    f"{missing_customer_name.sum():,}"
)

print(
    f"Blank Customer Names       : "
    f"{blank_customer_name.sum():,}"
)

print(
    f"Unique Customer Names      : "
    f"{customers['customer_name'].nunique(dropna=True):,}"
)

print(
    f"Duplicate-Name Records     : "
    f"{duplicate_customer_name.sum():,}"
)


if duplicate_customer_name.any():

    print("\nCustomers With Duplicate Names:")

    print(
        customers.loc[
            duplicate_customer_name,
            [
                "customer_id",
                "customer_name",
                "customer_segment",
                "region"
            ]
        ]
        .sort_values("customer_name")
        .to_string(index=False)
    )

Total Customers            : 2,500
Missing Customer Names     : 0
Blank Customer Names       : 0
Unique Customer Names      : 2,500
Duplicate-Name Records     : 0


### Product Name Completeness and Uniqueness

Validate the integrity of product names in the Product master.

The audit checks:

- Missing product names
- Blank or whitespace-only product names
- Number of unique product names
- Duplicate product-name records

Duplicate product names would require investigation because each product in the master is expected to represent a distinct SKU.

In [162]:
# Missing product names

missing_product_name = (
    products["product_name"].isna()
)


# Blank or whitespace-only product names

blank_product_name = (
    products["product_name"]
    .fillna("")
    .str.strip()
    .eq("")
)


# Duplicate product names

duplicate_product_name = (
    products["product_name"]
    .duplicated(keep=False)
)


print(
    f"Total Products            : "
    f"{len(products):,}"
)

print(
    f"Missing Product Names     : "
    f"{missing_product_name.sum():,}"
)

print(
    f"Blank Product Names       : "
    f"{blank_product_name.sum():,}"
)

print(
    f"Unique Product Names      : "
    f"{products['product_name'].nunique(dropna=True):,}"
)

print(
    f"Duplicate-Name Records    : "
    f"{duplicate_product_name.sum():,}"
)


if duplicate_product_name.any():

    print("\nProducts With Duplicate Names:")

    print(
        products.loc[
            duplicate_product_name,
            [
                "product_id",
                "product_name",
                "category",
                "subcategory"
            ]
        ]
        .sort_values("product_name")
        .to_string(index=False)
    )

Total Products            : 500
Missing Product Names     : 0
Blank Product Names       : 0
Unique Product Names      : 500
Duplicate-Name Records    : 0


### Text Whitespace Consistency Audit

Validate important categorical and descriptive text fields for leading or trailing whitespace.

Whitespace inconsistencies can create hidden duplicate categories, affect joins, and produce incorrect grouping results even when values appear visually identical.

This audit identifies records where the stored value differs from its trimmed version.

In [163]:
# Important text columns across master and transaction tables

text_columns_to_check = {
    "Customers": (
        customers,
        [
            "customer_id",
            "customer_name",
            "customer_segment",
            "region",
            "credit_tier"
        ]
    ),

    "Products": (
        products,
        [
            "product_id",
            "product_name",
            "category",
            "subcategory",
            "supplier_id",
            "active_flag"
        ]
    ),

    "Warehouses": (
        warehouses,
        [
            "warehouse_id",
            "warehouse_name",
            "city",
            "region"
        ]
    ),

    "Suppliers": (
        suppliers,
        [
            "supplier_id",
            "supplier_name",
            "supplier_region",
            "preferred_supplier_flag"
        ]
    ),

    "Sales Orders": (
        sales_orders,
        [
            "priority",
            "sales_channel",
            "order_status"
        ]
    ),

    "Sales Order Items": (
        sales_order_items,
        [
            "stockout_flag",
            "backlog_status"
        ]
    ),

    "Inventory": (
        inventory,
        [
            "stock_status"
        ]
    ),

    "Inventory Transactions": (
        inventory_transactions,
        [
            "transaction_type"
        ]
    ),

    "Shipments": (
        shipments,
        [
            "on_time_flag",
            "otif_flag"
        ]
    ),

    "Purchase Orders": (
        purchase_orders,
        [
            "po_status"
        ]
    )
}


whitespace_results = []


for table_name, (df, columns) in text_columns_to_check.items():

    for column in columns:

        non_null = df[column].dropna().astype(str)

        whitespace_count = (
            non_null.ne(non_null.str.strip())
        ).sum()

        whitespace_results.append(
            {
                "table": table_name,
                "column": column,
                "whitespace_records": whitespace_count
            }
        )


whitespace_audit = pd.DataFrame(
    whitespace_results
)


print(
    f"Fields Checked                 : "
    f"{len(whitespace_audit):,}"
)

print(
    f"Fields With Whitespace Issues  : "
    f"{(whitespace_audit['whitespace_records'] > 0).sum():,}"
)

print(
    f"Total Affected Records         : "
    f"{whitespace_audit['whitespace_records'].sum():,}"
)


print("\nWhitespace Audit Results:")

print(
    whitespace_audit
    .to_string(index=False)
)

Fields Checked                 : 29
Fields With Whitespace Issues  : 0
Total Affected Records         : 0

Whitespace Audit Results:
                 table                  column  whitespace_records
             Customers             customer_id                   0
             Customers           customer_name                   0
             Customers        customer_segment                   0
             Customers                  region                   0
             Customers             credit_tier                   0
              Products              product_id                   0
              Products            product_name                   0
              Products                category                   0
              Products             subcategory                   0
              Products             supplier_id                   0
              Products             active_flag                   0
            Warehouses            warehouse_id                 

### Text Case Consistency Audit

Validate categorical text fields for case inconsistencies.

Values that differ only by uppercase or lowercase formatting can create duplicate categories and inconsistent grouping.

This audit compares the number of original distinct values with their case-normalized equivalents.

In [165]:
# Categorical fields to check for case consistency

case_columns_to_check = {
    "Customers": (
        customers,
        [
            "customer_segment",
            "region",
            "credit_tier"
        ]
    ),

    "Products": (
        products,
        [
            "category",
            "subcategory",
            "active_flag"
        ]
    ),

    "Warehouses": (
        warehouses,
        [
            "region"
        ]
    ),

    "Suppliers": (
        suppliers,
        [
            "supplier_region",
            "preferred_supplier_flag"
        ]
    ),

    "Sales Orders": (
        sales_orders,
        [
            "priority",
            "sales_channel",
            "order_status"
        ]
    ),

    "Sales Order Items": (
        sales_order_items,
        [
            "stockout_flag",
            "backlog_status"
        ]
    ),

    "Inventory": (
        inventory,
        [
            "stock_status"
        ]
    ),

    "Inventory Transactions": (
        inventory_transactions,
        [
            "transaction_type"
        ]
    ),

    "Shipments": (
        shipments,
        [
            "on_time_flag",
            "otif_flag"
        ]
    ),

    "Purchase Orders": (
        purchase_orders,
        [
            "po_status"
        ]
    )
}


case_results = []


for table_name, (df, columns) in case_columns_to_check.items():

    for column in columns:

        values = (
            df[column]
            .dropna()
            .astype(str)
            .str.strip()
        )

        original_unique = values.nunique()

        normalized_unique = (
            values.str.lower().nunique()
        )

        case_results.append(
            {
                "table": table_name,
                "column": column,
                "original_unique": original_unique,
                "case_normalized_unique": normalized_unique,
                "possible_case_issue":
                    original_unique != normalized_unique
            }
        )


case_audit = pd.DataFrame(case_results)


print(
    f"Fields Checked              : "
    f"{len(case_audit):,}"
)

print(
    f"Fields With Case Issues     : "
    f"{case_audit['possible_case_issue'].sum():,}"
)


print("\nCase Consistency Audit:")

print(
    case_audit
    .to_string(index=False)
)

Fields Checked              : 19
Fields With Case Issues     : 0

Case Consistency Audit:
                 table                  column  original_unique  case_normalized_unique  possible_case_issue
             Customers        customer_segment                4                       4                False
             Customers                  region                5                       5                False
             Customers             credit_tier                3                       3                False
              Products                category                5                       5                False
              Products             subcategory               14                      14                False
              Products             active_flag                2                       2                False
            Warehouses                  region                3                       3                False
             Suppliers         supplie

### ID Format Consistency Audit

Validate whether key identifier fields follow their expected structural formats.

Consistent ID formats improve data integrity, traceability, joins, and database validation.

This audit checks the format of:

- Customer IDs
- Product IDs
- Supplier IDs
- Warehouse IDs
- Sales Order IDs
- Sales Order Item IDs
- Inventory Transaction IDs
- Shipment IDs
- Purchase Order IDs

In [166]:
# Expected ID formats

id_format_checks = [
    (
        "Customers",
        customers,
        "customer_id",
        r"^CUST\d+$"
    ),
    (
        "Products",
        products,
        "product_id",
        r"^SKU\d+$"
    ),
    (
        "Suppliers",
        suppliers,
        "supplier_id",
        r"^SUP\d+$"
    ),
    (
        "Warehouses",
        warehouses,
        "warehouse_id",
        r"^WH\d+$"
    ),
    (
        "Sales Orders",
        sales_orders,
        "order_id",
        r"^ORD\d+$"
    ),
    (
        "Sales Order Items",
        sales_order_items,
        "order_item_id",
        r"^OI\d+$"
    ),
    (
        "Inventory Transactions",
        inventory_transactions,
        "transaction_id",
        r"^TXN\d+$"
    ),
    (
        "Shipments",
        shipments,
        "shipment_id",
        r"^SHP\d+$"
    ),
    (
        "Purchase Orders",
        purchase_orders,
        "po_id",
        r"^PO\d+$"
    )
]


id_format_results = []


for table_name, df, column, pattern in id_format_checks:

    values = df[column].astype("string")

    missing_count = values.isna().sum()

    invalid_mask = (
        values.notna()
        & ~values.str.fullmatch(
            pattern,
            na=False
        )
    )

    id_format_results.append(
        {
            "table": table_name,
            "column": column,
            "total_records": len(df),
            "missing_ids": missing_count,
            "invalid_format": invalid_mask.sum()
        }
    )


id_format_audit = pd.DataFrame(
    id_format_results
)


print(
    f"ID Fields Checked           : "
    f"{len(id_format_audit):,}"
)

print(
    f"Fields With Invalid IDs     : "
    f"{(id_format_audit['invalid_format'] > 0).sum():,}"
)

print(
    f"Total Invalid ID Records    : "
    f"{id_format_audit['invalid_format'].sum():,}"
)


print("\nID Format Audit:")

print(
    id_format_audit
    .to_string(index=False)
)

ID Fields Checked           : 9
Fields With Invalid IDs     : 3
Total Invalid ID Records    : 196,442

ID Format Audit:
                 table         column  total_records  missing_ids  invalid_format
             Customers    customer_id           2500            0               0
              Products     product_id            500            0               0
             Suppliers    supplier_id            100            0               0
            Warehouses   warehouse_id              6            0               0
          Sales Orders       order_id          25000            0           25000
     Sales Order Items  order_item_id          61397            0           61397
Inventory Transactions transaction_id         110045            0          110045
             Shipments    shipment_id          24798            0               0
       Purchase Orders          po_id           8500            0               0


In [167]:
# Inspect actual ID formats for the three flagged fields

print("Sales Order IDs:")
print(
    sales_orders["order_id"]
    .head(10)
    .to_string(index=False)
)


print("\nSales Order Item IDs:")
print(
    sales_order_items["order_item_id"]
    .head(10)
    .to_string(index=False)
)


print("\nInventory Transaction IDs:")
print(
    inventory_transactions["transaction_id"]
    .head(10)
    .to_string(index=False)
)


print("\nID Lengths:")

print(
    "order_id:",
    sorted(
        sales_orders["order_id"]
        .astype(str)
        .str.len()
        .unique()
    )
)

print(
    "order_item_id:",
    sorted(
        sales_order_items["order_item_id"]
        .astype(str)
        .str.len()
        .unique()
    )
)

print(
    "transaction_id:",
    sorted(
        inventory_transactions["transaction_id"]
        .astype(str)
        .str.len()
        .unique()
    )
)

Sales Order IDs:
SO000001
SO000002
SO000003
SO000004
SO000005
SO000006
SO000007
SO000008
SO000009
SO000010

Sales Order Item IDs:
SOI0000001
SOI0000002
SOI0000003
SOI0000004
SOI0000005
SOI0000006
SOI0000007
SOI0000008
SOI0000009
SOI0000010

Inventory Transaction IDs:
ITX0083844
ITX0048832
ITX0013630
ITX0085776
ITX0036558
ITX0021904
ITX0101694
ITX0091793
ITX0047782
ITX0013566

ID Lengths:
order_id: [np.int64(8)]
order_item_id: [np.int64(10)]
transaction_id: [np.int64(10)]


In [168]:
print("First 20 Inventory Transaction IDs:")

print(
    inventory_transactions["transaction_id"]
    .head(20)
    .to_string(index=False)
)


print("\nLast 10 Inventory Transaction IDs:")

print(
    inventory_transactions["transaction_id"]
    .tail(10)
    .to_string(index=False)
)

First 20 Inventory Transaction IDs:
ITX0083844
ITX0048832
ITX0013630
ITX0085776
ITX0036558
ITX0021904
ITX0101694
ITX0091793
ITX0047782
ITX0013566
ITX0083806
ITX0021930
ITX0063412
ITX0061044
ITX0027859
ITX0021965
ITX0109418
ITX0030628
ITX0021478
ITX0097574

Last 10 Inventory Transaction IDs:
ITX0103209
ITX0039859
ITX0028419
ITX0090741
ITX0077572
ITX0040573
ITX0019431
ITX0078235
ITX0047399
ITX0025194


In [169]:
# Correct expected ID formats

id_format_checks = [
    (
        "Customers",
        customers,
        "customer_id",
        r"^CUST\d+$"
    ),
    (
        "Products",
        products,
        "product_id",
        r"^SKU\d+$"
    ),
    (
        "Suppliers",
        suppliers,
        "supplier_id",
        r"^SUP\d+$"
    ),
    (
        "Warehouses",
        warehouses,
        "warehouse_id",
        r"^WH\d+$"
    ),
    (
        "Sales Orders",
        sales_orders,
        "order_id",
        r"^SO\d{6}$"
    ),
    (
        "Sales Order Items",
        sales_order_items,
        "order_item_id",
        r"^SOI\d{7}$"
    ),
    (
        "Inventory Transactions",
        inventory_transactions,
        "transaction_id",
        r"^ITX\d{7}$"
    ),
    (
        "Shipments",
        shipments,
        "shipment_id",
        r"^SHP\d+$"
    ),
    (
        "Purchase Orders",
        purchase_orders,
        "po_id",
        r"^PO\d+$"
    )
]


id_format_results = []


for table_name, df, column, pattern in id_format_checks:

    values = df[column].astype("string")

    missing_count = values.isna().sum()

    invalid_mask = (
        values.notna()
        & ~values.str.fullmatch(
            pattern,
            na=False
        )
    )

    id_format_results.append(
        {
            "table": table_name,
            "column": column,
            "total_records": len(df),
            "missing_ids": missing_count,
            "invalid_format": invalid_mask.sum()
        }
    )


id_format_audit = pd.DataFrame(
    id_format_results
)


print(
    f"ID Fields Checked        : "
    f"{len(id_format_audit):,}"
)

print(
    f"Fields With Invalid IDs  : "
    f"{(id_format_audit['invalid_format'] > 0).sum():,}"
)

print(
    f"Total Invalid ID Records : "
    f"{id_format_audit['invalid_format'].sum():,}"
)


print("\nCorrected ID Format Audit:")

print(
    id_format_audit
    .to_string(index=False)
)

ID Fields Checked        : 9
Fields With Invalid IDs  : 0
Total Invalid ID Records : 0

Corrected ID Format Audit:
                 table         column  total_records  missing_ids  invalid_format
             Customers    customer_id           2500            0               0
              Products     product_id            500            0               0
             Suppliers    supplier_id            100            0               0
            Warehouses   warehouse_id              6            0               0
          Sales Orders       order_id          25000            0               0
     Sales Order Items  order_item_id          61397            0               0
Inventory Transactions transaction_id         110045            0               0
             Shipments    shipment_id          24798            0               0
       Purchase Orders          po_id           8500            0               0


### ID Length Consistency Audit

Validate whether identifier fields use a consistent character length.

Consistent ID lengths help detect:

- Truncated identifiers
- Extra characters
- Missing numeric padding
- Structurally malformed IDs

This complements the ID-format validation performed in the previous step.

In [170]:
# ID fields to audit

id_length_checks = [
    ("Customers", customers, "customer_id"),
    ("Products", products, "product_id"),
    ("Suppliers", suppliers, "supplier_id"),
    ("Warehouses", warehouses, "warehouse_id"),
    ("Sales Orders", sales_orders, "order_id"),
    ("Sales Order Items", sales_order_items, "order_item_id"),
    ("Inventory Transactions", inventory_transactions, "transaction_id"),
    ("Shipments", shipments, "shipment_id"),
    ("Purchase Orders", purchase_orders, "po_id")
]


id_length_results = []


for table_name, df, column in id_length_checks:

    lengths = (
        df[column]
        .dropna()
        .astype(str)
        .str.len()
    )

    id_length_results.append(
        {
            "table": table_name,
            "column": column,
            "min_length": lengths.min(),
            "max_length": lengths.max(),
            "unique_lengths": lengths.nunique(),
            "consistent_length": lengths.nunique() == 1
        }
    )


id_length_audit = pd.DataFrame(
    id_length_results
)


print(
    f"ID Fields Checked              : "
    f"{len(id_length_audit):,}"
)

print(
    f"Fields With Inconsistent Length: "
    f"{(~id_length_audit['consistent_length']).sum():,}"
)


print("\nID Length Audit:")

print(
    id_length_audit
    .to_string(index=False)
)

ID Fields Checked              : 9
Fields With Inconsistent Length: 0

ID Length Audit:
                 table         column  min_length  max_length  unique_lengths  consistent_length
             Customers    customer_id           9           9               1               True
              Products     product_id           7           7               1               True
             Suppliers    supplier_id           6           6               1               True
            Warehouses   warehouse_id           4           4               1               True
          Sales Orders       order_id           8           8               1               True
     Sales Order Items  order_item_id          10          10               1               True
Inventory Transactions transaction_id          10          10               1               True
             Shipments    shipment_id           9           9               1               True
       Purchase Orders          po_id  

### Percentage and Rate Field Range Validation

Validate percentage-based fields across the dataset to ensure values remain within their expected logical ranges.

Fields checked:

- Sales Order fill rate
- Sales Order Item discount percentage
- Sales Order Item fill rate

Expected ranges:

- Fill-rate percentages: 0 to 100
- Discount percentage: 0 to 1

Values outside these ranges would represent numerical data-quality issues.

In [171]:
# Percentage / rate range checks

rate_checks = [
    (
        "Sales Orders",
        sales_orders,
        "order_fill_rate_pct",
        0,
        100
    ),
    (
        "Sales Order Items",
        sales_order_items,
        "line_fill_rate_pct",
        0,
        100
    ),
    (
        "Sales Order Items",
        sales_order_items,
        "discount_pct",
        0,
        1
    )
]


rate_results = []


for table_name, df, column, lower, upper in rate_checks:

    series = df[column]

    missing_count = series.isna().sum()

    below_range = (
        series.notna()
        & (series < lower)
    ).sum()

    above_range = (
        series.notna()
        & (series > upper)
    ).sum()

    rate_results.append(
        {
            "table": table_name,
            "column": column,
            "min_value": series.min(),
            "max_value": series.max(),
            "missing_values": missing_count,
            "below_valid_range": below_range,
            "above_valid_range": above_range
        }
    )


rate_range_audit = pd.DataFrame(
    rate_results
)


print(
    f"Rate Fields Checked        : "
    f"{len(rate_range_audit):,}"
)

print(
    f"Fields With Range Issues   : "
    f"{(
        (rate_range_audit['below_valid_range'] > 0)
        | (rate_range_audit['above_valid_range'] > 0)
    ).sum():,}"
)


print("\nPercentage / Rate Range Audit:")

print(
    rate_range_audit
    .round(4)
    .to_string(index=False)
)

Rate Fields Checked        : 3
Fields With Range Issues   : 0

Percentage / Rate Range Audit:
            table              column  min_value  max_value  missing_values  below_valid_range  above_valid_range
     Sales Orders order_fill_rate_pct        0.0     100.00               0                  0                  0
Sales Order Items  line_fill_rate_pct        0.0     100.00               0                  0                  0
Sales Order Items        discount_pct        0.0       0.15               0                  0                  0


### Supplier Rating and Reliability Range Validation

Validate supplier performance score fields against their expected ranges.

Expected ranges:

- Quality Rating: 1 to 5
- Reliability Factor: 0 to 1

The previously identified missing quality rating is tracked separately and is not considered an out-of-range value.

In [172]:
# Supplier score range checks

supplier_score_checks = [
    (
        "quality_rating",
        1,
        5
    ),
    (
        "reliability_factor",
        0,
        1
    )
]


supplier_score_results = []


for column, lower, upper in supplier_score_checks:

    series = suppliers[column]

    missing_count = series.isna().sum()

    below_range = (
        series.notna()
        & (series < lower)
    ).sum()

    above_range = (
        series.notna()
        & (series > upper)
    ).sum()

    supplier_score_results.append(
        {
            "column": column,
            "min_value": series.min(),
            "max_value": series.max(),
            "missing_values": missing_count,
            "below_valid_range": below_range,
            "above_valid_range": above_range
        }
    )


supplier_score_audit = pd.DataFrame(
    supplier_score_results
)


print(
    f"Supplier Score Fields Checked : "
    f"{len(supplier_score_audit):,}"
)

print(
    f"Fields With Range Issues      : "
    f"{(
        (supplier_score_audit['below_valid_range'] > 0)
        | (supplier_score_audit['above_valid_range'] > 0)
    ).sum():,}"
)


print("\nSupplier Score Range Audit:")

print(
    supplier_score_audit
    .round(4)
    .to_string(index=False)
)

Supplier Score Fields Checked : 2
Fields With Range Issues      : 0

Supplier Score Range Audit:
            column  min_value  max_value  missing_values  below_valid_range  above_valid_range
    quality_rating       3.14       4.99               1                  0                  0
reliability_factor       0.57       0.98               0                  0                  0


### Lead-Time Range and Positivity Validation

Validate lead-time fields across the Supplier and Purchase Order tables.

Lead times are expected to be positive because a zero or negative lead time would be operationally invalid.

This audit checks:

- Missing values
- Zero values
- Negative values
- Minimum and maximum observed lead times

Cross-table agreement between supplier contract lead time and Purchase Order contract lead time has already been validated separately.

In [173]:
# Lead-time fields to validate

lead_time_checks = [
    (
        "Suppliers",
        suppliers,
        "contract_lead_time_days"
    ),
    (
        "Purchase Orders",
        purchase_orders,
        "contract_lead_time_days"
    )
]


lead_time_results = []


for table_name, df, column in lead_time_checks:

    series = df[column]

    lead_time_results.append(
        {
            "table": table_name,
            "column": column,
            "min_value": series.min(),
            "max_value": series.max(),
            "missing_values": series.isna().sum(),
            "zero_values": (series == 0).sum(),
            "negative_values": (series < 0).sum()
        }
    )


lead_time_audit = pd.DataFrame(
    lead_time_results
)


print(
    f"Lead-Time Fields Checked : "
    f"{len(lead_time_audit):,}"
)

print(
    f"Fields With Invalid Values: "
    f"{(
        (lead_time_audit['zero_values'] > 0)
        | (lead_time_audit['negative_values'] > 0)
    ).sum():,}"
)


print("\nLead-Time Range Audit:")

print(
    lead_time_audit
    .to_string(index=False)
)

Lead-Time Fields Checked : 2
Fields With Invalid Values: 0

Lead-Time Range Audit:
          table                  column  min_value  max_value  missing_values  zero_values  negative_values
      Suppliers contract_lead_time_days          5         24               0            0                0
Purchase Orders contract_lead_time_days          5         24               0            0                0


### Product Shelf-Life Positivity Validation

Validate the product shelf-life field for numerical integrity.

Shelf life is expected to:

- Be populated
- Be greater than zero
- Contain no negative values

The observed minimum and maximum are also reviewed to understand the overall range.

In [174]:
# Product shelf-life validation

shelf_life = products["shelf_life_days"]


print(
    f"Total Products              : "
    f"{len(products):,}"
)

print(
    f"Missing Shelf-Life Values   : "
    f"{shelf_life.isna().sum():,}"
)

print(
    f"Zero Shelf-Life Values      : "
    f"{(shelf_life == 0).sum():,}"
)

print(
    f"Negative Shelf-Life Values  : "
    f"{(shelf_life < 0).sum():,}"
)

print(
    f"Minimum Shelf Life          : "
    f"{shelf_life.min():,.0f} days"
)

print(
    f"Maximum Shelf Life          : "
    f"{shelf_life.max():,.0f} days"
)


print("\nShelf-Life Summary:")

print(
    shelf_life
    .describe()
    .round(2)
    .to_string()
)

Total Products              : 500
Missing Shelf-Life Values   : 0
Zero Shelf-Life Values      : 0
Negative Shelf-Life Values  : 0
Minimum Shelf Life          : 180 days
Maximum Shelf Life          : 1,095 days

Shelf-Life Summary:
count     500.00
mean      710.48
std       321.40
min       180.00
25%       365.00
50%       730.00
75%      1095.00
max      1095.00


### Base Monthly Demand Positivity Validation

Validate the product base monthly demand field for numerical integrity.

The audit checks for:

- Missing values
- Zero demand
- Negative demand
- Minimum and maximum observed demand

Negative demand would represent an invalid value, while zero-demand products would require further investigation.

In [175]:
# Base monthly demand validation

base_demand = products["base_monthly_demand"]


print(
    f"Total Products              : "
    f"{len(products):,}"
)

print(
    f"Missing Demand Values       : "
    f"{base_demand.isna().sum():,}"
)

print(
    f"Zero Demand Values          : "
    f"{(base_demand == 0).sum():,}"
)

print(
    f"Negative Demand Values      : "
    f"{(base_demand < 0).sum():,}"
)

print(
    f"Minimum Monthly Demand      : "
    f"{base_demand.min():,.0f}"
)

print(
    f"Maximum Monthly Demand      : "
    f"{base_demand.max():,.0f}"
)


print("\nBase Monthly Demand Summary:")

print(
    base_demand
    .describe()
    .round(2)
    .to_string()
)

Total Products              : 500
Missing Demand Values       : 0
Zero Demand Values          : 0
Negative Demand Values      : 0
Minimum Monthly Demand      : 15
Maximum Monthly Demand      : 792

Base Monthly Demand Summary:
count    500.00
mean     149.39
std      107.93
min       15.00
25%       76.75
50%      135.00
75%      199.00
max      792.00


### Inventory Average Daily Demand Validation

Validate the average daily demand field in the Inventory table.

The audit checks for:

- Missing values
- Zero demand
- Negative demand
- Minimum and maximum observed demand

Negative demand would represent an invalid value. Zero-demand records, if present, would be reviewed separately rather than automatically removed.

In [176]:
# Inventory average daily demand validation

avg_daily_demand = inventory["avg_daily_demand"]


print(
    f"Total Inventory Records    : "
    f"{len(inventory):,}"
)

print(
    f"Missing Demand Values      : "
    f"{avg_daily_demand.isna().sum():,}"
)

print(
    f"Zero Demand Values         : "
    f"{(avg_daily_demand == 0).sum():,}"
)

print(
    f"Negative Demand Values     : "
    f"{(avg_daily_demand < 0).sum():,}"
)

print(
    f"Minimum Daily Demand       : "
    f"{avg_daily_demand.min():,.2f}"
)

print(
    f"Maximum Daily Demand       : "
    f"{avg_daily_demand.max():,.2f}"
)


print("\nAverage Daily Demand Summary:")

print(
    avg_daily_demand
    .describe()
    .round(2)
    .to_string()
)

Total Inventory Records    : 3,000
Missing Demand Values      : 0
Zero Demand Values         : 0
Negative Demand Values     : 0
Minimum Daily Demand       : 0.08
Maximum Daily Demand       : 4.40

Average Daily Demand Summary:
count    3000.00
mean        0.83
std         0.60
min         0.08
25%         0.43
50%         0.75
75%         1.11
max         4.40


### Numeric Infinite-Value Sanity Check

Perform a final sanity check across all numeric columns for positive or negative infinite values.

Infinite values can arise from invalid calculations or division-by-zero operations and may cause problems during statistical analysis, SQL loading, and visualization.

Missing values are handled separately and are not classified as infinite values.

In [177]:
import numpy as np

# All project tables

project_tables = {
    "Customers": customers,
    "Products": products,
    "Sales Orders": sales_orders,
    "Sales Order Items": sales_order_items,
    "Inventory": inventory,
    "Inventory Transactions": inventory_transactions,
    "Warehouses": warehouses,
    "Shipments": shipments,
    "Suppliers": suppliers,
    "Purchase Orders": purchase_orders
}


infinite_results = []


for table_name, df in project_tables.items():

    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns

    for column in numeric_columns:

        inf_count = np.isinf(
            df[column].dropna()
        ).sum()

        infinite_results.append(
            {
                "table": table_name,
                "column": column,
                "infinite_values": inf_count
            }
        )


infinite_audit = pd.DataFrame(
    infinite_results
)


print(
    f"Numeric Fields Checked       : "
    f"{len(infinite_audit):,}"
)

print(
    f"Fields With Infinite Values  : "
    f"{(infinite_audit['infinite_values'] > 0).sum():,}"
)

print(
    f"Total Infinite Values        : "
    f"{infinite_audit['infinite_values'].sum():,}"
)


if (infinite_audit["infinite_values"] > 0).any():

    print("\nFields With Infinite Values:")

    print(
        infinite_audit[
            infinite_audit["infinite_values"] > 0
        ].to_string(index=False)
    )

else:

    print(
        "\nNo positive or negative infinite "
        "values detected in numeric fields."
    )

Numeric Fields Checked       : 40
Fields With Infinite Values  : 0
Total Infinite Values        : 0

No positive or negative infinite values detected in numeric fields.


### Final Duplicate and Key Integrity Summary

Consolidate duplicate and primary-key findings across all project tables.

This final summary distinguishes between:

- Duplicate full rows
- Duplicate primary/candidate key values
- Tables with clean key integrity

Inventory uses a composite key of `product_id + warehouse_id`.

Detailed investigation previously confirmed that duplicate Inventory Transaction IDs represent exact duplicate records rather than conflicting transactions.

In [179]:
# Final duplicate and key integrity summary

duplicate_key_summary = pd.DataFrame([
    {
        "table": "Customers",
        "duplicate_rows": customers.duplicated().sum(),
        "duplicate_keys": customers["customer_id"].duplicated().sum()
    },
    {
        "table": "Products",
        "duplicate_rows": products.duplicated().sum(),
        "duplicate_keys": products["product_id"].duplicated().sum()
    },
    {
        "table": "Sales Orders",
        "duplicate_rows": sales_orders.duplicated().sum(),
        "duplicate_keys": sales_orders["order_id"].duplicated().sum()
    },
    {
        "table": "Sales Order Items",
        "duplicate_rows": sales_order_items.duplicated().sum(),
        "duplicate_keys": sales_order_items["order_item_id"].duplicated().sum()
    },
    {
        "table": "Inventory",
        "duplicate_rows": inventory.duplicated().sum(),
        "duplicate_keys": inventory.duplicated(
            subset=["product_id", "warehouse_id"]
        ).sum()
    },
    {
        "table": "Inventory Transactions",
        "duplicate_rows": inventory_transactions.duplicated().sum(),
        "duplicate_keys": inventory_transactions[
            "transaction_id"
        ].duplicated().sum()
    },
    {
        "table": "Warehouses",
        "duplicate_rows": warehouses.duplicated().sum(),
        "duplicate_keys": warehouses["warehouse_id"].duplicated().sum()
    },
    {
        "table": "Shipments",
        "duplicate_rows": shipments.duplicated().sum(),
        "duplicate_keys": shipments["shipment_id"].duplicated().sum()
    },
    {
        "table": "Suppliers",
        "duplicate_rows": suppliers.duplicated().sum(),
        "duplicate_keys": suppliers["supplier_id"].duplicated().sum()
    },
    {
        "table": "Purchase Orders",
        "duplicate_rows": purchase_orders.duplicated().sum(),
        "duplicate_keys": purchase_orders["po_id"].duplicated().sum()
    }
])


duplicate_key_summary["requires_cleaning"] = (
    (duplicate_key_summary["duplicate_rows"] > 0)
    | (duplicate_key_summary["duplicate_keys"] > 0)
)


print("Final Duplicate & Key Integrity Summary:\n")

print(
    duplicate_key_summary
    .to_string(index=False)
)

print(
    "\nTables Requiring Duplicate Cleaning :",
    duplicate_key_summary.loc[
        duplicate_key_summary["requires_cleaning"],
        "table"
    ].tolist()
)

Final Duplicate & Key Integrity Summary:

                 table  duplicate_rows  duplicate_keys  requires_cleaning
             Customers               0               0              False
              Products               0               0              False
          Sales Orders               0               0              False
     Sales Order Items               0               0              False
             Inventory               0               0              False
Inventory Transactions              45              45               True
            Warehouses               0               0              False
             Shipments               0               0              False
             Suppliers               0               0              False
       Purchase Orders               0               0              False

Tables Requiring Duplicate Cleaning : ['Inventory Transactions']


### Final Missing-Value Classification

Consolidate all identified missing values and classify them according to their business meaning.

Missing values are separated into:

- Missing master-data attributes requiring a cleaning decision
- Structural missingness that is valid because of the business process

This prevents valid null values from being incorrectly imputed or removed.

In [180]:
# Final missing-value classification

missing_value_summary = pd.DataFrame([
    {
        "table": "Customers",
        "column": "credit_tier",
        "missing_count": customers["credit_tier"].isna().sum(),
        "classification": "Missing master attribute",
        "cleaning_required": True
    },

    {
        "table": "Products",
        "column": "subcategory",
        "missing_count": products["subcategory"].isna().sum(),
        "classification": "Missing master attribute",
        "cleaning_required": True
    },

    {
        "table": "Suppliers",
        "column": "quality_rating",
        "missing_count": suppliers["quality_rating"].isna().sum(),
        "classification": "Missing master attribute",
        "cleaning_required": True
    },

    {
        "table": "Purchase Orders",
        "column": "actual_receipt_date",
        "missing_count": purchase_orders[
            "actual_receipt_date"
        ].isna().sum(),
        "classification": "Structural missingness - Open PO",
        "cleaning_required": False
    },

    {
        "table": "Purchase Orders",
        "column": "supplier_delay_days",
        "missing_count": purchase_orders[
            "supplier_delay_days"
        ].isna().sum(),
        "classification": "Structural missingness - Open PO",
        "cleaning_required": False
    }
])


print("Final Missing-Value Classification:\n")

print(
    missing_value_summary
    .to_string(index=False)
)


print(
    "\nColumns Requiring Missing-Value Treatment :",
    missing_value_summary.loc[
        missing_value_summary["cleaning_required"],
        "column"
    ].tolist()
)

print(
    "\nStructurally Valid Missing Columns :",
    missing_value_summary.loc[
        ~missing_value_summary["cleaning_required"],
        "column"
    ].tolist()
)

Final Missing-Value Classification:

          table              column  missing_count                   classification  cleaning_required
      Customers         credit_tier             20         Missing master attribute               True
       Products         subcategory              3         Missing master attribute               True
      Suppliers      quality_rating              1         Missing master attribute               True
Purchase Orders actual_receipt_date             30 Structural missingness - Open PO              False
Purchase Orders supplier_delay_days             30 Structural missingness - Open PO              False

Columns Requiring Missing-Value Treatment : ['credit_tier', 'subcategory', 'quality_rating']

Structurally Valid Missing Columns : ['actual_receipt_date', 'supplier_delay_days']


### Final Data Quality Issue Summary

Consolidate the confirmed data-quality issues identified during the audit.

Only issues requiring data preparation or an explicit cleaning decision are included here.

Operational findings such as supplier delays, loss-making discounted sales, long transit times, and inactive-product inventory are not classified as data-quality errors because the underlying records are internally consistent.

In [181]:
# Final confirmed data-quality issue summary

data_quality_issues = pd.DataFrame([
    {
        "issue_id": "DQ01",
        "table": "Inventory Transactions",
        "column": "Entire Record / transaction_id",
        "issue": "Exact duplicate transactions",
        "affected_rows": 45,
        "severity": "Medium",
        "planned_action": "Remove exact duplicate rows"
    },

    {
        "issue_id": "DQ02",
        "table": "Customers",
        "column": "credit_tier",
        "issue": "Missing customer credit tier",
        "affected_rows": 20,
        "severity": "Low",
        "planned_action": "Handle missing category during cleaning"
    },

    {
        "issue_id": "DQ03",
        "table": "Products",
        "column": "subcategory",
        "issue": "Missing product subcategory",
        "affected_rows": 3,
        "severity": "Low",
        "planned_action": "Handle missing category during cleaning"
    },

    {
        "issue_id": "DQ04",
        "table": "Suppliers",
        "column": "quality_rating",
        "issue": "Missing supplier quality rating",
        "affected_rows": 1,
        "severity": "Low",
        "planned_action": "Handle missing rating during cleaning"
    },

    {
        "issue_id": "DQ05",
        "table": "Shipments",
        "column": "actual_delivery_date / shipment_date",
        "issue": "Actual delivery occurs before shipment date",
        "affected_rows": 4467,
        "severity": "High",
        "planned_action": "Correct or exclude invalid chronology for delivery KPIs"
    },

    {
        "issue_id": "DQ06",
        "table": "Multiple Tables",
        "column": "Date columns",
        "issue": "Date fields currently stored as string/object",
        "affected_rows": None,
        "severity": "Medium",
        "planned_action": "Convert validated date fields to datetime"
    }
])


print("Final Confirmed Data Quality Issues:\n")

print(
    data_quality_issues
    .to_string(index=False)
)


print("\nIssue Count by Severity:")

print(
    data_quality_issues["severity"]
    .value_counts()
    .to_string()
)


print(
    "\nTotal Confirmed Issue Types :",
    len(data_quality_issues)
)

Final Confirmed Data Quality Issues:

issue_id                  table                               column                                         issue  affected_rows severity                                          planned_action
    DQ01 Inventory Transactions       Entire Record / transaction_id                  Exact duplicate transactions           45.0   Medium                             Remove exact duplicate rows
    DQ02              Customers                          credit_tier                  Missing customer credit tier           20.0      Low                 Handle missing category during cleaning
    DQ03               Products                          subcategory                   Missing product subcategory            3.0      Low                 Handle missing category during cleaning
    DQ04              Suppliers                       quality_rating               Missing supplier quality rating            1.0      Low                   Handle missing rating dur

## Data Quality Audit Conclusion

The data quality audit was completed across all 10 datasets covering customer, product, sales, inventory, shipment, supplier, and procurement operations.

### Overall Audit Result

The dataset demonstrates strong structural and relational integrity.

The audit confirmed:

- Primary and candidate keys are valid across all tables except for 45 exact duplicate Inventory Transaction records.
- All tested foreign-key relationships are valid with no orphan records.
- Core quantity, monetary, percentage, inventory, order, shipment, and procurement calculations are internally consistent.
- Categorical fields follow valid and consistent domains.
- Text fields contain no significant whitespace or case-consistency issues.
- ID fields follow consistent formats and lengths.
- Date strings are fully parseable with no invalid date values.
- No infinite values were identified across numeric fields.
- Product pricing, inventory valuation, sales calculations, and purchase-order calculations reconcile correctly across related tables.

### Confirmed Data Quality Issues

Six issue types were identified for the cleaning stage:

1. **Inventory Transaction duplicates**
   - 45 exact duplicate records.
   - No conflicting duplicate transaction IDs were found.
   - One copy of each duplicate should be removed.

2. **Missing Customer Credit Tier**
   - 20 customers have missing `credit_tier`.
   - These customers account for approximately 0.70% of sales orders.
   - Missing values require an explicit treatment strategy.

3. **Missing Product Subcategory**
   - 3 products have missing `subcategory`.
   - Their historical sales are valid and should not be removed.
   - Missing subcategories require treatment without deleting associated transactions.

4. **Missing Supplier Quality Rating**
   - 1 active supplier has a missing `quality_rating`.
   - Delivery performance should not be used to infer the rating because the audit found essentially no relationship between quality rating and delivery performance.
   - The missing rating requires an explicit treatment strategy.

5. **Shipment Chronology Anomaly**
   - 4,467 shipment records have `actual_delivery_date < shipment_date`.
   - These records represent approximately 18.01% of shipments.
   - The anomaly occurs across years, warehouses, sales channels, and priorities, indicating a systematic data issue.
   - The anomaly materially overstates delivery-performance KPIs and therefore requires special treatment before shipment KPI analysis.

6. **Date Data Types**
   - Validated date fields are currently stored as string/object values.
   - These fields should be converted to proper datetime types during cleaning.

### Structurally Valid Missing Values

The following missing values should not be treated as data-quality errors:

- 30 missing `actual_receipt_date` values in Purchase Orders.
- 30 missing `supplier_delay_days` values in Purchase Orders.

All occur on Open Purchase Orders with zero received quantity and therefore represent valid structural missingness.

### Valid Business Observations — Not Cleaning Errors

Several unusual patterns were investigated and confirmed to represent legitimate business or operational observations rather than corrupted data:

- 169 long-transit shipments are genuine late deliveries.
- 902 severe supplier-delay observations represent supplier-performance issues.
- 500 loss-making sales lines are mathematically explained by 15% discounts applied to low-margin products.
- Historical sales for inactive products remain valid.
- Inactive products with current inventory or procurement activity require business review but cannot be classified as erroneous without product deactivation dates.
- `Accessories` appearing under both Automotive and Electronics is a valid shared subcategory.
- Early shipments and deliveries are legitimate operational records when chronology remains valid.

### Cleaning Priorities

The next cleaning phase will therefore focus only on confirmed issues:

1. Convert validated date columns to datetime.
2. Remove 45 exact duplicate Inventory Transaction records.
3. Handle missing `credit_tier`.
4. Handle missing product `subcategory`.
5. Handle missing supplier `quality_rating`.
6. Apply a defensible treatment strategy to the 4,467 invalid shipment chronology records.
7. Preserve structurally valid Open PO missing values.
8. Perform post-cleaning validation to confirm that cleaning did not break relational or business-rule integrity.

The audit findings will be used as the baseline for validating the cleaned datasets.